# LINet3 Training + Diagnostics on SUN RGB-D

**Complete training pipeline with gradient health monitoring, stream contribution analysis, and internal CNN visualization**

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime -> Change runtime type -> Hardware accelerator: GPU -> GPU type: A100
- [ ] **Mount Google Drive:** Your code and dataset will be stored on Drive
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_15_train_test.tar.gz` (train + test splits)

---

## What This Notebook Does:

**Training with Full Diagnostics:**
1. Train LINet3 (2-stream: RGB + Depth) on SUN RGB-D 15-category
2. Gradient health monitoring (vanishing/exploding/oscillating detection)
3. Per-stream training loss decomposition
4. Integration weight evolution tracking

**Post-Training Visualization Suite:**
5. Feature map visualization (full model, per-stream, ablation)
6. Stream contribution decomposition (what each stream contributes to each neuron)
7. Stream-decomposed Grad-CAM (where each stream focuses attention)
8. Integration weight analysis (learned fusion priorities per layer)
9. Stream redundancy analysis (are streams learning the same thing?)
10. Per-class stream dominance (which scenes rely on RGB vs Depth?)
11. Misclassification analysis with Grad-CAM comparison
12. Train vs test activation divergence + BN stats reset experiment

---

## About LINet3:

**LINet3** (Linear Integration Network v3) is an N-stream ResNet where fusion happens **inside each convolution neuron**:
- Per-stream independent convolution kernels (full spatial filters)
- Learned 1x1 integration weights that combine stream outputs at every layer
- Integrated pathway carries the fused representation forward

This allows the network to learn **layer-specific, spatially-aware integration strategies**.

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 15-category preprocessed (train + test splits, RGB + Depth)

In [ ]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"  # Extracted location

print("=" * 60)
print("SUN RGB-D 15-CATEGORY DATASET SETUP (TRAIN + TEST)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")

    # Verify structure
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    # Copy compressed file with progress
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} /dev/shm/sunrgbd_19_traintest.tar.gz

    # Extract to local disk
    print(f"\nExtracting dataset to local disk...")
    !tar -xzf /dev/shm/sunrgbd_19_traintest.tar.gz -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    # Remove tar file to save space
    !rm /dev/shm/sunrgbd_19_traintest.tar.gz

    print(f"\nDataset extracted to local disk")

    # Verify extraction
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

## 6. Setup Python Path & Import LINet3

In [ ]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and dataloaders
print("\nImporting LiNet, dataloaders, and visualization tools...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.common.model_helpers import load_pretrained_backbone
from src.models.linear_integration.li_net3.conv import LIBatchNorm2d
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders, SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig

# Import visualization suite
from src.utils.visualization import (
    FeatureMapVisualizer,
    StreamContributionVisualizer,
    StreamGradCAM,
    IntegrationWeightVisualizer,
    find_misclassified,
    compare_samples,
    StreamRedundancyAnalyzer,
    PerClassDominanceAnalyzer,
    ActivationDivergenceAnalyzer,
    IntegrationWeightEvolutionVisualizer,
    reset_bn_stats,
)

print("All imports successful!")

In [ ]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 152  # chosen for StratifiedGroupKFold: all 19 classes in val
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 7. Configuration

All hyperparameters and settings in one place. Modify these before running.

In [ ]:
from src.training.augmentation_config import AugmentationConfig

# ======================== DATASET ========================
DATASET_CONFIG = {
    'data_root': LOCAL_DATASET_PATH,
    'batch_size': 64,
    'num_workers': 2,
    'num_classes': 19,
    'seed': SEED
}

AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.04,
    rgb_aug_mag=1.24,
    depth_aug_prob=1.00,   # reverted from Phase 1b for clean CBBN attribution
    depth_aug_mag=1.29,    # reverted from Phase 1b for clean CBBN attribution
)

# ======================== MODEL ========================
MODEL_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 19,
    'stream_input_channels': [3, 1],  # RGB=3, Depth=1
    'width_multiplier': 0.75,
    'dropout_p': 0.69,
    'device': 'cuda',
    'use_amp': True,
}

STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== PRETRAINED WEIGHTS (Optional) ========================
# Set LOAD_PRETRAINED = True to initialize from ScanNet pretrained backbone.
# The fc head is skipped automatically since num_classes differs.
LOAD_PRETRAINED = True
PRETRAINED_WEIGHTS_PATH = "/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt"  # TODO: set path
FREEZE_BACKBONE_EPOCHS = 3  # Set > 0 to freeze backbone for N warmup epochs (only used when LOAD_PRETRAINED = True)
FREEZE_BACKBONE_LR = 1e-3     # Learning rate for classifier warmup phase

# ======================== OPTIMIZER ========================
STREAM_SPECIFIC_CONFIG = {
    'stream_lrs': [2.048e-04, 2.048e-04],        # [RGB, Depth]
    'shared_lr': 2.048e-04,
    'stream_weight_decays': [1.255e-03, 1.255e-03],  # [RGB, Depth]
    'integration_weight_decay': 1.255e-03,
    'stem_lr_multiplier': 3.10,  # >1.0 to boost stem LR (changes eta_min to 6 values)
}

SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 25,
    's1_eta': 2.414e-06,
    's2_eta': 2.414e-06,
    'eta_min': 2.414e-06,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2
}

# ======================== TRAINING ========================
TRAIN_CONFIG = {
    'epochs': 25,
    'grad_clip_norm': 0.90,
    'early_stopping': False,
    'restore_best_weights': False,
    'stream_monitoring': False,
    'modality_dropout': True,
    'modality_dropout_start':0,
    'modality_dropout_ramp':0,
    'modality_dropout_rate': 0.49,
    'label_smoothing': 0.06,
    # Gradient health monitoring
    'gradient_monitoring': False,
    'gradient_log_freq': 0,  # Last batch per epoch
    # Integration weight tracking
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
    'monitor': 'val_mca',
    # --- Training recipe (locked to best baseline from ablations) ---
    # NOTE on mixup + label_smoothing: the two compose multiplicatively.
    # Effective target is (1-s) * (lam*y_a + (1-lam)*y_b) + s/K. Fine at
    # moderate smoothing (s <= 0.08). s=0.06 here is safe.
    'use_mixup':    True,
    'mixup_alpha':  0.2,
    'use_sam':      False,
    'sam_rho':      0.05,
    # Class-balanced BatchNorm for the integrated pathway. When True, every
    # LIBatchNorm2d is converted to ClassBalancedLIBatchNorm2d in-place at
    # fit() start; running stats accumulate per-class-balanced batch means/
    # vars (equal class weight). Targets the BN-stats-mismatch mechanism that
    # AdaBN exploited at inference (cell 19.16: +4.95pp on REPRESENTATION drift
    # classes), but at training time so it doesn't hurt majority classes.
    # NOTE: incompatible with use_consistency=True (see below).
    'use_class_balanced_bn': False,
    # --- Phase 2: KL Consistency Regularization ---
    # When True, the train dataset returns 5-tuples (rgb_a, depth_a, rgb_b,
    # depth_b, label) and fit() adds an auxiliary KL term forcing the model
    # to predict the same softmax distribution for both augmented views of
    # each sample. Targets train/test distribution shift (per-sensor MCA
    # spread, centroid drift on rep-drift classes). Composes cleanly with
    # use_mixup via SHARED lam+perm across views (symmetric mixup).
    #
    # IMPORTANT: stacking views into a single 2B forward DOUBLES the
    # effective forward batch size. If GPU memory is tight, halve
    # DATASET_CONFIG['batch_size'] when use_consistency=True.
    'use_consistency':         True,
    'consistency_weight':      1.0,
    # T=2 follows FixMatch / Mean-Teacher conventions — softens distributions
    # so KL gradients stay tractable in early epochs. T=1.0 is too sharp.
    'consistency_temperature': 2.0,
}

# Print summary
print('All configs defined.')
print(f'  Dataset: {DATASET_CONFIG["data_root"]}')
print(f'  Model: LINet3-{MODEL_CONFIG["architecture"]} ({len(MODEL_CONFIG["stream_input_channels"])}-stream)')
print(f'  Streams: {STREAM_LABELS}')
print(f'  Epochs: {TRAIN_CONFIG["epochs"]}, Grad clip: {TRAIN_CONFIG["grad_clip_norm"]}')
print(f'  Gradient monitoring: {TRAIN_CONFIG["gradient_monitoring"]}')
print(f'  Integration weight tracking: {TRAIN_CONFIG["track_integration_weights"]}')
print(f'  Consistency: {TRAIN_CONFIG["use_consistency"]} '
      f'(weight={TRAIN_CONFIG["consistency_weight"]}, T={TRAIN_CONFIG["consistency_temperature"]})')

## 8. Load Dataset

In [ ]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"    {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"      {modality}/ - {len(list(mod_dir.glob('*.png')))} images")
        print(f"      labels.txt")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip() for line in f]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

In [ ]:
import json
import random

import numpy as np
import torch
from sklearn.model_selection import StratifiedGroupKFold

from src.training.samplers import build_sampler

print("=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET (TRAIN/VAL/TEST)")
print("=" * 60)

print(f"\nLoading dataset from: {DATASET_CONFIG['data_root']}")

# Create augmented train dataset and non-augmented val dataset (both from train/ split)
# Under use_consistency, the train dataset yields 5-tuples (rgb_a, depth_a,
# rgb_b, depth_b, label) — two independently-augmented views of each sample
# for KL consistency. The val dataset always uses the single-view 3-tuple form.
train_full_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
    return_two_views=TRAIN_CONFIG['use_consistency'],
    **AUGMENTATION_CONFIG.to_dict(),
)
val_full_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='train',
    normalize=True,
)
val_full_dataset.split = 'val'  # Disable augmentation in __getitem__

# 80/20 stratified split
all_labels = train_full_dataset.labels
# Load scene groups for group-aware splitting (no same-scene leakage)
with open(os.path.join(DATASET_CONFIG['data_root'], "train", "scene_groups.json")) as f:
    scene_groups = json.load(f)

# StratifiedGroupKFold: respects both group boundaries AND class stratification
# n_splits=5 gives ~80/20 train/val; we take fold 0 as our locked split
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_indices, val_indices = next(sgkf.split(
    X=list(range(len(all_labels))),
    y=all_labels,
    groups=scene_groups,
))
train_indices, val_indices = list(train_indices), list(val_indices)

train_subset = torch.utils.data.Subset(train_full_dataset, train_indices)
val_subset = torch.utils.data.Subset(val_full_dataset, val_indices)

# Class-only inverse-frequency sampling. V1/V2/V3 sensor-aware variants are
# still available in src.training.samplers for future exploration, but the
# ablation showed they do not materially move test MCA vs class_only on SUN
# RGB-D 19-category — so the notebook is locked to class_only here.
sample_weights, train_sampler = build_sampler(
    variant='class_only',
    train_indices=train_indices,
    all_labels=all_labels,
    data_root=DATASET_CONFIG['data_root'],
    seed=SEED,
)
num_train = len(sample_weights)
subset_labels = [all_labels[i] for i in train_indices]

def worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)

train_loader = torch.utils.data.DataLoader(
    train_subset,
    batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    sampler=train_sampler,
    num_workers=DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

val_loader = torch.utils.data.DataLoader(
    val_subset,
    batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATASET_CONFIG['num_workers'] // 2,
    prefetch_factor=2,
    persistent_workers=False,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

# Test set
test_dataset = SUNRGBDDataset(
    data_root=DATASET_CONFIG['data_root'],
    split='test',
    normalize=True,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATASET_CONFIG['num_workers'],
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

print(f"\nDataset loaded!")
print(f"  Train: {len(train_subset)} samples ({len(train_loader)} batches)")
print(f"  Val:   {len(val_subset)} samples ({len(val_loader)} batches)")
print(f"  Test:  {len(test_dataset)} samples ({len(test_loader)} batches)")
if TRAIN_CONFIG['use_consistency']:
    print(f"  [Consistency] train loader yields 5-tuples (two views); val/test 3-tuples.")

# Test loading a batch
# rgb_batch, depth_batch, label_batch = next(iter(train_loader))
# print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")

print("\n" + "=" * 60)

## 9. Create Model

In [ ]:
from src.models.linear_integration.li_net3 import li_resnet18
from thop import profile


print("=" * 60)
print("MODEL CREATION")
print("=" * 60)

model = li_resnet18(
    num_classes=MODEL_CONFIG['num_classes'],
    stream_input_channels=MODEL_CONFIG['stream_input_channels'],
    width_multiplier=MODEL_CONFIG['width_multiplier'],
    dropout_p=MODEL_CONFIG['dropout_p'],
    device=MODEL_CONFIG['device'],
    use_amp=MODEL_CONFIG['use_amp'],
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# GFLOPs calculation — register custom handlers for LI modules
from src.models.linear_integration.li_net3.conv import LIConv2d, LIBatchNorm2d
from src.models.linear_integration.li_net3.container import LIReLU
from src.models.linear_integration.li_net3.pooling import LIMaxPool2d, LIAdaptiveAvgPool2d

def _liconv2d_flops(module, input, output):
    stream_outs, integrated_out = output
    total = 0
    for w, s_out in zip(module.stream_weights, stream_outs):
        batch, out_c, out_h, out_w = s_out.shape
        kernel_ops = w.shape[1] * w.shape[2] * w.shape[3]
        total += batch * out_c * out_h * out_w * kernel_ops
    if module.integrated_weight.shape[1] > 0:
        batch, out_c, out_h, out_w = integrated_out.shape
        kernel_ops = module.integrated_weight.shape[1]
        total += batch * out_c * out_h * out_w * kernel_ops
    for iw in module.integration_from_streams:
        batch, out_c, out_h, out_w = integrated_out.shape
        kernel_ops = iw.shape[1]
        total += batch * out_c * out_h * out_w * kernel_ops
    module.total_ops += torch.DoubleTensor([total])

def _libn_flops(module, input, output):
    stream_outs, integrated_out = output
    total = sum(s.numel() for s in stream_outs) + integrated_out.numel()
    module.total_ops += torch.DoubleTensor([total * 4])

def _lirelu_flops(module, input, output):
    stream_outs, integrated_out = output
    module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

def _lipool_flops(module, input, output):
    stream_outs, integrated_out = output
    module.total_ops += torch.DoubleTensor([sum(s.numel() for s in stream_outs) + integrated_out.numel()])

custom_ops = {
    LIConv2d: _liconv2d_flops,
    LIBatchNorm2d: _libn_flops,
    LIReLU: _lirelu_flops,
    LIMaxPool2d: _lipool_flops,
    LIAdaptiveAvgPool2d: _lipool_flops,
}

dummy_streams = [torch.randn(1, ch, 224, 224).to(MODEL_CONFIG['device']) for ch in MODEL_CONFIG['stream_input_channels']]
li_flops, _ = profile(model, inputs=(dummy_streams,), custom_ops=custom_ops, verbose=False)
del dummy_streams

print(f"\nLINet3-{MODEL_CONFIG['architecture'].upper()} created")
print(f"  Total parameters: {total_params:,}")
print(f"  GFLOPs: {li_flops / 1e9:.3f}")
print(f"  Model size: {total_params * 4 / 1024**2:.2f} MB (FP32)")
print(f"  Streams: {STREAM_LABELS}")
print(f"  AMP: {MODEL_CONFIG['use_amp']}")

# Load pretrained backbone weights (optional)
if LOAD_PRETRAINED:
    print(f"\nLoading pretrained backbone from: {PRETRAINED_WEIGHTS_PATH}")
    transfer_info = load_pretrained_backbone(model, PRETRAINED_WEIGHTS_PATH)
    print(f"  Loaded: {len(transfer_info['loaded'])} keys (backbone)")
    print(f"  Skipped: {len(transfer_info['skipped'])} keys (classifier head)")
    # Hard abort if anything other than the expected classifier-head keys is
    # missing/unexpected. After the head refactor, the inner Linear is at
    # `fc.fc.{weight,bias}`. Legal skipped keys are therefore either the
    # OLD flat form (`fc.{weight,bias}`, from pre-refactor checkpoints) or
    # the NEW nested form (`fc.fc.{weight,bias}`, from post-refactor
    # checkpoints whose num_classes differs). Anything else in `skipped`
    # means a backbone key drifted and the run is invalid.
    _expected_skipped = {"fc.weight", "fc.bias", "fc.fc.weight", "fc.fc.bias"}
    _skipped_surprises = set(transfer_info["skipped"]) - _expected_skipped
    if _skipped_surprises:
        raise RuntimeError(
            "Pretrained load: unexpected state_dict mismatch beyond classifier head.\n"
            f"  unexpected skipped keys: {sorted(_skipped_surprises)}\n"
            "Backbone weights have not loaded correctly — abort and investigate."
        )
else:
    print("\nTraining from scratch (LOAD_PRETRAINED = False)")

print("\n" + "=" * 60)

## 10. Compile Model (Optimizer + Scheduler)

In [ ]:
import os
from datetime import datetime
from pathlib import Path

# Create checkpoint directory on Google Drive (persistent storage)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/run_{timestamp}"

Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

print(f"Checkpoint directory: {checkpoint_dir}")

In [ ]:
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler

print("=" * 60)
print("MODEL COMPILATION")
print("=" * 60)

# Add checkpoint-dependent paths to TRAIN_CONFIG
TRAIN_CONFIG['save_path'] = f"{checkpoint_dir}/best_model.pt"
TRAIN_CONFIG['integration_snapshot_path'] = f"{checkpoint_dir}/integration_snapshots"


def _create_main_optimizer_and_scheduler():
    """Create the main optimizer + scheduler (used after optional warmup)."""
    opt = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=STREAM_SPECIFIC_CONFIG['stream_lrs'],
        stream_weight_decays=STREAM_SPECIFIC_CONFIG['stream_weight_decays'],
        shared_lr=STREAM_SPECIFIC_CONFIG['shared_lr'],
        integration_weight_decay=STREAM_SPECIFIC_CONFIG['integration_weight_decay'],
        stem_lr_multiplier=STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'],
    )
    sched = setup_scheduler(
        opt,
        scheduler_type=SCHEDULER_CONFIG['scheduler_type'],
        train_loader_len=len(train_loader),
        t_max=SCHEDULER_CONFIG['t_max'],
        eta_min=(
            ([SCHEDULER_CONFIG['s1_eta'] * STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'],
              SCHEDULER_CONFIG['s2_eta'] * STREAM_SPECIFIC_CONFIG['stem_lr_multiplier']]
             if STREAM_SPECIFIC_CONFIG['stem_lr_multiplier'] != 1.0 else []) +
            [SCHEDULER_CONFIG['s1_eta'], SCHEDULER_CONFIG['s2_eta'],
             SCHEDULER_CONFIG['eta_min'], SCHEDULER_CONFIG['eta_min']]
        ),
        warmup_epochs=SCHEDULER_CONFIG['warmup_epochs'],
        warmup_start_factor=SCHEDULER_CONFIG['warmup_start_factor']
    )
    return opt, sched


# --- Optional backbone freeze warmup ---
freeze_epochs = FREEZE_BACKBONE_EPOCHS if LOAD_PRETRAINED else 0
warmup_history = None

if freeze_epochs > 0:
    print(f"\nBACKBONE FREEZE WARMUP ({freeze_epochs} epochs)")
    print("-" * 40)

    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_count = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Frozen: {frozen_count:,} params | Trainable: {trainable_count:,} params")

    fc_optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=FREEZE_BACKBONE_LR,
    )
    model.compile(
        optimizer=fc_optimizer,
        scheduler=None,
        loss='cross_entropy',
        label_smoothing=TRAIN_CONFIG['label_smoothing'],
        gpu_augmentation=False,
        **AUGMENTATION_CONFIG.to_dict(),
    )

    warmup_history = model.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=freeze_epochs,
        verbose=True,
        stream_monitoring=False,
        modality_dropout=False,
        gradient_monitoring=False,
        track_integration_weights=False,
    )

    for param in model.parameters():
        param.requires_grad = True
    print(f"\n  All parameters unfrozen.")

elif LOAD_PRETRAINED:
    print("Backbone freeze warmup: DISABLED (FREEZE_BACKBONE_EPOCHS = 0)")

# --- Main compile (always exactly once) ---
optimizer, scheduler = _create_main_optimizer_and_scheduler()

model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **AUGMENTATION_CONFIG.to_dict(),
)

print(f"\nOptimizer: {optimizer.__class__.__name__}")
for i, group in enumerate(optimizer.param_groups):
    num_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i+1}: lr={group['lr']:.2e}, wd={group['weight_decay']:.2e}, params={num_params:,}")
print("\nModel compiled!")
print("=" * 60)



## 10b. Optional Backbone Freeze Warmup

If `LOAD_PRETRAINED = True` and `FREEZE_BACKBONE_EPOCHS > 0`, freeze the pretrained backbone and train only the classifier head for a few epochs. This lets the new head calibrate before the backbone starts adapting.

## 11. Train with Full Diagnostics

All diagnostics enabled: gradient health monitoring, per-stream training loss decomposition, integration weight norm tracking + periodic full snapshots, stream-specific accuracy monitoring.

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    all_keys = set(h1.keys()) | set(h2.keys())
    for key in all_keys:
        v1 = h1.get(key)
        v2 = h2.get(key)
        if v1 is None and v2 is None:
            merged[key] = None
        elif v1 is None:
            merged[key] = v2
        elif v2 is None:
            merged[key] = v1
        elif isinstance(v1, dict) and isinstance(v2, dict):
            # Recursively merge nested dicts (e.g. per-stream tracking)
            merged[key] = merge_histories(v1, v2)
        elif isinstance(v1, list) and isinstance(v2, list):
            merged[key] = v1 + v2
        else:
            # Scalar or unknown — just keep v2 (phase 2 wins)
            merged[key] = v2
    return merged

In [ ]:
import warnings
import os

# Suppress PyTorch SequentialLR deprecation warning
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

# Create integration snapshot directory
os.makedirs(TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

print("=" * 60)
print("TRAINING WITH FULL DIAGNOSTICS")
print("=" * 60)

print(f"Configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")


print("=" * 60 + "\n")

# Train with all diagnostics enabled
history = model.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=TRAIN_CONFIG['save_path'],
    early_stopping=TRAIN_CONFIG['early_stopping'],
    patience=15,
    restore_best_weights=TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=TRAIN_CONFIG['stream_monitoring'],
    monitor=TRAIN_CONFIG['monitor'],
    modality_dropout=TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=TRAIN_CONFIG['modality_dropout_rate'],
    # Training recipe toggles (defaults False - preserves baseline behavior)
    use_mixup=TRAIN_CONFIG['use_mixup'],
    mixup_alpha=TRAIN_CONFIG['mixup_alpha'],
    use_sam=TRAIN_CONFIG['use_sam'],
    sam_rho=TRAIN_CONFIG['sam_rho'],
    use_class_balanced_bn=TRAIN_CONFIG['use_class_balanced_bn'],
    # Phase 2: KL consistency regularization (two-view aug-invariance)
    use_consistency=TRAIN_CONFIG['use_consistency'],
    consistency_weight=TRAIN_CONFIG['consistency_weight'],
    consistency_temperature=TRAIN_CONFIG['consistency_temperature'],
    # # Gradient health monitoring
    # gradient_monitoring=TRAIN_CONFIG['gradient_monitoring'],
    # gradient_log_freq=TRAIN_CONFIG['gradient_log_freq'],
    # # Integration weight tracking
    # track_integration_weights=TRAIN_CONFIG['track_integration_weights'],
    # integration_snapshot_path=TRAIN_CONFIG['integration_snapshot_path'],
    # integration_snapshot_freq=TRAIN_CONFIG['integration_snapshot_freq'],
)

# Merge warmup + full history if freeze warmup was used
if warmup_history is not None:
    history = merge_histories(warmup_history, history)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)

## 12. Single-Stream Robustness Evaluation

How much does the model degrade when a stream is missing? Tests full model, RGB-only, and Depth-only.

In [ ]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

# Evaluate with all streams (normal)
print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(test_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%  MCA: {results_both['mean_class_accuracy']*100:.2f}%")

# Evaluate with RGB only (Depth blanked)
print("\n[2/3] Evaluating with RGB ONLY (Depth blanked):")
results_rgb_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%  MCA: {results_rgb_only['mean_class_accuracy']*100:.2f}%")

# Evaluate with Depth only (RGB blanked)
print("\n[3/3] Evaluating with DEPTH ONLY (RGB blanked):")
results_depth_only = model.evaluate(test_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%  MCA: {results_depth_only['mean_class_accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  Acc={results_both['accuracy']*100:.2f}%  MCA={results_both['mean_class_accuracy']*100:.2f}%")
print(f"  RGB only:      Acc={results_rgb_only['accuracy']*100:.2f}%  MCA={results_rgb_only['mean_class_accuracy']*100:.2f}% (Depth missing)")
print(f"  Depth only:    Acc={results_depth_only['accuracy']*100:.2f}%  MCA={results_depth_only['mean_class_accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100

print(f"\n  Degradation when Depth missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")
print("\n" + "=" * 60)

## 13. Test Set Evaluation + Pathway Analysis

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Evaluate on test set
results = model.evaluate(data_loader=test_loader, stream_monitoring=True)

print(f"\nTest Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")
print(f"  Mean Class Accuracy: {results['mean_class_accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

# Pathway analysis
print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")
print(f"\nAnalyzing stream pathways and integrated pathway contributions...")

pathway_analysis = model.analyze_pathways(data_loader=test_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

# Accuracy
print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")
# acc_int = pathway_analysis['accuracy']['integrated_only']
# contrib_int = pathway_analysis['accuracy']['integrated_contribution']
# print(f"  Integrated only: {acc_int*100:.2f}%  (contribution ratio: {contrib_int:.3f})")

# Loss
print("\nLoss:")
print(f"  Full model:      {pathway_analysis['loss']['full_model']:.4f}")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    loss_i = pathway_analysis['loss'][f'stream{i}_only']
    loss_contrib = pathway_analysis['loss'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {loss_i:.4f}  (loss ratio: {loss_contrib:.3f})")
# loss_int = pathway_analysis['loss']['integrated_only']
# loss_int_contrib = pathway_analysis['loss']['integrated_contribution']
# print(f"  Integrated only: {loss_int:.4f}  (loss ratio: {loss_int_contrib:.3f})")

# Feature norms
print("\nFeature Norms (mean +/- std):")
for i in range(len(MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

# Training summary
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"  Initial train loss: {history['train_loss'][0]:.4f}")
print(f"  Final train loss:   {history['train_loss'][-1]:.4f}")
print(f"  Initial train acc:  {history['train_accuracy'][0]*100:.2f}%")
print(f"  Final train acc:    {history['train_accuracy'][-1]*100:.2f}%")
print(f"  Test accuracy:      {results['accuracy']*100:.2f}%")
print(f"  Test MCA:           {results['mean_class_accuracy']*100:.2f}%")
print(f"  Total epochs:       {len(history['train_loss'])}")

print("\n" + "=" * 60)

## 14. Training Curves + Gradient Health + Stream Loss Decomposition

In [ ]:
# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# # --- Row 1: Standard training curves (restored from original + adapted for no val set) ---

# # Loss curve
# axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
# axes[0, 0].set_xlabel('Epoch', fontsize=12)
# axes[0, 0].set_ylabel('Loss', fontsize=12)
# axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
# axes[0, 0].legend(fontsize=11)
# axes[0, 0].grid(True, alpha=0.3)

# # Accuracy curve with per-stream curves
# axes[0, 1].plot([acc*100 for acc in history['train_accuracy']], label='Full Model Train', linewidth=2, color='green')
# if 'train_mca' in history and history['train_mca']:
#     axes[0, 1].plot([m*100 for m in history['train_mca']], label='Train MCA', linewidth=2, color='darkorange', linestyle=':')

# # Add per-stream curves (always available with stream_monitoring=True)
# stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
# stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
# for i in range(len(MODEL_CONFIG['stream_input_channels'])):
#     color_idx = i % len(stream_train_colors)
#     axes[0, 1].plot([acc*100 for acc in history[f'stream_{i}_train_acc']],
#                 label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
#                 color=stream_train_colors[color_idx])

# axes[0, 1].set_xlabel('Epoch', fontsize=12)
# axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
# axes[0, 1].set_yticks([20, 40, 60, 80, 100])
# axes[0, 1].set_title('Training Accuracy\n(Full Model = Integrated Stream)', fontsize=14, fontweight='bold')
# axes[0, 1].legend(fontsize=9, loc='lower right')
# # Draw gridlines manually: alpha=0.3 for multiples of 10, alpha=0.2 for 5, 15, 25...
# for y in range(0, 101, 10):
#     axes[0, 1].axhline(y=y, color='gray', alpha=0.3, linewidth=0.5)
# for y in range(5, 100, 10):
#     axes[0, 1].axhline(y=y, color='gray', alpha=0.2, linewidth=0.5)
# axes[0, 1].grid(True, axis='x', alpha=0.3)

# # Learning rate curve with per-stream LRs
# sampled_lrs = history['learning_rates'][::max(1, len(history['learning_rates'])//100)]
# axes[0, 2].plot(sampled_lrs, linewidth=2, color='green', label='Base LR')

# # Add per-stream LRs (always available with stream_monitoring=True)
# lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
# for i in range(len(MODEL_CONFIG['stream_input_channels'])):
#     color_idx = i % len(lr_colors)
#     axes[0, 2].plot(history[f'stream_{i}_lr'], linewidth=1, alpha=0.7, linestyle='--',
#                 color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} LR')
#     # Plot stem LR if stem_lr_multiplier was active (separate higher LR for conv1)
#     if f'stem_{i}_lr' in history:
#         axes[0, 2].plot(history[f'stem_{i}_lr'], linewidth=1.5, alpha=0.5, linestyle=':',
#                     color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} Stem LR')

# axes[0, 2].set_xlabel('Epoch', fontsize=12)
# axes[0, 2].set_ylabel('Learning Rate', fontsize=12)
# axes[0, 2].set_yscale('log')
# axes[0, 2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
# axes[0, 2].legend(fontsize=9, loc='upper right')
# axes[0, 2].grid(True, alpha=0.3)

# # --- Row 2: New diagnostics ---

# # Gradient norms over epochs (values are dicts with mean/max/min)
# if 'gradient_norms' in history and history['gradient_norms']:
#     grad_epochs = range(len(history['gradient_norms']))
#     for i in range(len(MODEL_CONFIG['stream_input_channels'])):
#         key = f'stream_{i}'
#         norms = [d.get(key, {}).get('mean', 0) for d in history['gradient_norms']]
#         color = stream_val_colors[i % len(stream_val_colors)]
#         axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5)
#     shared_norms = [d.get('shared', {}).get('mean', 0) for d in history['gradient_norms']]
#     axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
#     axes[1, 0].set_yscale('log')
#     axes[1, 0].set_xlabel('Epoch', fontsize=12)
#     axes[1, 0].set_ylabel('Gradient Norm (pre-clip, log)', fontsize=12)
#     axes[1, 0].set_title('Per-Stream Gradient Norms (mean)', fontsize=14, fontweight='bold')
#     axes[1, 0].legend(fontsize=9)
#     axes[1, 0].grid(True, alpha=0.3)
# else:
#     axes[1, 0].text(0.5, 0.5, 'No gradient data\n(gradient_monitoring=False)', ha='center', va='center',
#                     transform=axes[1, 0].transAxes, fontsize=12)
#     axes[1, 0].set_title('Per-Stream Gradient Norms', fontsize=14, fontweight='bold')

# contrib_keys = [f'stream_{i}_train_acc' for i in range(len(MODEL_CONFIG['stream_input_channels']))]
# if contrib_keys[0] in history:
#     import math
#     n_streams = len(MODEL_CONFIG['stream_input_channels'])
#     baseline_vals = history['train_accuracy']
#     for i in range(n_streams):
#         color = stream_val_colors[i % len(stream_val_colors)]
#         other = (i + 1) % n_streams if n_streams == 2 else i
#         other_vals = history[f'stream_{other}_train_acc']
#         contrib = []
#         epochs_eval = []
#         for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
#             if not math.isnan(other_acc):
#                 contrib.append((base - other_acc) * 100)
#                 epochs_eval.append(e)
#         axes[1, 1].plot(epochs_eval, contrib,
#                        label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5, marker='o', markersize=3)
#     axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
#     axes[1, 1].set_xlabel('Epoch', fontsize=12)
#     axes[1, 1].set_ylabel('Contribution (pp)', fontsize=12)
#     axes[1, 1].set_title('Per-Stream Contribution\n(Baseline − Acc w/o Stream)', fontsize=14, fontweight='bold')
#     axes[1, 1].legend(fontsize=9)
#     axes[1, 1].grid(True, alpha=0.3)
# else:
#     axes[1, 1].text(0.5, 0.5, 'No stream data\n(stream_monitoring=False)', ha='center', va='center',
#                     transform=axes[1, 1].transAxes, fontsize=12)
#     axes[1, 1].set_title('Per-Stream Contribution', fontsize=14, fontweight='bold')




# # Gradient health status summary
# if 'gradient_health' in history and history['gradient_health']:
#     axes[1, 2].axis('off')
#     health_text = "Gradient Health Summary:\n\n"
#     status_counts = {}
#     for h in history['gradient_health']:
#         status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
#         status_counts[status] = status_counts.get(status, 0) + 1
#     for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
#         health_text += f"  {status}: {count} epochs\n"
#     axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
#                     fontsize=10, verticalalignment='top', fontfamily='monospace')
#     axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')
# else:
#     axes[1, 2].text(0.5, 0.5, 'No gradient health data\n(gradient_monitoring=False)', ha='center', va='center',
#                     transform=axes[1, 2].transAxes, fontsize=12)
#     axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')

# plt.tight_layout()
# plt.savefig(f"{checkpoint_dir}/training_diagnostics.pdf", dpi=150, bbox_inches='tight')
# plt.show()

# print(f"Training diagnostics saved to: {checkpoint_dir}/training_diagnostics.pdf")

## 15. Integration Weight Evolution During Training

How did the learned integration priorities change over training? Did the model start RGB-heavy and shift toward Depth?

In [ ]:
# # Integration weight evolution visualization
# evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# # Stream backbone weight norm evolution (RGB, Depth, Integrated)
# if 'stream_weight_norms' in history:
#     evo_viz.plot_stream_weight_norms(history, save_path=f"{checkpoint_dir}/stream_weight_evolution.pdf")
#     print(f"Stream weight norm evolution saved.")
# else:
#     print("No stream weight norm data found in history.")

In [ ]:
# # Plot norm evolution from training history
# if 'integration_weight_norms' in history:
#     evo_viz.plot_norm_evolution(history, save_path=f"{checkpoint_dir}/integration_weight_evolution.pdf")
#     print(f"Integration weight norm evolution saved.")
# else:
#     print("No integration weight norm data found in history.")

In [ ]:
# # Plot full weight snapshots if saved
# snapshot_dir = TRAIN_CONFIG.get('integration_snapshot_path')
# if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
#     # Full grid as PNG (all layers, raster — too heavy for PDF)
#     evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{checkpoint_dir}/integration_weight_snapshots.png")
#     print(f"Integration weight snapshot heatmaps saved (full, PNG).")

#     # Early layers only as PDF (vector, paper-ready)
#     for layer_name in ['conv1', 'layer1']:
#         evo_viz.plot_snapshot_heatmaps(
#             snapshot_dir,
#             layer_filter=layer_name,
#             save_path=f"{checkpoint_dir}/integration_weight_snapshots_{layer_name}.pdf"
#         )
#     print(f"Integration weight snapshot heatmaps saved (conv1 + layer1, PDF).")
# else:
#     print("No integration weight snapshots found.")

In [ ]:
# # Visualize learned first-layer conv filters (7x7 kernels)
# iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)
# iw_viz.visualize_conv1_filters(save_path=f'{checkpoint_dir}/conv1_filters.pdf')
# print('Conv1 filter visualization saved.')

## 16. Save Results & Model

In [ ]:
import json
import torch

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{checkpoint_dir}/training_history.json"
with open(history_path, 'w') as f:
    # Build pathway analysis dict with all returned data
    pa_json = {
        'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
        'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
        'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
        'samples_analyzed': pathway_analysis['samples_analyzed'],
    }

    json_history = {
        'train_loss': [float(x) for x in history['train_loss']],
        'train_accuracy': [float(x) for x in history['train_accuracy']],
        'learning_rates': [float(x) for x in history['learning_rates']],
        # Per-stream and stem LR curves (when stream_monitoring=True)
        **{f'stream_{i}_lr': [float(x) for x in history.get(f'stream_{i}_lr', [])]
           for i in range(2)},
        **{f'stem_{i}_lr': [float(x) for x in history.get(f'stem_{i}_lr', [])]
           for i in range(2) if f'stem_{i}_lr' in history},
        'model_config': MODEL_CONFIG,
        'dataset_config': DATASET_CONFIG,
        'augmentation_config': AUGMENTATION_CONFIG.to_dict(),
        'stream_specific_config': STREAM_SPECIFIC_CONFIG,
        'scheduler_config': SCHEDULER_CONFIG,
        'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v for k, v in TRAIN_CONFIG.items()},
        'train_mca': [float(x) for x in history.get('train_mca', [])],
        'test_results': {
            'loss': float(results['loss']),
            'accuracy': float(results['accuracy']),
            'mean_class_accuracy': float(results.get('mean_class_accuracy', 0)),
        },
        'pathway_analysis': pa_json,
    }

    json.dump(json_history, f, indent=2)

print(f"Training history saved: {history_path}")

# Save final model
final_model_path = f"{checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': MODEL_CONFIG,
    'history': history,
    'test_accuracy': results['accuracy']
}, final_model_path)

print(f"Final model saved: {final_model_path}")

# List saved files
print(f"\nAll results saved to: {checkpoint_dir}")
!ls -lh {checkpoint_dir}

print("\n" + "=" * 60)

## 17. Internal CNN Visualization Suite

Everything below runs on the **trained model** with the **test set**. Each cell is independent — run whichever analyses interest you.

In [ ]:
# # --- 17a. Feature Map Visualization ---
# # "What does the CNN see at each layer?"
# # Three modes: full model, single-stream isolated, ablation
# # Compare layer1 (early/texture) vs layer4 (late/semantic)

# fm_viz = FeatureMapVisualizer(model, stream_labels=STREAM_LABELS)

# # Get a single test sample
# test_iter = iter(test_loader)
# sample_batch = next(test_iter)
# *stream_batches, labels = sample_batch
# # Take first sample
# stream_inputs = [s[0:1].to(model.device) for s in stream_batches]

# print(f"Sample label: {labels[0].item()} ({class_names[labels[0].item()] if 'class_names' in dir() else '?'})")

# for layer in ['layer1', 'layer4']:
#     print(f"\n{'='*60}")
#     print(f"  {layer.upper()} FEATURE MAPS")
#     print(f"{'='*60}")

#     # Mode 1: Full model view (all streams + integrated)
#     print(f"\n--- Full Model View ({layer}) ---")
#     fm_viz.visualize(stream_inputs, layer=layer, top_k=8,
#                      save_path=f"{checkpoint_dir}/featuremaps_full_{layer}.png")

#     # Mode 2: Per-stream isolated views
#     for i, label in STREAM_LABELS.items():
#         print(f"\n--- {label} Stream Isolated View ({layer}) ---")
#         fm_viz.visualize(stream_inputs, layer=layer, mode='stream', stream_idx=i, top_k=8,
#                          save_path=f"{checkpoint_dir}/featuremaps_{label.lower()}_{layer}.png")

#     # Mode 3: Ablation (what happens when we remove a stream?)
#     for i, label in STREAM_LABELS.items():
#         print(f"\n--- Ablation: {label} Blanked ({layer}) ---")
#         fm_viz.visualize(stream_inputs, layer=layer, mode='ablation', stream_idx=i, top_k=8,
#                          save_path=f"{checkpoint_dir}/featuremaps_ablation_{label.lower()}_{layer}.png")

# # Batch-averaged feature maps at both layers
# for layer in ['layer1', 'layer4']:
#     print(f"\n--- Batch-Averaged Feature Maps ({layer}, 32 samples) ---")
#     fm_viz.visualize_batch(test_loader, layer=layer, n=32, top_k=8,
#                            save_path=f"{checkpoint_dir}/featuremaps_batch_avg_{layer}.png")

# print("\nFeature map visualizations complete!")

In [ ]:
# # --- 17b. Stream Contribution Decomposition ---
# # THE unique LINet3 visualization: how much does each stream contribute
# # to each neuron's activation in the integrated pathway?

# contrib_viz = StreamContributionVisualizer(model, stream_labels=STREAM_LABELS)

# # Single image contribution at layer4
# print("--- Stream Contributions (layer4, single sample) ---")
# contrib_viz.visualize(stream_inputs, layer='layer4',
#                       save_path=f"{checkpoint_dir}/contributions_layer4.pdf")

# # Batch-averaged contributions (more representative)
# print("\n--- Batch-Averaged Contributions (layer4, 32 samples) ---")
# contrib_viz.visualize_batch(test_loader, layer='layer4', n=32,
#                             save_path=f"{checkpoint_dir}/contributions_batch_layer4.pdf")

# # Multi-layer comparison
# for layer in ['layer1', 'layer2', 'layer3', 'layer4']:
#     print(f"\n--- Contributions at {layer} ---")
#     contrib_viz.visualize(stream_inputs, layer=layer,
#                           save_path=f"{checkpoint_dir}/contributions_{layer}.pdf")

# print("\nStream contribution decomposition complete!")

In [ ]:
# # --- 17c. Stream-Decomposed Grad-CAM ---
# # Where does each stream focus its attention?

# gradcam = StreamGradCAM(model, stream_labels=STREAM_LABELS)

# # Integrated Grad-CAM (standard: where does the full model look?)
# print("--- Integrated Grad-CAM (layer4) ---")
# gradcam.visualize(stream_inputs, layer='layer4', mode='integrated',
#                   save_path=f"{checkpoint_dir}/gradcam_integrated_layer4.png")

# # Per-stream isolated Grad-CAM (where does each stream look independently?)
# for i, label in STREAM_LABELS.items():
#     print(f"\n--- {label} Stream Grad-CAM (layer4) ---")
#     gradcam.visualize(stream_inputs, layer='layer4', mode='stream', stream_idx=i,
#                       save_path=f"{checkpoint_dir}/gradcam_{label.lower()}_layer4.png")

# # Decomposed mode: contribution maps weighted by Grad-CAM importance
# print("\n--- Decomposed Grad-CAM (layer4) ---")
# gradcam.visualize(stream_inputs, layer='layer4', mode='decomposed',
#                   save_path=f"{checkpoint_dir}/gradcam_decomposed_layer4.png")

# # Multi-layer Grad-CAM (early=texture, late=semantics)
# for layer in ['layer2', 'layer3', 'layer4']:
#     print(f"\n--- Integrated Grad-CAM at {layer} ---")
#     gradcam.visualize(stream_inputs, layer=layer, mode='integrated',
#                       save_path=f"{checkpoint_dir}/gradcam_integrated_{layer}.png")

# print("\nGrad-CAM visualizations complete!")

In [ ]:
# # --- 17d. Integration Weight Visualization ---
# # Visualize the learned integration_from_streams weights per layer

# iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)

# # Weight heatmaps per layer and stream
# print('--- Integration Weights (Heatmaps) ---')
# iw_viz.visualize_weights(save_path=f'{checkpoint_dir}/integration_weights.png')

# # Cross-stream comparison (relative weight magnitudes per layer)
# print('\n--- Cross-Stream Weight Comparison ---')
# iw_viz.visualize_cross_stream(save_path=f'{checkpoint_dir}/integration_cross_stream.pdf')

# # Effective rank via SVD (how low-dimensional is the integration?)
# print('\n--- Effective Rank (SVD) ---')
# ranks = iw_viz.compute_effective_rank()
# for layer, r in ranks.items():
#     print(f'  {layer}: {[f"{x:.1f}" for x in r]}')

# print('\nIntegration weight visualization complete!')

In [ ]:
# # --- 17e. Stream Redundancy Analysis ---
# # Are RGB and Depth learning the same features? Or complementary ones?
# # Uses centered cosine similarity between stream feature maps at each layer.

# redundancy = StreamRedundancyAnalyzer(model, stream_labels=STREAM_LABELS)

# print('--- Stream Redundancy (Centered Cosine Similarity) ---')
# sim_results = redundancy.analyze(
#     test_loader,
#     n=128,  # Average over 128 samples
#     save_path=f'{checkpoint_dir}/stream_redundancy.pdf'
# )

# # Print similarity matrices
# for layer_name, sim_matrix in sim_results.items():
#     print(f'\n{layer_name}:')
#     for i in range(sim_matrix.shape[0]):
#         row = '  '.join(f'{sim_matrix[i,j]:.3f}' for j in range(sim_matrix.shape[1]))
#         print(f'  {STREAM_LABELS.get(i, f"S{i}")}: {row}')

# print('\nStream redundancy analysis complete!')

In [ ]:
# # --- 17f. Per-Class Stream Dominance ---
# # Which scenes rely on RGB vs Depth?
# # "Depth matters more for bathrooms, RGB dominates corridors"

# # Build class name mapping
# class_name_map = {i: name for i, name in enumerate(class_names)} if 'class_names' in dir() else None

# dominance = PerClassDominanceAnalyzer(model, stream_labels=STREAM_LABELS)

# print('--- Per-Class Stream Dominance (layer4) ---')
# class_dominance = dominance.analyze(
#     test_loader,
#     layer='layer4',
#     class_names=class_name_map,
#     save_path=f'{checkpoint_dir}/per_class_dominance.pdf'
# )

# # Print per-class ratios
# print('\nPer-class stream contribution ratios:')
# for cls_idx, ratios in sorted(class_dominance.items()):
#     name = class_name_map[cls_idx] if class_name_map else f'Class {cls_idx}'
#     ratio_str = ', '.join(f'{STREAM_LABELS.get(i, f"S{i}")}: {r:.2%}' for i, r in enumerate(ratios))
#     print(f'  {name}: {ratio_str}')

# print('\nPer-class dominance analysis complete!')

In [ ]:
# # --- 17g. Misclassification Analysis + Sample Comparison ---
# # Find misclassified samples and compare with correctly classified ones

# print('--- Finding Misclassified Samples ---')
# misclassified = find_misclassified(model, test_loader, n=10)

# print(f'Found {len(misclassified)} misclassified samples:')
# for i, mc in enumerate(misclassified[:5]):
#     true_name = class_names[mc['true_label']] if 'class_names' in dir() else str(mc['true_label'])
#     pred_name = class_names[mc['predicted_label']] if 'class_names' in dir() else str(mc['predicted_label'])
#     print(f'  [{i}] True: {true_name}, Predicted: {pred_name}, Confidence: {mc["confidence"]:.2%}')

# # Grad-CAM on first misclassified sample
# if misclassified:
#     mc_sample = misclassified[0]
#     mc_inputs = [s.to(model.device) for s in mc_sample['stream_inputs']]
#     true_name = class_names[mc_sample['true_label']] if 'class_names' in dir() else str(mc_sample['true_label'])
#     pred_name = class_names[mc_sample['predicted_label']] if 'class_names' in dir() else str(mc_sample['predicted_label'])
#     print(f'\n--- Grad-CAM on Misclassified: True={true_name}, Pred={pred_name} ---')
#     gradcam.visualize(mc_inputs, layer='layer4', mode='decomposed',
#                       save_path=f'{checkpoint_dir}/gradcam_misclassified_0.png')

# # Compare correct vs misclassified from same class
# if misclassified:
#     target_class = misclassified[0]['true_label']
#     print(f'\n--- Finding correctly classified sample from class {class_names[target_class] if "class_names" in dir() else target_class} ---')

#     # Find a correctly classified sample from the same class
#     correct_sample = None
#     model.eval()
#     with torch.no_grad():
#         for batch_data in test_loader:
#             *stream_batches, targets = batch_data
#             stream_batches_dev = [s.to(model.device) for s in stream_batches]
#             targets_dev = targets.to(model.device)
#             logits = model(stream_batches_dev)
#             preds = logits.argmax(dim=1)
#             # Find correctly classified samples of the target class
#             mask = (targets_dev == target_class) & (preds == target_class)
#             if mask.any():
#                 idx = mask.nonzero(as_tuple=True)[0][0].item()
#                 correct_sample = {
#                     'stream_inputs': [s[idx:idx+1].cpu() for s in stream_batches],
#                     'true_label': target_class,
#                     'predicted_label': target_class,
#                     'confidence': torch.softmax(logits[idx], dim=0)[target_class].item(),
#                 }
#                 break

#     if correct_sample is not None:
#         print(f'  Found correct sample (confidence: {correct_sample["confidence"]:.2%})')
#         print('\n--- Correct vs Misclassified Comparison ---')
#         compare_samples(
#             model,
#             correct_sample=correct_sample,
#             misclassified_sample=misclassified[0],
#             layer='layer4',
#             stream_labels=STREAM_LABELS,
#             save_path=f'{checkpoint_dir}/compare_samples.png'
#         )
#     else:
#         print('  No correctly classified sample found for this class.')

# print('\nMisclassification analysis complete!')

In [ ]:
# # --- 17h. Train vs Test Activation Divergence ---
# # Does the model see different activation distributions on train vs test?
# # Uses MMD (Maximum Mean Discrepancy) per layer.

# div_analyzer = ActivationDivergenceAnalyzer(model)

# print('--- Train vs Test Activation Divergence ---')
# divergence = div_analyzer.analyze(
#     train_loader,
#     test_loader,
#     n=128,
#     save_path=f'{checkpoint_dir}/activation_divergence.pdf'
# )

# for layer_name, metrics in divergence.items():
#     print(f'  {layer_name}: MMD={metrics["mmd"]:.4f}')

# print('\nActivation divergence analysis complete!')

In [ ]:
# # --- 17i. BN Stats Reset Experiment (Oracle Diagnostic) ---
# # WARNING: This is a DIAGNOSTIC tool, not a deployable fix.
# # It recomputes BN running stats on test data (oracle) to check if
# # BN statistics drift causes the generalization gap.

# import copy

# # Save original accuracy
# original_test_results = model.evaluate(test_loader)
# original_acc = original_test_results['accuracy']
# print(f'Original test accuracy: {original_acc*100:.2f}%')

# # Control: recompute BN stats on TRAIN set (should be ~same)
# print('\n--- Control: Recompute BN stats on TRAIN set ---')
# model_control = copy.deepcopy(model)
# reset_bn_stats(model_control, train_loader)
# control_results = model_control.evaluate(test_loader)
# control_acc = control_results['accuracy']
# print(f'After train BN reset: {control_acc*100:.2f}% (delta: {(control_acc-original_acc)*100:+.2f}%)')

# # Oracle: recompute BN stats on TEST set
# print('\n--- Oracle: Recompute BN stats on TEST set ---')
# model_oracle = copy.deepcopy(model)
# reset_bn_stats(model_oracle, test_loader)
# oracle_results = model_oracle.evaluate(test_loader)
# oracle_acc = oracle_results['accuracy']
# print(f'After test BN reset (oracle): {oracle_acc*100:.2f}% (delta: {(oracle_acc-original_acc)*100:+.2f}%)')

# # Interpretation
# print('\n--- Interpretation ---')
# oracle_delta = (oracle_acc - original_acc) * 100
# if abs(oracle_delta) > 2:
#     print(f'BN stats drift accounts for ~{oracle_delta:+.1f}% of the gap.')
#     print('Consider: test-time BN adaptation, larger batch size, or more training data.')
# else:
#     print(f'BN stats drift is minimal ({oracle_delta:+.1f}%). Gap is likely from other sources.')

# del model_control, model_oracle  # Free memory
# print('\nBN reset experiment complete!')

## 18. Summary

All training diagnostics and visualization analyses are saved to the checkpoint directory on Google Drive.

**Saved models:**
- `best_model.pt` - Best model checkpoint (by training loss, since no val set)
- `final_model.pt` - Final model with full state dict, optimizer, scheduler, history

**Saved data:**
- `training_history.json` - Full training history, configs, test results, pathway analysis
- `integration_snapshots/` - Periodic full integration weight snapshots (every N epochs)

**Saved visualizations:**
- `training_diagnostics.png` - 2x3 grid: loss, accuracy, LR, gradient norms, stream losses, gradient health
- `integration_weight_evolution.png` - Per-stream integration weight norms over training epochs
- `integration_weight_snapshots.png` - Detailed weight heatmaps at snapshot epochs
- `featuremaps_*.png` - What the CNN sees (full model, per-stream isolated, ablation, batch-averaged)
- `contributions_*.png` - Per-stream contribution magnitudes at each layer
- `gradcam_*.png` - Spatial attention maps (integrated, per-stream, decomposed, multi-layer)
- `gradcam_misclassified_0.png` - Decomposed Grad-CAM on a misclassified sample
- `integration_weights.png` - Learned fusion weight heatmaps per layer
- `integration_cross_stream.png` - Cross-stream weight magnitude comparison
- `stream_redundancy.png` - Centered cosine similarity between stream features per layer
- `per_class_dominance.png` - Which scenes rely on RGB vs Depth
- `activation_divergence.png` - Train vs test activation distribution shift (MMD) per layer
- `compare_samples.png` - Correct vs misclassified side-by-side (Grad-CAM + contributions)

## 19. Per-Class Generalization Diagnostics

Runs the full (a)–(d) analysis suite on the trained `model`. Because this
notebook holds `val_indices` genuinely out of training, the hpo_val metrics
here are honest (not circular like the full-training run).

- Per-class accuracy across hpo_train / hpo_val / official_test (same weights)
- Test confusion matrix
- Penultimate-feature MMD whole-split and **per-class** train↔test
- **(a)** Per-class accuracy gap `a_c - b_c`, ranked, weighted by `min(N_val_c, N_test_c)`
- **(b)** Synthetic test accuracy `Σ (N_test_c / N_test) · a_c` vs actual test accuracy
       (quantifies how much of the gap is prior shift vs per-class generalization failure)
- **(c)** Clean-test subset: remove test samples whose scene also appears in hpo_train
- **(d)** Bootstrap 95% CI on hpo_val MCA (sample-size noise check)

All outputs saved to `{checkpoint_dir}/gen_diagnostics/`.

In [ ]:
# --- 19.0 Setup: build eval loaders + out dir ---
# Principle: reuse training's loaders unchanged, and when we have to build a
# new one use the EXACT same DataLoader kwargs training uses for val_loader.
#
#   - hpo_val:       reuse training's `val_loader` directly (no-aug, shuffle=False, no sampler).
#   - official_test: reuse training's `test_loader` directly.
#   - hpo_train:     build a new loader, mirroring the training val_loader block in
#                    cell 19 verbatim (same kwargs, same worker_init_fn). We wrap
#                    `val_full_dataset` (no-aug) with `train_indices` so we get the
#                    un-augmented training samples for evaluation. Training's
#                    `train_loader` can't be reused because it uses the augmented
#                    dataset + WeightedRandomSampler.
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

GEN_DIR = os.path.join(checkpoint_dir, "gen_diagnostics")
os.makedirs(GEN_DIR, exist_ok=True)

# --- hpo_train eval loader: same pattern as training's val_loader (cell 19) ---
hpo_train_eval_subset = torch.utils.data.Subset(val_full_dataset, train_indices)
hpo_train_eval_loader = torch.utils.data.DataLoader(
    hpo_train_eval_subset,
    batch_size=DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=DATASET_CONFIG['num_workers'] // 2,
    prefetch_factor=2,
    persistent_workers=False,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

eval_loaders = {
    "hpo_train":     hpo_train_eval_loader,   # built above; same kwargs as training's val_loader
    "hpo_val":       val_loader,              # reuse training's val_loader directly
    "official_test": test_loader,             # reuse training's test_loader directly
}

# Test scene groups (needed by 19.7 clean-test analysis)
with open(os.path.join(DATASET_CONFIG['data_root'], "test", "scene_groups.json")) as f:
    test_scene_groups = json.load(f)

NUM_CLASSES = DATASET_CONFIG['num_classes']
CLASS_NAMES = class_names
print(f"Eval loaders built.")
print(f"  hpo_train (new subset loader, val_loader kwargs): {len(hpo_train_eval_subset)} samples")
print(f"  hpo_val   (reused training val_loader):           {len(val_loader.dataset)} samples")
print(f"  test      (reused training test_loader):          {len(test_loader.dataset)} samples")


In [ ]:
# --- 19.0b Pre-flight pipeline-parity check (HARD FAIL on divergence) ---
# Section 19 / 20 numbers are only trustworthy if the diagnostic eval path uses
# the same normalization, dataset construction, and model configuration as the
# training path. This cell hard-raises on any mismatch so we never spend an hour
# chasing ghosts because test_dataset drifted from train_dataset.
#
# Checks (all must pass):
#   1. train / val / test dataset .normalize flags agree
#   2. model.gpu_augmentation XOR dataset.normalize (no double-norm / no missing-norm)
#   3. val_full_dataset.split == 'val' so augmentation is OFF during eval
#   4. train_indices and val_indices are disjoint
#   5. first-batch RGB and depth input ranges agree across hpo_train / hpo_val / test
#      -- this is the decisive check that catches the normalize=False bug, gpu_aug
#      asymmetry, or any other dataset-level drift.
import numpy as np
import torch

print("=== Pipeline-parity pre-flight ===")
print(f"  SEED (used by training + eval)  = {SEED}")
print(f"  model.gpu_augmentation          = {getattr(model, 'gpu_augmentation', None)}")
print(f"  train_full_dataset.normalize    = {train_full_dataset.normalize}")
print(f"  val_full_dataset.normalize      = {val_full_dataset.normalize}")
print(f"  test_dataset.normalize          = {test_dataset.normalize}")
print(f"  train_full_dataset.split        = {train_full_dataset.split!r}  (should be 'train')")
print(f"  val_full_dataset.split          = {val_full_dataset.split!r}  (should be 'val')")
print(f"  test_dataset.split              = {test_dataset.split!r}  (should be 'test')")
print(f"  len(train_indices), len(val_indices) = {len(train_indices)}, {len(val_indices)}")

errors = []

# 1. normalize flags consistent across datasets
norms = {train_full_dataset.normalize, val_full_dataset.normalize, test_dataset.normalize}
if len(norms) != 1:
    errors.append(
        f"dataset.normalize disagrees: train={train_full_dataset.normalize} "
        f"val={val_full_dataset.normalize} test={test_dataset.normalize}"
    )

# 2. normalize XOR gpu_augmentation: exactly one path must normalize
ga  = getattr(model, 'gpu_augmentation', False)
dsn = train_full_dataset.normalize
if ga and dsn:
    errors.append("double normalization: gpu_augmentation=True AND dataset.normalize=True")
if (not ga) and (not dsn):
    errors.append("no normalization: gpu_augmentation=False AND dataset.normalize=False")

# 3. val_full_dataset must be in 'val' mode (no augmentation) for eval
if val_full_dataset.split != 'val':
    errors.append(f"val_full_dataset.split = {val_full_dataset.split!r} — augmentation ON during eval")

# 4. index sets disjoint
overlap = set(train_indices) & set(val_indices)
if overlap:
    errors.append(f"train/val indices overlap by {len(overlap)} samples")

# 5. first-batch input ranges across eval loaders (the decisive check)
@torch.no_grad()
def first_batch_ranges(loader):
    batch = next(iter(loader))
    *streams, _ = batch
    streams = [s.float() for s in streams]
    return {
        'rgb':   (float(streams[0].min()), float(streams[0].max())),
        'depth': (float(streams[1].min()), float(streams[1].max())),
    }

ranges = {name: first_batch_ranges(loader) for name, loader in eval_loaders.items()}
print(f"\n  first-batch input ranges (training-path loaders should agree):")
for name, r in ranges.items():
    print(f"    {name:<14}  rgb=[{r['rgb'][0]:+.3f},{r['rgb'][1]:+.3f}]  "
          f"depth=[{r['depth'][0]:+.3f},{r['depth'][1]:+.3f}]")

def ranges_agree(a, b, tol=0.5):
    return (abs(a['rgb'][0]   - b['rgb'][0])   < tol and
            abs(a['rgb'][1]   - b['rgb'][1])   < tol and
            abs(a['depth'][0] - b['depth'][0]) < tol and
            abs(a['depth'][1] - b['depth'][1]) < tol)

tr, va, te = ranges['hpo_train'], ranges['hpo_val'], ranges['official_test']
if not ranges_agree(tr, va):
    errors.append("hpo_train vs hpo_val input ranges disagree (same dataset, different indices — should be identical)")
if not ranges_agree(tr, te):
    errors.append(
        f"hpo_train vs official_test input ranges disagree.  "
        f"train rgb={tr['rgb']} depth={tr['depth']}, test rgb={te['rgb']} depth={te['depth']}.  "
        f"Most likely: test_dataset.normalize != train_full_dataset.normalize, or gpu_augmentation "
        f"was enabled/disabled asymmetrically between training and eval."
    )

if errors:
    print("\n--- ERRORS ---")
    for e in errors:
        print(f"  ! {e}")
    raise RuntimeError(
        f"Pre-flight FAILED with {len(errors)} issue(s). The diagnostics below would be misleading. "
        f"Fix the issues above and rerun.  (If you recently edited cells in Colab, close and reopen "
        f"the notebook to pick up the committed version.)"
    )

print("\n  [OK] diagnostic eval path matches training path. Section 19 / 20 numbers are trustworthy.")

In [ ]:
# --- 19.1 Evaluate on all 3 splits, canonically ---
# Two-pass design per split:
#   Pass A: model.evaluate(loader, stream_monitoring=False)
#     -> canonical loss / accuracy / mca via the exact same _validate path
#        training uses for val metrics. Cannot drift from training.
#   Pass B: forward-hook pass
#     -> captures per-sample penultimate features + predictions for MMD,
#        confusion matrix, and per-class accuracy. The hook pass mirrors
#        _validate (same gpu_aug conditional, same AMP autocast) but exists
#        ONLY because model.evaluate doesn't return per-sample outputs.
#   Parity assertion:
#     -> Pass B's accuracy must match Pass A's to within 0.1pp. If not,
#        the hook pass has drifted from _validate (e.g. a new flag added
#        to _validate that the hook pass doesn't know about). Hard fail.
#
# Why not a single pass? Because model.evaluate is the training pipeline's
# eval method — making IT the source of truth for metrics removes an entire
# class of silent-divergence bugs. The hook pass is a thin, auditable
# supplement for features+preds, guarded by the inline parity check.
@torch.no_grad()
def hook_capture(model, loader, hook_module):
    """Forward pass that mirrors _validate's forward, capturing penultimate
    features and predictions. Used ONLY to supply features + per-sample preds;
    metrics come from model.evaluate."""
    captured = {"features": []}
    def _hook(module, inputs, output):
        x = inputs[0] if isinstance(inputs, tuple) else inputs
        if x.ndim > 2:
            x = torch.flatten(x, 1)
        captured["features"].append(x.detach().float().cpu())
    handle = hook_module.register_forward_hook(_hook)

    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        *streams, labels = batch
        streams = [s.to(model.device, non_blocking=True) for s in streams]
        # Match _validate's forward path exactly (li_net.py:1893-1916)
        if getattr(model, 'gpu_augmentation', False) and getattr(model, 'gpu_aug', None) is not None:
            streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])
        with torch.amp.autocast(device_type=model.device.type, enabled=getattr(model, 'use_amp', False)):
            logits = model(streams)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
    handle.remove()
    return (np.concatenate(all_preds),
            np.concatenate(all_labels),
            torch.cat(captured["features"], dim=0).numpy())

PARITY_TOL_PP = 0.1  # hook-pass vs model.evaluate accuracy must agree within 0.1pp

gen_results = {}
for name, loader in eval_loaders.items():
    print(f"Evaluating on {name} ...")

    # Pass A: canonical metrics via training's own eval method.
    m = model.evaluate(loader, stream_monitoring=False)

    # Pass B: hook pass for per-sample features + predictions.
    preds, labels_np, features = hook_capture(model, loader, hook_module=model.fc)

    # Parity: hook pass must agree with model.evaluate on accuracy.
    hook_acc = float((preds == labels_np).mean())
    if abs(hook_acc - m['accuracy']) * 100 > PARITY_TOL_PP:
        raise RuntimeError(
            f"[19.1] hook pass diverged from model.evaluate on {name}:\n"
            f"  model.evaluate acc = {m['accuracy']*100:.4f}%\n"
            f"  hook-pass acc      = {hook_acc*100:.4f}%\n"
            f"  diff               = {abs(hook_acc - m['accuracy'])*100:.4f}pp  (tol {PARITY_TOL_PP}pp)\n"
            f"The hook pass no longer mirrors _validate. Check if _validate "
            f"(src/models/linear_integration/li_net3/li_net.py) has changed."
        )

    # Per-class accuracy derived from the hook-pass preds (already parity-verified above).
    per_class_acc = np.zeros(NUM_CLASSES)
    per_class_n   = np.zeros(NUM_CLASSES, dtype=np.int64)
    for c in range(NUM_CLASSES):
        mk = labels_np == c
        per_class_n[c] = int(mk.sum())
        per_class_acc[c] = float((preds[mk] == c).mean()) if mk.any() else float('nan')

    gen_results[name] = {
        "preds": preds,
        "labels": labels_np,
        "features": features,
        "acc": m['accuracy'],                  # from model.evaluate (canonical)
        "mca": m['mean_class_accuracy'],       # from model.evaluate (canonical)
        "loss": m['loss'],                     # from model.evaluate (canonical)
        "per_class_acc": per_class_acc,
        "per_class_n": per_class_n,
    }
    r = gen_results[name]
    print(f"  n={len(r['labels']):>5}  loss={r['loss']:.4f}  acc={r['acc']*100:.2f}%  "
          f"mca={r['mca']*100:.2f}%  feat.shape={r['features'].shape}  "
          f"(hook-parity ok: |delta|={abs(hook_acc - m['accuracy'])*100:.4f}pp)")

# Cache predictions + features so 19.4-19.8 can be rerun without re-evaluating
cache_path = os.path.join(GEN_DIR, "eval_cache.npz")
np.savez_compressed(
    cache_path,
    **{f"{s}_preds":    gen_results[s]["preds"]    for s in eval_loaders},
    **{f"{s}_labels":   gen_results[s]["labels"]   for s in eval_loaders},
    **{f"{s}_features": gen_results[s]["features"] for s in eval_loaders},
)
print(f"\nSaved eval cache -> {cache_path}")


In [ ]:
# --- 19.2 Per-class accuracy table + bar chart (all 3 splits, same weights) ---
rows = []
for c in range(NUM_CLASSES):
    row = {"class_idx": c, "class": CLASS_NAMES[c]}
    for s in eval_loaders:
        row[f"{s}_acc"] = gen_results[s]["per_class_acc"][c]
        row[f"{s}_n"]   = int(gen_results[s]["per_class_n"][c])
    row["val_minus_test_acc"] = row["hpo_val_acc"] - row["official_test_acc"]
    tn = row["hpo_train_n"]; ten = row["official_test_n"]
    row["train_test_n_ratio"] = (tn / ten) if ten > 0 else float("inf")
    rows.append(row)
pc_df = pd.DataFrame(rows)
pc_df.to_csv(os.path.join(GEN_DIR, "per_class_accuracy.csv"), index=False)

display(pc_df.style.format({
    "hpo_train_acc": "{:.3f}", "hpo_val_acc": "{:.3f}", "official_test_acc": "{:.3f}",
    "train_test_n_ratio": "{:.2f}", "val_minus_test_acc": "{:+.3f}",
}).background_gradient(subset=["val_minus_test_acc"], cmap="RdYlGn_r"))

x = np.arange(NUM_CLASSES)
width = 0.26
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})
for i, s in enumerate(eval_loaders):
    ax1.bar(x + (i - 1) * width, [gen_results[s]["per_class_acc"][c] for c in range(NUM_CLASSES)],
            width, label=s)
ax1.set_ylim(0, 1); ax1.set_ylabel("per-class accuracy"); ax1.legend()
ax1.grid(axis="y", alpha=0.3); ax1.set_title("Per-class accuracy (same weights on 3 splits)")

ax2.bar(x - width/2, [gen_results["hpo_train"]["per_class_n"][c] for c in range(NUM_CLASSES)],
        width, label="hpo_train n", alpha=0.7)
ax2.bar(x + width/2, [gen_results["official_test"]["per_class_n"][c] for c in range(NUM_CLASSES)],
        width, label="test n", alpha=0.7)
ax2.set_ylabel("samples"); ax2.legend(); ax2.grid(axis="y", alpha=0.3)
ax2.set_xticks(x); ax2.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(GEN_DIR, "per_class_accuracy.png"), dpi=120)
plt.show()

In [ ]:
# --- 19.3 Test confusion matrix ---
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
for t, p in zip(gen_results["official_test"]["labels"], gen_results["official_test"]["preds"]):
    cm[t, p] += 1
with np.errstate(invalid="ignore"):
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Test confusion (row-normalized)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=7)
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(os.path.join(GEN_DIR, "test_confusion.png"), dpi=120)
plt.show()

pd.DataFrame(cm, index=[f"t_{n}" for n in CLASS_NAMES],
             columns=[f"p_{n}" for n in CLASS_NAMES]).to_csv(
    os.path.join(GEN_DIR, "test_confusion_matrix.csv"))

In [ ]:
# --- 19.4 Penultimate-feature MMD: whole-split + per-class ---
# Naming note: `feat_mean_dist_sq` = ||mean(x) - mean(y)||^2 (squared L2 between
# mean feature embeddings). This is algebraically identical to MMD^2 with a
# linear kernel k(a,b)=a.b, but the intuitive name is "feature mean distance"
# to avoid confusion with the more common RBF-kernel MMD.
# Per-class RBF MMD is only meaningful when both sides have >= ~30 samples;
# median-distance gamma on a few samples is very noisy, so we skip rarer classes.
def feat_mean_dist_sq(x, y):
    return float(np.sum((x.mean(axis=0) - y.mean(axis=0)) ** 2))

def rbf_mmd_sq(x, y, gamma=None, max_samples=1024, seed=0):
    rng = np.random.default_rng(seed)
    xi = rng.choice(len(x), size=min(max_samples, len(x)), replace=False)
    yi = rng.choice(len(y), size=min(max_samples, len(y)), replace=False)
    x = x[xi]; y = y[yi]
    if gamma is None:
        z = np.concatenate([x, y], axis=0)
        m = min(512, len(z))
        zi = rng.choice(len(z), size=m, replace=False)
        d2 = np.sum((z[zi][:, None, :] - z[zi][None, :, :]) ** 2, axis=-1)
        med = np.median(d2[d2 > 0])
        gamma = 1.0 / med if med > 0 else 1.0
    def k(a, b):
        d2 = np.sum((a[:, None, :] - b[None, :, :]) ** 2, axis=-1)
        return np.exp(-gamma * d2)
    return float(k(x, x).mean() + k(y, y).mean() - 2 * k(x, y).mean()), float(gamma)

PER_CLASS_MMD_MIN_N = 30  # Minimum N on each side for per-class RBF MMD to be trustworthy

feats = {s: gen_results[s]["features"] for s in eval_loaders}

print("Whole-split feature MMD (RBF) and mean distance:")
pair_rows = []
for a, b in [("hpo_train", "hpo_val"), ("hpo_train", "official_test"), ("hpo_val", "official_test")]:
    md = feat_mean_dist_sq(feats[a], feats[b])
    rbf, gamma = rbf_mmd_sq(feats[a], feats[b])
    pair_rows.append({"pair": f"{a} vs {b}", "feat_mean_dist_sq": md,
                      "rbf_mmd_sq": rbf, "rbf_gamma": gamma})
    print(f"  {a:>14} vs {b:<14}  feat_mean_dist^2={md:.4f}   RBF_MMD^2={rbf:.4f}  (gamma={gamma:.3e})")
pd.DataFrame(pair_rows).to_csv(os.path.join(GEN_DIR, "penultimate_mmd_pairs.csv"), index=False)

print(f"\nPer-class feature MMD, hpo_train vs official_test (classes with min(n) < {PER_CLASS_MMD_MIN_N} shown as NaN):")
per_rows = []
for c in range(NUM_CLASSES):
    tr = gen_results["hpo_train"]["labels"] == c
    te = gen_results["official_test"]["labels"] == c
    ntr, nte = int(tr.sum()), int(te.sum())
    rel = min(ntr, nte) >= PER_CLASS_MMD_MIN_N
    row = {"class": CLASS_NAMES[c], "hpo_train_n": ntr, "test_n": nte,
           "reliable_rbf": bool(rel),
           "hpo_val_acc": gen_results["hpo_val"]["per_class_acc"][c],
           "test_acc":    gen_results["official_test"]["per_class_acc"][c]}
    if ntr >= 2 and nte >= 2:
        row["feat_mean_dist_sq"] = feat_mean_dist_sq(feats["hpo_train"][tr], feats["official_test"][te])
    else:
        row["feat_mean_dist_sq"] = np.nan
    if rel:
        rbf, _ = rbf_mmd_sq(feats["hpo_train"][tr], feats["official_test"][te], max_samples=256, seed=c)
        row["rbf_mmd_sq"] = rbf
    else:
        row["rbf_mmd_sq"] = np.nan
    per_rows.append(row)
pc_mmd = pd.DataFrame(per_rows).sort_values("rbf_mmd_sq", ascending=False, na_position="last")
pc_mmd.to_csv(os.path.join(GEN_DIR, "per_class_penultimate_mmd.csv"), index=False)
display(pc_mmd.style.format({
    "feat_mean_dist_sq": "{:.4f}", "rbf_mmd_sq": "{:.4f}",
    "hpo_val_acc": "{:.3f}", "test_acc": "{:.3f}",
}).background_gradient(subset=["rbf_mmd_sq"], cmap="Reds"))

# Make the rare-class limitation explicit: list which classes lack reliable RBF MMD
_excluded_rbf = pc_mmd[~pc_mmd["reliable_rbf"]].copy()
if len(_excluded_rbf) > 0:
    print(f"\nClasses excluded from reliable RBF MMD (min(N_train, N_test) < {PER_CLASS_MMD_MIN_N}):")
    for _, _row in _excluded_rbf.iterrows():
        print(f"  {_row['class']:<18}  hpo_train_n={int(_row['hpo_train_n']):>4}  "
              f"test_n={int(_row['test_n']):>4}  "
              f"(feat_mean_dist^2={_row['feat_mean_dist_sq']:.4f} — use with caution)")
    print(f"=> {len(_excluded_rbf)}/{NUM_CLASSES} classes have NO reliable feature-drift evidence; "
          f"for these, reason from per-class accuracy + sample-size only.")
else:
    print(f"\nAll {NUM_CLASSES} classes clear the reliable-MMD threshold (min N >= {PER_CLASS_MMD_MIN_N}).")

In [ ]:
# --- 19.5 Diagnostic (a): per-class accuracy gap a_c - b_c, ranked ---
# Interpretation:
#   |gap| large AND N_val tiny  -> noise / HPO-selection story
#   |gap| large AND N_val large -> genuine per-class generalization failure
# The `reliability` column is min(N_val, N_test). Rows with reliability < RELIABILITY_MIN
# are flagged as noise-dominated: they contribute to MCA but the per-class estimate is noisy.
RELIABILITY_MIN = 30

val_acc  = gen_results["hpo_val"]["per_class_acc"]
test_acc = gen_results["official_test"]["per_class_acc"]
val_n    = gen_results["hpo_val"]["per_class_n"]
test_n   = gen_results["official_test"]["per_class_n"]

gap_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "val_acc":  val_acc,
    "test_acc": test_acc,
    "gap_val_minus_test": val_acc - test_acc,
    "N_val": val_n,
    "N_test": test_n,
    "reliability": np.minimum(val_n, test_n),
})
gap_df["reliable"] = gap_df["reliability"] >= RELIABILITY_MIN
gap_df["abs_gap"] = np.abs(gap_df["gap_val_minus_test"])
gap_df = gap_df.sort_values("abs_gap", ascending=False).drop(columns="abs_gap")
gap_df.to_csv(os.path.join(GEN_DIR, "diag_a_per_class_gap.csv"), index=False)

print("Diagnostic (a): per-class val-test gap, ranked by |gap|")
print(f"  reliable = min(N_val, N_test) >= {RELIABILITY_MIN}")
display(gap_df.style.format({
    "val_acc": "{:.3f}", "test_acc": "{:.3f}", "gap_val_minus_test": "{:+.3f}",
}).background_gradient(subset=["gap_val_minus_test"], cmap="RdYlGn_r")
   .background_gradient(subset=["reliability"], cmap="Greens"))

# MCA-gap contribution: each class contributes (1/K)(a_c - b_c).
# Top-5 split into reliable vs low-N so you don't act on noise.
mca_gap = gen_results["hpo_val"]["mca"] - gen_results["official_test"]["mca"]
contrib = (val_acc - test_acc) / NUM_CLASSES
reliability = np.minimum(val_n, test_n)
reliable_mask = reliability >= RELIABILITY_MIN

print(f"\nOverall MCA gap (val - test): {mca_gap*100:+.2f}pp")

def show_top(mask, label, k=5):
    if mask.sum() == 0:
        print(f"{label}: (none)"); return
    order = np.argsort(-np.abs(contrib * mask))
    print(label)
    shown = 0
    for c in order:
        if not mask[c]:
            continue
        print(f"  {CLASS_NAMES[c]:<18}  contrib={contrib[c]*100:+.2f}pp  "
              f"(val_n={int(val_n[c])}, test_n={int(test_n[c])}, reliability={int(reliability[c])})")
        shown += 1
        if shown >= k:
            break

show_top(reliable_mask,  "\nTop reliable contributors (signal):")
show_top(~reliable_mask, "\nTop low-N contributors (noise-dominated, interpret with caution):")

In [ ]:
# --- 19.6 Diagnostic (b): synthetic test accuracy (prior-shift test) ---
# synthetic_test_acc = Sigma_c (N_test_c / N_test) * a_c
#   a_c = per-class accuracy measured on hpo_val
# If synthetic ~= actual test accuracy -> prior shift is the whole story for OVERALL acc
# If synthetic << actual              -> residual per-class generalization failure
#
# Uncertainty comes from noisy a_c for small val_n classes. We stratified-bootstrap
# val per-class (resample within each class) and propagate to a synthetic-acc CI.
N_BOOT_B = 400
v_labels = gen_results["hpo_val"]["labels"]
v_preds  = gen_results["hpo_val"]["preds"]
per_class_val_idx = [np.where(v_labels == c)[0] for c in range(NUM_CLASSES)]

N_test_total = test_n.sum()
test_priors  = test_n / N_test_total
synthetic_test_acc = float(np.sum(test_priors * np.nan_to_num(val_acc, nan=0.0)))
actual_test_acc    = gen_results["official_test"]["acc"]
val_actual_acc     = gen_results["hpo_val"]["acc"]

rng_b = np.random.default_rng(0)
synth_samples = np.zeros(N_BOOT_B)
for b in range(N_BOOT_B):
    pca_b = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        idxs = per_class_val_idx[c]
        if len(idxs) == 0:
            pca_b[c] = 0.0  # prior-weighted; test_priors[c] ~= 0 for absent classes
            continue
        samp = rng_b.choice(idxs, size=len(idxs), replace=True)
        pca_b[c] = float((v_preds[samp] == c).mean())
    synth_samples[b] = float(np.sum(test_priors * pca_b))
synth_lo = float(np.percentile(synth_samples, 2.5))
synth_hi = float(np.percentile(synth_samples, 97.5))

# Inverse-direction sanity: val-prior-weighted test per-class accuracies
N_val_total  = val_n.sum()
val_priors   = val_n / max(N_val_total, 1)
reverse_check = float(np.sum(val_priors * np.nan_to_num(test_acc, nan=0.0)))

delta_pp = (synthetic_test_acc - actual_test_acc) * 100
# Half-width of CI as 1-sigma-ish uncertainty in pp
unc_pp = (synth_hi - synth_lo) * 100 / 2

diag_b = {
    "val_MCA":                         gen_results["hpo_val"]["mca"],
    "test_MCA":                        gen_results["official_test"]["mca"],
    "val_overall_accuracy":            val_actual_acc,
    "test_overall_accuracy":           actual_test_acc,
    "synthetic_test_accuracy":         synthetic_test_acc,
    "synthetic_test_acc_ci95":         [synth_lo, synth_hi],
    "synthetic_minus_actual_test_pp":  delta_pp,
    "uncertainty_halfwidth_pp":        unc_pp,
    "reverse_val_prior_weighted":      reverse_check,
}
print("Diagnostic (b): prior-shift decomposition of overall accuracy")
for k, v in diag_b.items():
    if isinstance(v, list):
        print(f"  {k:<32} = [{v[0]*100:.2f}%, {v[1]*100:.2f}%]")
    elif isinstance(v, float) and k.endswith("_pp"):
        print(f"  {k:<32} = {v:+.2f}pp")
    elif isinstance(v, float):
        print(f"  {k:<32} = {v*100:.2f}%")

print("\nInterpretation (uncertainty-aware):")
if synth_lo <= actual_test_acc <= synth_hi:
    print(f"  synthetic = {synthetic_test_acc*100:.2f}% [CI {synth_lo*100:.2f}%-{synth_hi*100:.2f}%]  "
          f"brackets actual test acc ({actual_test_acc*100:.2f}%)")
    print("  => prior shift alone explains the overall accuracy gap within noise")
else:
    residual = delta_pp
    print(f"  synthetic - actual = {delta_pp:+.2f}pp ({synth_lo*100:.2f}%-{synth_hi*100:.2f}% CI)")
    if residual > unc_pp:
        print(f"  => residual generalization failure ~ {delta_pp - unc_pp:+.1f}-{delta_pp + unc_pp:+.1f}pp beyond prior shift")
    else:
        print(f"  => val appears pessimistic vs test per-class (unusual; check class coverage)")

with open(os.path.join(GEN_DIR, "diag_b_prior_shift.json"), "w") as f:
    json.dump(diag_b, f, indent=2)

In [ ]:
# --- 19.7 Diagnostic (c): clean-test subset (scenes not in hpo_train) ---
# The official SUN RGB-D split leaks ~26 scenes between train and test;
# ~8.5% of test samples share a scene with training and may be "easier."
# Re-run test metrics on the clean subset only.
# Load train scene_groups locally so this cell is independent of cell 19 scope
with open(os.path.join(DATASET_CONFIG['data_root'], 'train', 'scene_groups.json')) as _f:
    _train_scene_groups = json.load(_f)
hpo_train_scenes_set = {_train_scene_groups[i] for i in train_indices}
test_clean_mask  = np.array([s not in hpo_train_scenes_set for s in test_scene_groups])
test_leaked_mask = ~test_clean_mask

t_labels = gen_results["official_test"]["labels"]
t_preds  = gen_results["official_test"]["preds"]

def subset_metrics(mask, tag):
    if mask.sum() == 0:
        return None
    y = t_labels[mask]; p = t_preds[mask]
    acc = float((p == y).mean())
    pca = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        m = y == c
        pca[c] = float((p[m] == c).mean()) if m.any() else np.nan
    mca = float(np.nanmean(pca))
    print(f"  {tag:<14} n={int(mask.sum()):>5}  acc={acc*100:.2f}%  mca={mca*100:.2f}%")
    return {"tag": tag, "n": int(mask.sum()), "acc": acc, "mca": mca}

print("Diagnostic (c): scene-leakage decomposition of official test")
rows_c = [
    subset_metrics(np.ones_like(t_labels, dtype=bool), "full_test"),
    subset_metrics(test_clean_mask, "clean_test"),
    subset_metrics(test_leaked_mask, "leaked_test"),
]
leak_df = pd.DataFrame([r for r in rows_c if r])
leak_df.to_csv(os.path.join(GEN_DIR, "diag_c_scene_leakage.csv"), index=False)

leakage_inflation_acc = (rows_c[2]['acc'] - rows_c[1]['acc']) * 100 if rows_c[1] and rows_c[2] else 0.0
leakage_inflation_mca = (rows_c[2]['mca'] - rows_c[1]['mca']) * 100 if rows_c[1] and rows_c[2] else 0.0
print(f"\nLeaked-test minus clean-test:  acc={leakage_inflation_acc:+.2f}pp  mca={leakage_inflation_mca:+.2f}pp")
if rows_c[1]:
    true_gap = (gen_results['hpo_val']['mca'] - rows_c[1]['mca']) * 100
    print(f"True val<->clean-test MCA gap (post-scene-leakage correction): {true_gap:+.2f}pp")

In [ ]:
# --- 19.8 Diagnostic (d): stratified bootstrap 95% CI on hpo_val MCA ---
# Stratified resampling: for each class, bootstrap within its val indices to
# keep N_c fixed across iterations. This avoids the unstratified-bootstrap bias
# where iterations that happen to draw 0 samples for small classes silently
# drop those classes from MCA (nanmean), making the distribution look tighter
# than the underlying variance warrants.
BOOTSTRAP_ITERS = 1000
rng = np.random.default_rng(0)
# per_class_val_idx already computed in 19.6, but recompute defensively
per_class_val_idx = [np.where(v_labels == c)[0] for c in range(NUM_CLASSES)]

mca_samples = np.zeros(BOOTSTRAP_ITERS)
for b in range(BOOTSTRAP_ITERS):
    pca = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        idxs = per_class_val_idx[c]
        if len(idxs) == 0:
            pca[c] = np.nan
            continue
        samp = rng.choice(idxs, size=len(idxs), replace=True)
        pca[c] = float((v_preds[samp] == c).mean())
    mca_samples[b] = float(np.nanmean(pca))

mean_mca = float(mca_samples.mean())
lo = float(np.percentile(mca_samples, 2.5))
hi = float(np.percentile(mca_samples, 97.5))

print("Diagnostic (d): stratified bootstrap 95% CI on hpo_val MCA")
print(f"  point estimate (full val): {gen_results['hpo_val']['mca']*100:.2f}%")
print(f"  bootstrap mean:            {mean_mca*100:.2f}%")
print(f"  95% CI:                    [{lo*100:.2f}%, {hi*100:.2f}%]   (width {(hi-lo)*100:.2f}pp)")
print(f"  official_test MCA:         {gen_results['official_test']['mca']*100:.2f}%")
print(f"  gap (val point - test):    {(gen_results['hpo_val']['mca']-gen_results['official_test']['mca'])*100:+.2f}pp")
print(f"  gap (val 95% lower - test):{(lo-gen_results['official_test']['mca'])*100:+.2f}pp")

if (gen_results['official_test']['mca'] >= lo) and (gen_results['official_test']['mca'] <= hi):
    print("\n  [test MCA inside val 95% CI -> val is a plausible noisy estimate of test MCA]")
else:
    print("\n  [test MCA OUTSIDE val 95% CI -> real distribution/generalization difference]")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mca_samples, bins=40, alpha=0.8)
ax.axvline(gen_results["hpo_val"]["mca"], color="C1", linestyle="--", label="val point estimate")
ax.axvline(gen_results["official_test"]["mca"], color="C3", linestyle="--", label="test MCA")
ax.axvline(lo, color="gray", linestyle=":")
ax.axvline(hi, color="gray", linestyle=":", label="95% CI")
ax.set_xlabel("hpo_val MCA (stratified bootstrap)"); ax.set_ylabel("count")
ax.set_title(f"hpo_val MCA stratified bootstrap ({BOOTSTRAP_ITERS} iters)"); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(GEN_DIR, "diag_d_val_mca_bootstrap.png"), dpi=120)
plt.show()

In [ ]:
# --- 19.9 Headline scatter: per-class accuracy gap vs per-class feature MMD ---
# One chart summarizing the story: do the classes with the biggest feature drift
# also have the biggest accuracy gap? Point size ~ val_n (small = noisy).
# Correlation is computed only on reliable classes (val_n >= RELIABILITY_MIN).
from scipy.stats import spearmanr

xs, ys, names_arr, val_ns_arr, rel_arr = [], [], [], [], []
for _, row in pc_mmd.iterrows():
    c = CLASS_NAMES.index(row["class"])
    val_acc_c = gen_results["hpo_val"]["per_class_acc"][c]
    test_acc_c = gen_results["official_test"]["per_class_acc"][c]
    val_nc = int(gen_results["hpo_val"]["per_class_n"][c])
    mmd = row["rbf_mmd_sq"] if not np.isnan(row["rbf_mmd_sq"]) else row["feat_mean_dist_sq"]
    if np.isnan(mmd) or np.isnan(val_acc_c) or np.isnan(test_acc_c):
        continue
    xs.append(mmd)
    ys.append(val_acc_c - test_acc_c)
    names_arr.append(row["class"])
    val_ns_arr.append(val_nc)
    rel_arr.append(bool(row["reliable_rbf"]))

xs = np.array(xs); ys = np.array(ys); val_ns_arr = np.array(val_ns_arr); rel_arr = np.array(rel_arr)

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(xs, ys, s=20 + 3 * val_ns_arr, alpha=0.65,
                c=val_ns_arr, cmap="viridis", edgecolors="k", linewidth=0.5)
# Mark unreliable (small-n or mean-distance-only) with hollow ring overlay
for i, (x_, y_, nm, rel) in enumerate(zip(xs, ys, names_arr, rel_arr)):
    ax.annotate(nm, (x_, y_), fontsize=8, alpha=0.8, xytext=(4, 4), textcoords="offset points")
    if not rel:
        ax.scatter([x_], [y_], s=30 + 3 * val_ns_arr[i], facecolors="none",
                   edgecolors="red", linewidth=1.2)
ax.axhline(0, color="gray", alpha=0.3); ax.axvline(0, color="gray", alpha=0.3)
ax.set_xlabel("per-class feature MMD  (RBF; falls back to mean-distance when N<30)")
ax.set_ylabel("per-class accuracy gap  (val - test)")
ax.set_title("Per-class feature drift vs accuracy gap\n(point size ~ val_n; red ring = unreliable MMD)")
plt.colorbar(sc, ax=ax, label="val_n")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(GEN_DIR, "scatter_gap_vs_mmd.png"), dpi=120)
plt.show()

if rel_arr.sum() >= 3:
    rho, pval = spearmanr(xs[rel_arr], ys[rel_arr])
    print(f"Spearman correlation (reliable classes only, n={int(rel_arr.sum())}): "
          f"rho={rho:+.3f}  p={pval:.3f}")
    if rho > 0.3 and pval < 0.1:
        print("  => classes with larger feature drift also have larger accuracy gap (feature-drift story supported)")
    elif rho < -0.1:
        print("  => negative correlation (feature drift does NOT predict accuracy gap — reconsider hypothesis)")
    else:
        print("  => weak/no correlation (feature drift alone doesn't explain gaps; consider other causes)")
else:
    print("Too few reliable classes for correlation test.")

In [ ]:
# --- 19.10 Combined per-class diagnostics table (single reference) ---
# Merges accuracy + sample-size + feature-drift columns. Sorted so reliable rows
# come first (min(N_val, N_test) >= RELIABILITY_MIN), then by test_acc ascending
# within each reliability group: worst-performing reliable classes at the top,
# then the lowest-test-acc unreliable classes. Use this as the "what is failing
# and why" reference.
combined = gap_df[["class", "val_acc", "test_acc", "gap_val_minus_test",
                   "N_val", "N_test", "reliability", "reliable"]].copy()
_mmd_subset = pc_mmd[["class", "feat_mean_dist_sq", "rbf_mmd_sq", "reliable_rbf"]]
combined = combined.merge(_mmd_subset, on="class", how="left")
combined = combined.sort_values(
    by=["reliable", "test_acc"], ascending=[False, True]
).reset_index(drop=True)
combined.to_csv(os.path.join(GEN_DIR, "combined_per_class.csv"), index=False)

print("Combined per-class diagnostics (reliable classes on top, sorted by test_acc asc within group):")
display(combined.style.format({
    "val_acc": "{:.3f}", "test_acc": "{:.3f}", "gap_val_minus_test": "{:+.3f}",
    "feat_mean_dist_sq": "{:.4f}", "rbf_mmd_sq": "{:.4f}",
}).background_gradient(subset=["test_acc"], cmap="RdYlGn")
   .background_gradient(subset=["gap_val_minus_test"], cmap="RdYlGn_r")
   .background_gradient(subset=["rbf_mmd_sq"], cmap="Reds"))

# Quick read-out: intersection of "worst test acc" AND "large feature drift" AND reliable
_reliable = combined[combined["reliable"] & combined["reliable_rbf"]]
if len(_reliable) > 0:
    print("\nReliable classes ranked by test_acc (worst first):")
    for _, r in _reliable.head(5).iterrows():
        print(f"  {r['class']:<18}  test_acc={r['test_acc']*100:5.1f}%  "
              f"gap={r['gap_val_minus_test']*100:+5.1f}pp  "
              f"rbf_mmd={r['rbf_mmd_sq']:.3f}  (N_val={int(r['N_val'])}, N_test={int(r['N_test'])})")

_unreliable = combined[~combined["reliable"]]
if len(_unreliable) > 0:
    print("\nUnreliable (low-N) classes — evidence from accuracy + N only, not feature drift:")
    for _, r in _unreliable.head(5).iterrows():
        print(f"  {r['class']:<18}  test_acc={r['test_acc']*100:5.1f}%  "
              f"gap={r['gap_val_minus_test']*100:+5.1f}pp  "
              f"(N_val={int(r['N_val'])}, N_test={int(r['N_test'])})")

In [ ]:
# --- 19.11 Summary JSON ---
summary_gen = {
    "checkpoint_dir": checkpoint_dir,
    "seed": SEED,
    "n_samples": {s: int(len(gen_results[s]["labels"])) for s in eval_loaders},
    "per_split_metrics": {
        s: {"loss": gen_results[s]["loss"],
            "accuracy": gen_results[s]["acc"],
            "mca": gen_results[s]["mca"]} for s in eval_loaders
    },
    "diag_b_prior_shift":        diag_b,
    "diag_c_scene_leakage":      [r for r in rows_c if r],
    "diag_d_val_mca_bootstrap":  {"mean": mean_mca, "ci95": [lo, hi],
                                  "point_estimate": gen_results["hpo_val"]["mca"],
                                  "method": "stratified"},
}
with open(os.path.join(GEN_DIR, "summary.json"), "w") as f:
    json.dump(summary_gen, f, indent=2)
print(json.dumps(summary_gen, indent=2))
print(f"\nAll generalization diagnostics written to {GEN_DIR}")

In [ ]:
# --- 19.13 Multi-modal hypothesis diagnostic (retroactive validation of max-out motivation) ---
# ADR-002 motivated MaxoutHead with the claim: "classes have multi-modal visual
# structure (bars vs. corridor dining areas, RGB vs. depth sensor differences)
# that a single linear hyperplane can't separate." The cell 19.12 entropy
# results + marginal max-out gains (+0.5pp) suggest that claim may not hold.
# This cell directly tests the hypothesis with three feature-space diagnostics
# on hpo_train penultimate features (already captured in gen_results).
#
# All three use the hook features from cell 19.1 (penultimate, N_train x 384).
# L2-normalize before clustering so Euclidean distance ≈ cosine distance
# (standard for neural features; magnitude-invariant).
#
# Interpretation guide:
#   Metric 1 (K-means silhouette per class):
#     >0.3 avg  → strong multi-modal structure within classes
#     <0.1 avg  → classes are essentially unimodal
#   Metric 2 (intra-class vs inter-class distance ratio):
#     <0.5      → classes well-separated and tight (unimodal + separable)
#     >1.0      → classes spread out OR inter-class overlap
#   Metric 3 (max sensor-centroid spread / class intra-var):
#     >1.0      → sensor IS the dominant axis of within-class variance
#                 (max-out-via-sensor motivation validated)
#     <0.3      → sensor doesn't meaningfully condition features
import json
import os
from collections import defaultdict

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def _clean_name_1913(raw):
    return raw.split(': ', 1)[1] if ': ' in raw else raw
_clean_names = [_clean_name_1913(n) for n in CLASS_NAMES]

feats = gen_results['hpo_train']['features']                    # (N, 384)
labels = gen_results['hpo_train']['labels']                     # (N,)
# Sensor per position — gen_results['hpo_train'] iterates train_indices in order
with open(os.path.join(DATASET_CONFIG['data_root'], 'train', 'sample_paths.json')) as _f:
    _train_paths_all_1913 = json.load(_f)
hpo_train_sensors_1913 = np.array(
    [_train_paths_all_1913[i].split('/', 1)[0] for i in train_indices]
)

# L2-normalize features (cosine geometry on the unit hypersphere)
feats_n = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-12)

# ---------------------------------------------------------------------------
# Metric 1: Per-class K-means silhouette at K=2 and K=3
# ---------------------------------------------------------------------------
print("=== Metric 1: Per-class K-means silhouette score ===")
print("  >0.3 = strong multi-modal structure; <0.1 = unimodal; 0.1-0.3 = ambiguous\n")
print(f"  {'class':<22}{'N':>6}{'silhouette_K=2':>18}{'silhouette_K=3':>18}")
sil_rows = []
for c in range(NUM_CLASSES):
    mask = labels == c
    X_c = feats_n[mask]
    if X_c.shape[0] < 20:
        continue
    km2 = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(X_c)
    km3 = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit(X_c)
    s2 = silhouette_score(X_c, km2.labels_)
    s3 = silhouette_score(X_c, km3.labels_)
    sil_rows.append({'class': _clean_names[c], 'n': int(X_c.shape[0]),
                     's_k2': float(s2), 's_k3': float(s3)})
    print(f"  {_clean_names[c]:<22}{X_c.shape[0]:>6}{s2:>18.3f}{s3:>18.3f}")

mean_s2 = float(np.mean([r['s_k2'] for r in sil_rows]))
mean_s3 = float(np.mean([r['s_k3'] for r in sil_rows]))
print(f"\n  Mean silhouette: K=2={mean_s2:.3f}  K=3={mean_s3:.3f}")
if mean_s2 > 0.3:
    verdict_1 = "STRONG multi-modality → max-out hypothesis supported"
elif mean_s2 < 0.1:
    verdict_1 = "WEAK/NO multi-modality → max-out hypothesis NOT supported"
else:
    verdict_1 = "AMBIGUOUS"
print(f"  Verdict: {verdict_1}")

# ---------------------------------------------------------------------------
# Metric 2: Intra-class vs inter-class distance ratio
# ---------------------------------------------------------------------------
# [silenced — 19.15 per-class failure-mode table is the authoritative view; computations still run for downstream vars]
# print(f"\n=== Metric 2: Intra-class vs inter-class distance ===")
class_centroids_1913 = {}
intra_var_1913 = {}
for c in range(NUM_CLASSES):
    mask = labels == c
    X_c = feats_n[mask]
    if X_c.shape[0] == 0:
        continue
    mu = X_c.mean(axis=0)
    class_centroids_1913[c] = mu
    intra_var_1913[c] = float(np.mean(np.linalg.norm(X_c - mu, axis=1)))

centroid_mat = np.array([class_centroids_1913[c] for c in sorted(class_centroids_1913)])
inter_dists = []
for i in range(len(centroid_mat)):
    for j in range(i + 1, len(centroid_mat)):
        inter_dists.append(float(np.linalg.norm(centroid_mat[i] - centroid_mat[j])))
mean_intra = float(np.mean(list(intra_var_1913.values())))
mean_inter = float(np.mean(inter_dists))
ratio = mean_intra / mean_inter if mean_inter > 0 else float('inf')
# print(f"  Mean within-class distance (samples -> own centroid): {mean_intra:.3f}")
# print(f"  Mean between-class distance (centroid -> centroid):   {mean_inter:.3f}")
# print(f"  Ratio (intra / inter): {ratio:.3f}")
if ratio < 0.5:
    verdict_2 = "Classes are tight + separable → UNIMODAL, good boundaries"
elif ratio > 1.0:
    verdict_2 = "Classes are spread out like between-class gaps → MULTI-MODAL or INTER-CLASS OVERLAP"
else:
    verdict_2 = "Moderate intra-class spread; hybrid regime"
# print(f"  Verdict: {verdict_2}")

# ---------------------------------------------------------------------------
# Metric 3: Per-(class x sensor) centroid spread vs class intra-variance
# ---------------------------------------------------------------------------
print(f"\n=== Metric 3: Sensor-conditional feature modes per class ===")
print(f"  ratio > 1 → sensor dominates within-class variance (the max-out-via-sensor story)")
print(f"  ratio < 0.3 → sensor barely matters\n")
print(f"  {'class':<22}{'max_sensor_centroid_dist':>28}{'class_intra_var':>18}{'ratio':>10}")
sensor_ratio_rows = []
for c in range(NUM_CLASSES):
    mask = labels == c
    if mask.sum() < 10:
        continue
    X_c = feats_n[mask]
    s_c = hpo_train_sensors_1913[mask]
    sensor_centroids = {}
    for s in np.unique(s_c):
        if (s_c == s).sum() >= 5:
            sensor_centroids[s] = X_c[s_c == s].mean(axis=0)
    if len(sensor_centroids) < 2:
        continue
    sc_list = list(sensor_centroids.values())
    max_sensor_dist = max(
        float(np.linalg.norm(sc_list[i] - sc_list[j]))
        for i in range(len(sc_list)) for j in range(i + 1, len(sc_list))
    )
    cls_intra = intra_var_1913.get(c, 0)
    ratio_c = (max_sensor_dist / cls_intra) if cls_intra > 0 else float('inf')
    sensor_ratio_rows.append({
        'class': _clean_names[c],
        'max_sensor_centroid_dist': max_sensor_dist,
        'class_intra_var': cls_intra,
        'ratio': ratio_c,
        'num_sensors': len(sensor_centroids),
    })
    print(f"  {_clean_names[c]:<22}{max_sensor_dist:>28.3f}{cls_intra:>18.3f}{ratio_c:>10.3f}")

mean_sensor_ratio = float(np.mean([r['ratio'] for r in sensor_ratio_rows]))
print(f"\n  Mean sensor-mode ratio across classes: {mean_sensor_ratio:.3f}")
if mean_sensor_ratio > 1.0:
    verdict_3 = "Sensor IS the dominant axis of within-class variance → max-out-via-sensor story VALIDATED"
elif mean_sensor_ratio < 0.3:
    verdict_3 = "Sensor barely conditions features → max-out-via-sensor story NOT supported"
else:
    verdict_3 = "Moderate sensor conditioning; partial support for max-out-via-sensor"
print(f"  Verdict: {verdict_3}")

# ---------------------------------------------------------------------------
# Summary + persist
# ---------------------------------------------------------------------------
print(f"\n=== Overall verdict ===")
print(f"  Metric 1 (silhouette):     {verdict_1}")
print(f"  Metric 2 (intra/inter):    {verdict_2}")
print(f"  Metric 3 (sensor ratio):   {verdict_3}")

_diag_out = {
    'silhouette_per_class': sil_rows,
    'mean_silhouette_k2': mean_s2,
    'mean_silhouette_k3': mean_s3,
    'intra_vs_inter': {
        'mean_intra': mean_intra,
        'mean_inter': mean_inter,
        'ratio': ratio,
    },
    'sensor_mode_ratio_per_class': sensor_ratio_rows,
    'mean_sensor_mode_ratio': mean_sensor_ratio,
    'verdicts': {'metric_1': verdict_1, 'metric_2': verdict_2, 'metric_3': verdict_3},
}
_out_path = os.path.join(GEN_DIR, 'multimodal_diagnostic.json')
with open(_out_path, 'w') as _f:
    json.dump(_diag_out, _f, indent=2)
print(f"\nSaved → {_out_path}")


In [ ]:
# --- 19.14 Multi-modal hypothesis: NN test→train match + scenario synthesis ---
# Extends 19.13 with three additional signals:
#   Metric 4 — within-class sensor dist vs between-class dist (complementary to
#              Metric 3 in 19.13; compares sensor variance to the actual class-
#              boundary scale, not internal noise scale).
#   Metric 5 — nearest-neighbor test→train class match rate. Discriminates
#              "classifier fails on correctly-located features" (high NN match,
#              low test acc) from "features are in wrong region" (low NN match).
#   Metric 6 — train vs test centroid drift per class. Tests Scenario C:
#              features are unimodal per class in each split but train/test
#              distributions don't overlap.
#   t-SNE on 3 hypothesis + 2 baseline classes, colored by sensor (visual).
# Final section synthesizes all verdicts into one of four scenarios.
#
# Requires cell 19.13 to have run in the same kernel (reads sil_rows, mean_s2,
# mean_s3, mean_intra, mean_inter, ratio, mean_sensor_ratio, intra_var_1913,
# class_centroids_1913, _clean_names, feats_n, hpo_train_sensors_1913).
import itertools
import json as _json_14
import os as _os_14

import numpy as _np_14
import matplotlib.pyplot as _plt_14
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

assert 'sil_rows' in dir(), "Run cell 19.13 first — this cell reads its variables."

# ---------------------------------------------------------------------------
# Metric 4: within-class sensor dist vs between-class centroid dist
# ---------------------------------------------------------------------------
# Complement to 19.13's Metric 3. Instead of normalizing sensor-centroid spread
# by internal class variance, normalize by the scale of class separation. If
# sensor variance within a class approaches the scale separating classes,
# sensor is a structurally important axis; if it's <20% of class separation,
# sensor is a minor perturbation.
# [silenced — 19.15 View 1 per-sensor MCA spread is the actionable form of this signal; computations still run for Metric 6 (needs mean_between_class) and JSON persistence]
# print("=== Metric 4: Within-class sensor distances vs between-class distances ===")
labels_14 = gen_results['hpo_train']['labels']
# Per-(class, sensor) centroids (from L2-normalized features computed in 19.13)
class_sensor_centroids = {}
for c in range(NUM_CLASSES):
    mask_c = labels_14 == c
    if mask_c.sum() < 10:
        continue
    class_sensor_centroids[c] = {}
    s_c = hpo_train_sensors_1913[mask_c]
    X_c = feats_n[mask_c]
    for s in _np_14.unique(s_c):
        if (s_c == s).sum() >= 5:
            class_sensor_centroids[c][s] = X_c[s_c == s].mean(axis=0)

within_sensor_dists = []
for c, per_sensor in class_sensor_centroids.items():
    for s1, s2 in itertools.combinations(sorted(per_sensor.keys()), 2):
        d = float(_np_14.linalg.norm(per_sensor[s1] - per_sensor[s2]))
        within_sensor_dists.append(d)

between_class_dists = []
cs = sorted(class_centroids_1913)
for i, j in itertools.combinations(range(len(cs)), 2):
    d = float(_np_14.linalg.norm(class_centroids_1913[cs[i]] - class_centroids_1913[cs[j]]))
    between_class_dists.append(d)

mean_within_sensor = float(_np_14.mean(within_sensor_dists)) if within_sensor_dists else 0.0
mean_between_class = float(_np_14.mean(between_class_dists))
sensor_vs_class_frac = mean_within_sensor / mean_between_class if mean_between_class > 0 else float('inf')
# print(f"  Mean within-class sensor-centroid distance:  {mean_within_sensor:.4f}")
# print(f"  Mean between-class centroid distance:        {mean_between_class:.4f}")
# print(f"  Fraction (sensor / class-separation):        {sensor_vs_class_frac:.3f}")
if sensor_vs_class_frac > 0.5:
    verdict_4 = "Sensor modes are ~half the scale of class separation → sensor is a STRUCTURAL axis"
elif sensor_vs_class_frac < 0.2:
    verdict_4 = "Sensor modes are <20% of class separation → sensor is a MINOR perturbation"
else:
    verdict_4 = "Moderate sensor contribution relative to class separation"
# print(f"  Verdict: {verdict_4}")

# ---------------------------------------------------------------------------
# Metric 5: Nearest-neighbor test→train class match rate
# ---------------------------------------------------------------------------
# For each test sample, find its nearest train sample in feature space. If the
# train NN has the same class, features are "in the right region"; if not,
# test features have drifted out of their training-class region. This directly
# separates classifier-boundary problems from feature-location problems.
print(f"\n=== Metric 5: Nearest-neighbor test→train class match ===")
train_feats_n = feats_n  # (N_train, 384), L2-normalized from 19.13
train_labels_n = labels_14
test_feats = gen_results['official_test']['features']
test_feats_n = test_feats / (_np_14.linalg.norm(test_feats, axis=1, keepdims=True) + 1e-12)
test_labels_n = gen_results['official_test']['labels']

nn = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(train_feats_n)
_, idx = nn.kneighbors(test_feats_n)
test_nn_class = train_labels_n[idx.flatten()]
nn_match_agg = float((test_nn_class == test_labels_n).mean())
print(f"  Aggregate NN match rate: {nn_match_agg*100:.1f}%")
print(f"  (test acc was {gen_results['official_test']['acc']*100:.1f}% for reference)\n")
print(f"  {'class':<22}{'N_test':>8}{'NN_match':>12}{'test_acc':>12}{'gap':>10}")
per_class_nn_rows = []
for c in range(NUM_CLASSES):
    m = test_labels_n == c
    if m.sum() == 0:
        continue
    match = float((test_nn_class[m] == c).mean())
    test_acc_c = float(gen_results['official_test']['per_class_acc'][c])
    gap = match - test_acc_c  # if high, classifier lags NN; if low, features are in wrong region
    per_class_nn_rows.append({
        'class': _clean_names[c], 'n_test': int(m.sum()),
        'nn_match': match, 'test_acc': test_acc_c, 'classifier_gap': gap,
    })
    print(f"  {_clean_names[c]:<22}{int(m.sum()):>8}{match*100:>11.1f}%{test_acc_c*100:>11.1f}%{gap*100:>+9.1f}pp")

# Hypothesis-class NN match (the classes max-out was supposed to help)
HYP_CLASSES = ['library', 'dining_area', 'classroom', 'office', 'bedroom', 'study_space']
hyp_nn_match = float(_np_14.mean([
    r['nn_match'] for r in per_class_nn_rows if r['class'] in HYP_CLASSES
]))
baseline_nn_match = float(_np_14.mean([
    r['nn_match'] for r in per_class_nn_rows if r['class'] in ['bathroom', 'kitchen', 'corridor']
]))
print(f"\n  Mean NN match on hypothesis classes ({', '.join(HYP_CLASSES)}): {hyp_nn_match*100:.1f}%")
print(f"  Mean NN match on unimodal baselines  (bathroom, kitchen, corridor):  {baseline_nn_match*100:.1f}%")
if hyp_nn_match > 0.65:
    verdict_5 = "NN match high → features ARE in the right region → CLASSIFIER-BOUNDARY problem"
elif hyp_nn_match < 0.50:
    verdict_5 = "NN match low → test features drifted from train class regions → FEATURE/DISTRIBUTION problem"
else:
    verdict_5 = "Intermediate NN match → mixed classifier + feature contributions"
print(f"  Verdict: {verdict_5}")

# ---------------------------------------------------------------------------
# Metric 6: Train vs test centroid drift per class (Scenario C test)
# ---------------------------------------------------------------------------
print(f"\n=== Metric 6: Train vs test class-centroid drift ===")
# Per-class centroid in train (L2-normalized) and test (L2-normalized).
# Drift = distance between train_c and test_c centroids. If drift is large
# relative to between-class distance for hypothesis classes, the class
# distribution has moved between splits (Scenario C: distribution shift).
drift_rows = []
for c in range(NUM_CLASSES):
    tr_mask = labels_14 == c
    te_mask = test_labels_n == c
    if tr_mask.sum() < 10 or te_mask.sum() < 10:
        continue
    tr_mu = feats_n[tr_mask].mean(axis=0)
    te_mu = test_feats_n[te_mask].mean(axis=0)
    drift = float(_np_14.linalg.norm(tr_mu - te_mu))
    drift_rel = drift / mean_between_class if mean_between_class > 0 else float('inf')
    drift_rows.append({'class': _clean_names[c], 'drift': drift, 'drift_rel_to_class_sep': drift_rel})

drift_rows.sort(key=lambda r: -r['drift_rel_to_class_sep'])
print(f"  (ranked by drift / between-class-distance; >0.5 = significant shift)\n")
print(f"  {'class':<22}{'drift':>10}{'drift / between_class':>26}")
for r in drift_rows[:10]:
    print(f"  {r['class']:<22}{r['drift']:>10.4f}{r['drift_rel_to_class_sep']:>26.3f}")
mean_drift_rel = float(_np_14.mean([r['drift_rel_to_class_sep'] for r in drift_rows]))
hyp_drift_rel = float(_np_14.mean([
    r['drift_rel_to_class_sep'] for r in drift_rows if r['class'] in HYP_CLASSES
]))
print(f"\n  Mean drift / between-class across all:        {mean_drift_rel:.3f}")
print(f"  Mean drift / between-class on hypothesis:     {hyp_drift_rel:.3f}")
if hyp_drift_rel > 0.5:
    verdict_6 = "Hypothesis-class centroids drift LARGE between train/test → DISTRIBUTION SHIFT (Scenario C)"
elif hyp_drift_rel < 0.2:
    verdict_6 = "Train/test centroids aligned per class → no distribution shift"
else:
    verdict_6 = "Moderate centroid drift"
print(f"  Verdict: {verdict_6}")

# ---------------------------------------------------------------------------
# t-SNE visualization per class (optional — only 5 classes for speed)
# ---------------------------------------------------------------------------
print(f"\n=== t-SNE per class (colored by sensor) — hypothesis + baseline ===")
classes_to_visualize = ['dining_area', 'library', 'classroom', 'bathroom', 'kitchen']
_fig, _axes = _plt_14.subplots(1, len(classes_to_visualize), figsize=(4 * len(classes_to_visualize), 4))
SENSOR_COLOR = {'kv1': 'tab:blue', 'kv2': 'tab:orange', 'realsense': 'tab:green', 'xtion': 'tab:red'}
for ax, cname in zip(_axes, classes_to_visualize):
    if cname not in _clean_names:
        ax.set_title(f"{cname} (not found)")
        continue
    c = _clean_names.index(cname)
    mask = labels_14 == c
    X = feats_n[mask]
    s = hpo_train_sensors_1913[mask]
    if X.shape[0] < 30:
        ax.set_title(f"{cname} (N={X.shape[0]}, too small)")
        continue
    perp = min(30, X.shape[0] // 4)
    proj = TSNE(n_components=2, perplexity=perp, random_state=SEED, init='pca').fit_transform(X)
    for sensor in _np_14.unique(s):
        m = s == sensor
        ax.scatter(proj[m, 0], proj[m, 1], s=18, alpha=0.7,
                   color=SENSOR_COLOR.get(sensor, 'gray'), label=f"{sensor} (n={m.sum()})")
    ax.set_title(f"{cname} (N={X.shape[0]})")
    ax.legend(fontsize=8, loc='best')
    ax.set_xticks([]); ax.set_yticks([])
_plt_14.tight_layout()
_plt_14.show()

# ---------------------------------------------------------------------------
# Scenario synthesis
# ---------------------------------------------------------------------------
# A: multi-modal hypothesis confirmed → sub-nodes should help more than they do
# B: inter-class overlap dominates → better features needed, not multi-modal classifier
# C: distribution shift between train/test → domain adaptation / invariance needed
# D: mixed → different classes, different failure modes
# [silenced — 19.15 per-class failure-mode classification supersedes this scenario labeling; variables still computed for JSON persistence]
# print(f"\n{'='*70}\n=== SCENARIO SYNTHESIS ===\n{'='*70}")
# print(f"  Metric 1 (silhouette K=2 mean):             {mean_s2:.3f}")
# print(f"  Metric 2 (intra/inter ratio):               {ratio:.3f}")
# print(f"  Metric 3 (my sensor ratio / intra-var):     {mean_sensor_ratio:.3f}")
# print(f"  Metric 4 (sensor / between-class frac):     {sensor_vs_class_frac:.3f}")
# print(f"  Metric 5 (NN match on hyp classes):         {hyp_nn_match*100:.1f}%")
# print(f"  Metric 6 (centroid drift on hyp classes):   {hyp_drift_rel:.3f}")
# print()

# Boolean signals
multimodal_confirmed = mean_s2 > 0.2 and sensor_vs_class_frac > 0.3
interclass_overlap   = mean_s2 < 0.15 and hyp_nn_match < 0.55
distribution_shift   = hyp_drift_rel > 0.5 and hyp_nn_match < 0.60
classifier_problem   = hyp_nn_match > 0.65  # features in right place, classifier failing

scenarios = []
if multimodal_confirmed:
    scenarios.append(('A', 'Multi-modal hypothesis CONFIRMED',
                      'Classes have real multi-modal structure and sensor is a structural axis. '
                      'Max-out SHOULD help more than it does. Investigate max-out training dynamics '
                      '(maybe try K=5, different init, sub-node-specific LR).'))
if interclass_overlap:
    scenarios.append(('B', 'Inter-class overlap DOMINATES',
                      'Features are unimodal within classes but classes sit too close together '
                      'in feature space. Need better backbone, contrastive/triplet losses, or '
                      'explicit margin-based classifier. Max-out/sampling cannot fix this.'))
if distribution_shift:
    scenarios.append(('C', 'DISTRIBUTION SHIFT train→test',
                      'Features are unimodal per class within each split, but train and test '
                      'class distributions do not overlap. Need domain-adaptation techniques, '
                      'feature invariance losses, or augmentation targeting the shift.'))
if classifier_problem and not scenarios:
    scenarios.append(('A*', 'Classifier-boundary problem (features OK)',
                      'Test features are in the right class regions — the classifier head is '
                      'failing to route them correctly. Revisit head architecture or boundary sharpness.'))

# if not scenarios:
#     print(f"  Scenario D (MIXED or AMBIGUOUS) — ...")
# else:
#     for code, title, action in scenarios:
#         print(f"  Scenario {code}: {title}")
#         print(f"    Action: {action}\n")
scenarios = []  # populated when synthesis block is re-enabled
# Persist
_out14 = {
    'metric_4_within_sensor_vs_between_class': {
        'mean_within_sensor': mean_within_sensor,
        'mean_between_class': mean_between_class,
        'fraction': sensor_vs_class_frac,
        'verdict': verdict_4,
    },
    'metric_5_nn_match': {
        'aggregate': nn_match_agg,
        'per_class': per_class_nn_rows,
        'hypothesis_mean': hyp_nn_match,
        'baseline_mean': baseline_nn_match,
        'verdict': verdict_5,
    },
    'metric_6_centroid_drift': {
        'per_class': drift_rows,
        'mean': mean_drift_rel,
        'hypothesis_mean': hyp_drift_rel,
        'verdict': verdict_6,
    },
    'scenarios': [{'code': s[0], 'title': s[1], 'action': s[2]} for s in scenarios],
}
with open(_os_14.path.join(GEN_DIR, 'multimodal_diagnostic_extended.json'), 'w') as _f:
    _json_14.dump(_out14, _f, indent=2)
print(f"\nSaved → {GEN_DIR}/multimodal_diagnostic_extended.json")


In [ ]:
# --- 19.15 Per-sensor test breakdown + per-class failure-mode classification ---
# Pure diagnostic — no model changes, no retraining. Uses gen_results from 19.1
# and metric outputs from 19.13/19.14.
#
# Four views (per team proposal):
#   1. Per-sensor aggregate test MCA + NN match
#   2. Per-(class × sensor) test accuracy matrix
#   3. Train vs test sensor distribution comparison
#   4. Per-class failure-mode classification:
#        sensor-driven drift  → high per-sensor accuracy variance + moderate/high NN match
#        scene/semantic drift → low per-sensor variance + low NN match
#        representation drift → high centroid drift + low NN match
#        classifier problem   → features in right region (high NN match), low test acc
#
# Requires cells 19.1, 19.13, 19.14 to have run (reads their variables).
import json as _json_15
import os as _os_15
from collections import defaultdict as _dd_15

import numpy as _np_15
import pandas as _pd_15

assert 'test_nn_class' in dir() and 'per_class_nn_rows' in dir(), \
    "Run cells 19.13 and 19.14 first — this cell reads their outputs."

SENSORS = ('kv1', 'kv2', 'realsense', 'xtion')

# Test-set sensors (aligned with gen_results['official_test']['labels'])
with open(_os_15.path.join(DATASET_CONFIG['data_root'], 'test', 'sample_paths.json')) as _f:
    _test_paths_1915 = _json_15.load(_f)
test_sensors_1915 = _np_15.array([p.split('/', 1)[0] for p in _test_paths_1915])
test_labels_1915 = gen_results['official_test']['labels']
test_preds_1915 = gen_results['official_test']['preds']

# Train-set sensors already built in 19.13 as hpo_train_sensors_1913
train_sensors_1915 = hpo_train_sensors_1913
train_labels_1915 = gen_results['hpo_train']['labels']

# ---------------------------------------------------------------------------
# View 1: Per-sensor aggregate test accuracy, MCA, NN match
# ---------------------------------------------------------------------------
print("=== View 1: Per-sensor aggregate test metrics ===")
print(f"  {'sensor':<12}{'N_test':>8}{'test_acc':>12}{'test_mca':>12}{'nn_match':>12}")
per_sensor_rows = []
for s in SENSORS:
    mask = test_sensors_1915 == s
    if mask.sum() == 0:
        continue
    acc = float((test_preds_1915[mask] == test_labels_1915[mask]).mean())
    # MCA over classes present in this sensor's test slice
    per_class_acc_s = []
    for c in range(NUM_CLASSES):
        cm = mask & (test_labels_1915 == c)
        if cm.sum() > 0:
            per_class_acc_s.append(float((test_preds_1915[cm] == c).mean()))
    mca = float(_np_15.mean(per_class_acc_s)) if per_class_acc_s else float('nan')
    nn_match = float((test_nn_class[mask] == test_labels_1915[mask]).mean())
    per_sensor_rows.append({
        'sensor': s, 'n_test': int(mask.sum()),
        'test_acc': acc, 'test_mca': mca, 'nn_match': nn_match,
    })
    print(f"  {s:<12}{int(mask.sum()):>8}{acc*100:>11.1f}%{mca*100:>11.1f}%{nn_match*100:>11.1f}%")

agg_acc = gen_results['official_test']['acc']
agg_mca = gen_results['official_test']['mca']
print(f"  {'(overall)':<12}{len(test_labels_1915):>8}{agg_acc*100:>11.1f}%{agg_mca*100:>11.1f}%"
      f"{float((test_nn_class == test_labels_1915).mean())*100:>11.1f}%")

sensor_mca_values = [r['test_mca'] for r in per_sensor_rows if not _np_15.isnan(r['test_mca'])]
sensor_mca_spread = (max(sensor_mca_values) - min(sensor_mca_values)) * 100
print(f"\n  Per-sensor MCA spread: {sensor_mca_spread:.1f}pp")
if sensor_mca_spread > 10:
    print(f"  → Sensor is a MAJOR axis of variation (>10pp spread)")
elif sensor_mca_spread < 5:
    print(f"  → Sensor is a MINOR axis of variation (<5pp spread)")
else:
    print(f"  → Sensor is a MODERATE axis of variation (5-10pp spread)")

# ---------------------------------------------------------------------------
# View 2: Per-(class × sensor) test accuracy matrix
# ---------------------------------------------------------------------------
print("\n=== View 2: Per-(class × sensor) test accuracy ===")
print(f"  (cells with N<10 shown as '—'; rightmost col is per-class MAX−MIN spread)\n")
cs_grid = _np_15.full((NUM_CLASSES, len(SENSORS)), _np_15.nan)
cs_n = _np_15.zeros((NUM_CLASSES, len(SENSORS)), dtype=int)
for c in range(NUM_CLASSES):
    for si, s in enumerate(SENSORS):
        mask = (test_labels_1915 == c) & (test_sensors_1915 == s)
        cs_n[c, si] = int(mask.sum())
        if mask.sum() >= 10:
            cs_grid[c, si] = float((test_preds_1915[mask] == c).mean())

# Build display rows
print(f"  {'class':<22}" + "".join(f"{s:>10}" for s in SENSORS) + f"{'spread':>10}")
cs_rows = []
for c in range(NUM_CLASSES):
    cname = _clean_names[c]
    cells = []
    vals = []
    for si, s in enumerate(SENSORS):
        if cs_n[c, si] < 10:
            cells.append("   —   ")
        else:
            cells.append(f"{cs_grid[c, si]*100:>7.1f}%")
            vals.append(cs_grid[c, si])
    spread = (max(vals) - min(vals)) * 100 if len(vals) >= 2 else float('nan')
    spread_str = f"{spread:.1f}pp" if not _np_15.isnan(spread) else "  —  "
    cs_rows.append({'class': cname, 'sensor_accs': dict(zip(SENSORS, cs_grid[c].tolist())),
                    'n_per_sensor': dict(zip(SENSORS, cs_n[c].tolist())), 'spread_pp': spread})
    print(f"  {cname:<22}" + "".join(f"{cell:>10}" for cell in cells) + f"{spread_str:>10}")

# ---------------------------------------------------------------------------
# View 3: Train vs test sensor distribution
# ---------------------------------------------------------------------------
print("\n=== View 3: Train vs test sensor distribution (aggregate) ===")
print(f"  {'sensor':<12}{'train_n':>10}{'train_%':>10}{'test_n':>10}{'test_%':>10}{'delta':>10}")
train_totals = {s: int((train_sensors_1915 == s).sum()) for s in SENSORS}
test_totals = {s: int((test_sensors_1915 == s).sum()) for s in SENSORS}
n_tr = sum(train_totals.values()); n_te = sum(test_totals.values())
for s in SENSORS:
    tr_pct = train_totals[s] / n_tr * 100
    te_pct = test_totals[s] / n_te * 100
    delta = te_pct - tr_pct
    print(f"  {s:<12}{train_totals[s]:>10}{tr_pct:>9.1f}%{test_totals[s]:>10}{te_pct:>9.1f}%{delta:>+9.1f}pp")

# ---------------------------------------------------------------------------
# View 4: Per-class failure-mode classification
# ---------------------------------------------------------------------------
# Decision rules (rough but interpretable):
#   - NN match ≥ 0.60  + low per-sensor spread (<8pp) → CLASSIFIER problem (features are fine)
#   - NN match ≥ 0.60  + high per-sensor spread (≥8pp) → SENSOR-DRIVEN drift
#   - NN match < 0.45  + centroid drift ≥ 0.40      → REPRESENTATION drift (upstream fix)
#   - NN match < 0.45  + drift < 0.40               → SCENE/SEMANTIC drift
#   - otherwise → MIXED / intermediate
print("\n=== View 4: Per-class failure-mode classification ===")
print(f"  (NN match and centroid drift from cells 19.13/19.14; sensor spread from View 2)\n")
print(f"  {'class':<22}{'test_acc':>10}{'NN_match':>10}{'drift':>10}{'sensor_spread':>16}{'failure_mode':>22}")
failure_rows = []
nn_lookup = {r['class']: r for r in per_class_nn_rows}
drift_lookup = {r['class']: r['drift_rel_to_class_sep'] for r in drift_rows}
cs_lookup = {r['class']: r['spread_pp'] for r in cs_rows}

def _classify(nn_match, drift, sensor_spread):
    if _np_15.isnan(sensor_spread):
        sensor_spread = 0.0
    if nn_match >= 0.60:
        if sensor_spread >= 8:
            return "SENSOR drift"
        return "CLASSIFIER problem"
    if nn_match < 0.45:
        if drift >= 0.40:
            return "REPRESENTATION drift"
        return "SCENE/SEMANTIC drift"
    return "MIXED / unclear"

for cname in _clean_names:
    if cname not in nn_lookup:
        continue
    nn = nn_lookup[cname]
    test_acc = nn['test_acc']
    drift = drift_lookup.get(cname, float('nan'))
    spread = cs_lookup.get(cname, float('nan'))
    mode = _classify(nn['nn_match'], drift, spread)
    failure_rows.append({
        'class': cname, 'test_acc': test_acc, 'nn_match': nn['nn_match'],
        'drift': drift, 'sensor_spread_pp': spread, 'failure_mode': mode,
    })
    drift_str = f"{drift:.3f}" if not _np_15.isnan(drift) else "  —  "
    spread_str = f"{spread:.1f}pp" if not _np_15.isnan(spread) else "  —  "
    print(f"  {cname:<22}{test_acc*100:>9.1f}%{nn['nn_match']*100:>9.1f}%{drift_str:>10}{spread_str:>16}{mode:>22}")

# Aggregate summary
mode_counts = _dd_15(int)
mode_weights = _dd_15(int)  # weighted by test samples
for r in failure_rows:
    n_test = nn_lookup[r['class']]['n_test']
    mode_counts[r['failure_mode']] += 1
    mode_weights[r['failure_mode']] += n_test
print(f"\n  Failure-mode breakdown:")
for mode in ['CLASSIFIER problem', 'SENSOR drift', 'REPRESENTATION drift',
            'SCENE/SEMANTIC drift', 'MIXED / unclear']:
    n_cls = mode_counts.get(mode, 0)
    n_samp = mode_weights.get(mode, 0)
    pct = n_samp / len(test_labels_1915) * 100
    print(f"    {mode:<24}  {n_cls:>3} classes  {n_samp:>5} test samples ({pct:>5.1f}%)")

# ---------------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------------
_out15 = {
    'per_sensor': per_sensor_rows,
    'sensor_mca_spread_pp': sensor_mca_spread,
    'per_class_sensor_grid': cs_rows,
    'train_sensor_distribution': {s: {'n': train_totals[s], 'pct': train_totals[s]/n_tr*100} for s in SENSORS},
    'test_sensor_distribution':  {s: {'n': test_totals[s],  'pct': test_totals[s]/n_te*100}  for s in SENSORS},
    'failure_mode_per_class': failure_rows,
    'failure_mode_aggregate': {
        mode: {'n_classes': mode_counts.get(mode, 0),
               'n_test_samples': mode_weights.get(mode, 0)}
        for mode in ['CLASSIFIER problem', 'SENSOR drift', 'REPRESENTATION drift',
                     'SCENE/SEMANTIC drift', 'MIXED / unclear']
    },
}
with open(_os_15.path.join(GEN_DIR, 'per_sensor_breakdown.json'), 'w') as _f:
    _json_15.dump(_out15, _f, indent=2, default=float)
print(f"\nSaved → {GEN_DIR}/per_sensor_breakdown.json")


In [ ]:
# --- 19.16 AdaBN diagnostic: BN batch stats vs running stats at test time ---
# At inference, switch BatchNorm layers to use BATCH statistics from each test
# batch instead of running mean/var learned during training. Everything else
# (classifier dropout, modality dropout) stays in eval mode.
#
# Hypothesis from 19.15 failure-mode breakdown: 25.6% of test samples are in
# classes flagged as REPRESENTATION drift (high centroid drift, low NN match).
# If part of that drift is BN-stats mismatch between train and test
# distributions, AdaBN recovers it without retraining. If AdaBN doesn't move
# the dial, the drift lives in the learned features themselves and only
# feature-level interventions (Phase 2 SupCon / domain adaptation) can fix it.
#
# Caveat: AdaBN needs batch >= 32 for stable batch statistics. test_loader's
# batch=64 is fine here. Single-sample inference can't use AdaBN, so this is
# a diagnostic + production-only-with-batched-inference signal.
import contextlib
import json as _json_16
import os as _os_16

import numpy as _np_16
import torch as _torch_16
import torch.nn as _nn_16

from src.models.linear_integration.li_net3.conv import LIBatchNorm2d as _LIBN_16

assert 'failure_rows' in dir(), "Run cell 19.15 first — this cell reads its outputs."

@contextlib.contextmanager
def _adabn_eval(model):
    """Context manager: model in eval mode globally (dropout off, MD off, etc.)
    EXCEPT BatchNorm modules, which run with .training=True and momentum=0.
    Training=True makes F.batch_norm use batch statistics; momentum=0 prevents
    running_mean / running_var from being mutated by the diagnostic pass.
    Original BN .training and .momentum values are restored on exit.
    """
    bn_classes = (_nn_16.BatchNorm1d, _nn_16.BatchNorm2d, _nn_16.BatchNorm3d,
                  _nn_16.SyncBatchNorm, _LIBN_16)
    backups = []
    model.eval()
    for m in model.modules():
        if isinstance(m, bn_classes):
            backups.append((m, m.training, getattr(m, 'momentum', None)))
            m.train()
            if hasattr(m, 'momentum') and m.momentum is not None:
                m.momentum = 0
    try:
        yield
    finally:
        for m, was_training, was_momentum in backups:
            m.train(was_training)
            if was_momentum is not None:
                m.momentum = was_momentum

assert test_loader.batch_size >= 32, (
    f"AdaBN needs test batch >= 32 for stable batch stats; got {test_loader.batch_size}"
)

@_torch_16.no_grad()
def _adabn_forward(loader):
    """Mirror of cell 19.1's hook_capture forward path, but BN uses batch stats.
    No hook needed — we only collect (preds, labels) here."""
    all_preds, all_labels = [], []
    with _adabn_eval(model):
        for batch in loader:
            *streams, labels = batch
            streams = [s.to(model.device, non_blocking=True) for s in streams]
            if getattr(model, 'gpu_augmentation', False) and getattr(model, 'gpu_aug', None) is not None:
                streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])
            with _torch_16.amp.autocast(device_type=model.device.type,
                                        enabled=getattr(model, 'use_amp', False)):
                logits = model(streams)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(labels.numpy())
    return _np_16.concatenate(all_preds), _np_16.concatenate(all_labels)

print("Running AdaBN forward pass on test set...")
adabn_preds, adabn_labels = _adabn_forward(test_loader)
adabn_acc = float((adabn_preds == adabn_labels).mean())
adabn_per_class_acc = _np_16.zeros(NUM_CLASSES)
adabn_per_class_n = _np_16.zeros(NUM_CLASSES, dtype=int)
for c in range(NUM_CLASSES):
    m = adabn_labels == c
    adabn_per_class_n[c] = int(m.sum())
    if m.sum() > 0:
        adabn_per_class_acc[c] = float((adabn_preds[m] == c).mean())
adabn_mca = float(_np_16.mean([adabn_per_class_acc[c] for c in range(NUM_CLASSES)
                                if adabn_per_class_n[c] > 0]))

# Baseline (running-stats BN) — already in gen_results from cell 19.1
baseline_acc = float(gen_results['official_test']['acc'])
baseline_mca = float(gen_results['official_test']['mca'])
baseline_per_class_acc = _np_16.asarray(gen_results['official_test']['per_class_acc'])
baseline_preds = gen_results['official_test']['preds']

# ---------------------------------------------------------------------------
# Top-line summary
# ---------------------------------------------------------------------------
print()
print("=== Top-line: AdaBN vs running-stats BN ===")
print(f"  {'metric':<14}{'baseline':>12}{'AdaBN':>12}{'delta':>12}")
print(f"  {'test acc':<14}{baseline_acc*100:>11.2f}%{adabn_acc*100:>11.2f}%"
      f"{(adabn_acc - baseline_acc)*100:>+11.2f}pp")
print(f"  {'test MCA':<14}{baseline_mca*100:>11.2f}%{adabn_mca*100:>11.2f}%"
      f"{(adabn_mca - baseline_mca)*100:>+11.2f}pp")

# ---------------------------------------------------------------------------
# Per-sensor breakdown
# ---------------------------------------------------------------------------
print(f"\n=== Per-sensor test MCA: AdaBN vs running-stats ===")
print(f"  {'sensor':<12}{'N_test':>8}{'baseline':>12}{'AdaBN':>12}{'delta':>12}")
per_sensor_adabn = []
for s in ('kv1', 'kv2', 'realsense', 'xtion'):
    mask = test_sensors_1915 == s
    if mask.sum() == 0:
        continue
    base_per, ada_per = [], []
    for c in range(NUM_CLASSES):
        cm = mask & (test_labels_1915 == c)
        if cm.sum() > 0:
            base_per.append(float((baseline_preds[cm] == c).mean()))
            ada_per.append(float((adabn_preds[cm] == c).mean()))
    base_mca_s = float(_np_16.mean(base_per)) if base_per else float('nan')
    ada_mca_s = float(_np_16.mean(ada_per)) if ada_per else float('nan')
    delta = ada_mca_s - base_mca_s
    per_sensor_adabn.append({'sensor': s, 'n_test': int(mask.sum()),
                              'baseline_mca': base_mca_s, 'adabn_mca': ada_mca_s, 'delta': delta})
    print(f"  {s:<12}{int(mask.sum()):>8}{base_mca_s*100:>11.1f}%{ada_mca_s*100:>11.1f}%"
          f"{delta*100:>+11.1f}pp")

# ---------------------------------------------------------------------------
# Per-class deltas — top movers
# ---------------------------------------------------------------------------
deltas = []
for c in range(NUM_CLASSES):
    if adabn_per_class_n[c] >= 10:
        d = float(adabn_per_class_acc[c] - baseline_per_class_acc[c])
        deltas.append({
            'class': _clean_names[c],
            'baseline': float(baseline_per_class_acc[c]),
            'adabn': float(adabn_per_class_acc[c]),
            'delta': d, 'n': int(adabn_per_class_n[c]),
        })
deltas.sort(key=lambda r: r['delta'])

print(f"\n=== Per-class deltas (sorted) ===")
print(f"  Top 5 REGRESSIONS:")
print(f"  {'class':<22}{'N':>6}{'baseline':>12}{'AdaBN':>12}{'delta':>12}")
for r in deltas[:5]:
    print(f"  {r['class']:<22}{r['n']:>6}{r['baseline']*100:>11.1f}%"
          f"{r['adabn']*100:>11.1f}%{r['delta']*100:>+11.1f}pp")
print(f"\n  Top 5 IMPROVEMENTS:")
for r in deltas[-5:][::-1]:
    print(f"  {r['class']:<22}{r['n']:>6}{r['baseline']*100:>11.1f}%"
          f"{r['adabn']*100:>11.1f}%{r['delta']*100:>+11.1f}pp")

# ---------------------------------------------------------------------------
# Did AdaBN help the REPRESENTATION drift classes specifically?
# ---------------------------------------------------------------------------
rep_drift_classes = [r['class'] for r in failure_rows if r['failure_mode'] == 'REPRESENTATION drift']
sensor_drift_classes = [r['class'] for r in failure_rows if r['failure_mode'] == 'SENSOR drift']

print(f"\n=== Effect on REPRESENTATION drift classes (the Phase-2 hypothesis target) ===")
print(f"  {'class':<22}{'baseline':>12}{'AdaBN':>12}{'delta':>12}")
rep_deltas = []
for cname in rep_drift_classes:
    for r in deltas:
        if r['class'] == cname:
            rep_deltas.append(r['delta'])
            print(f"  {cname:<22}{r['baseline']*100:>11.1f}%{r['adabn']*100:>11.1f}%"
                  f"{r['delta']*100:>+11.1f}pp")
            break
rep_mean = float(_np_16.mean(rep_deltas)) if rep_deltas else 0.0
print(f"\n  Mean delta on REPRESENTATION drift: {rep_mean*100:+.2f}pp")
if rep_mean > 0.02:
    rep_verdict = "AdaBN MATERIALLY HELPS representation drift → BN-stats mismatch IS a contributor"
elif rep_mean < -0.02:
    rep_verdict = "AdaBN HURTS representation drift → drift is in learned features, not BN stats"
else:
    rep_verdict = "AdaBN has minimal effect → drift is in learned features, not BN stats"
print(f"  Verdict: {rep_verdict}")

print(f"\n=== Effect on SENSOR drift classes (Phase 1b depth-aug target) ===")
print(f"  {'class':<22}{'baseline':>12}{'AdaBN':>12}{'delta':>12}")
sens_deltas = []
for cname in sensor_drift_classes:
    for r in deltas:
        if r['class'] == cname:
            sens_deltas.append(r['delta'])
            print(f"  {cname:<22}{r['baseline']*100:>11.1f}%{r['adabn']*100:>11.1f}%"
                  f"{r['delta']*100:>+11.1f}pp")
            break
sens_mean = float(_np_16.mean(sens_deltas)) if sens_deltas else 0.0
print(f"\n  Mean delta on SENSOR drift: {sens_mean*100:+.2f}pp")

# ---------------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------------
_out16 = {
    'baseline': {'test_acc': baseline_acc, 'test_mca': baseline_mca,
                 'per_class_acc': baseline_per_class_acc.tolist()},
    'adabn': {'test_acc': adabn_acc, 'test_mca': adabn_mca,
              'per_class_acc': adabn_per_class_acc.tolist()},
    'top_line_delta': {'test_acc_pp': (adabn_acc - baseline_acc) * 100,
                        'test_mca_pp': (adabn_mca - baseline_mca) * 100},
    'per_sensor': per_sensor_adabn,
    'per_class_deltas': deltas,
    'representation_drift_mean_delta_pp': rep_mean * 100,
    'sensor_drift_mean_delta_pp': sens_mean * 100,
    'representation_verdict': rep_verdict,
}
with open(_os_16.path.join(GEN_DIR, 'adabn_diagnostic.json'), 'w') as _f:
    _json_16.dump(_out16, _f, indent=2, default=float)
print(f"\nSaved → {GEN_DIR}/adabn_diagnostic.json")


In [ ]:
# --- 19.17 Source-target BN mixing: α sweep at inference ---
# AdaBN (cell 19.16) gave +4.95pp on REPRESENTATION drift but cost -1.31pp on
# top-line acc. Source-target mixing interpolates between AdaBN (α=0, pure batch
# stats) and standard eval (α=1, pure running stats), letting us tune for the
# sweet spot where rep-drift gains without majority-class regression.
#
# Pure inference-time intervention. No retraining. No model mutation (running
# stats are NOT touched by the blended forward — just normalization is).
#
# Selection: sweep α on hpo_val (avoid tuning on test). Pick α* maximizing val
# MCA. Apply to test for the final number. Report top-line + per-class deltas.
import contextlib
import json as _json_17
import os as _os_17

import numpy as _np_17
import torch as _torch_17

from src.models.linear_integration.li_net3.bn_source_target_mix import (
    bn_source_target_mix,
)


@_torch_17.no_grad()
def _eval_under_alpha(loader, alpha):
    """Forward pass over loader with α-blended BN; collect (preds, labels)."""
    all_preds, all_labels = [], []
    with bn_source_target_mix(model, alpha=alpha):
        for batch in loader:
            *streams, labels = batch
            streams = [s.to(model.device, non_blocking=True) for s in streams]
            if (getattr(model, 'gpu_augmentation', False) and
                getattr(model, 'gpu_aug', None) is not None):
                streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])
            with _torch_17.amp.autocast(device_type=model.device.type,
                                        enabled=getattr(model, 'use_amp', False)):
                logits = model(streams)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(labels.numpy())
    return _np_17.concatenate(all_preds), _np_17.concatenate(all_labels)


def _per_class_acc(preds, labels):
    pca = _np_17.zeros(NUM_CLASSES)
    pcn = _np_17.zeros(NUM_CLASSES, dtype=int)
    for c in range(NUM_CLASSES):
        m = labels == c
        pcn[c] = int(m.sum())
        if m.sum() > 0:
            pca[c] = float((preds[m] == c).mean())
    return pca, pcn


# ---------------------------------------------------------------------------
# Sweep α on hpo_val to pick α*
# ---------------------------------------------------------------------------
ALPHAS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
print("=== α sweep on hpo_val ===")
print(f"  {'alpha':<8}{'val_acc':>10}{'val_mca':>10}")
val_results = {}
for alpha in ALPHAS:
    p, l = _eval_under_alpha(eval_loaders['hpo_val'], alpha)
    acc = float((p == l).mean())
    pca, pcn = _per_class_acc(p, l)
    mca = float(_np_17.mean([pca[c] for c in range(NUM_CLASSES) if pcn[c] > 0]))
    val_results[alpha] = {'acc': acc, 'mca': mca, 'preds': p, 'labels': l,
                          'pca': pca.tolist(), 'pcn': pcn.tolist()}
    print(f"  {alpha:<8.2f}{acc*100:>9.2f}%{mca*100:>9.2f}%")

# Pick α* by val MCA. Tie-break: prefer α closer to 1 (more conservative).
best_alpha = max(ALPHAS, key=lambda a: (val_results[a]['mca'], a))
print(f"\n  α* = {best_alpha:.2f}  (selected by max val MCA, ties → larger α)")

# ---------------------------------------------------------------------------
# Apply α* to test set + diff vs standard-eval baseline (α=1.0)
# ---------------------------------------------------------------------------
print(f"\n=== Test eval at α* = {best_alpha:.2f} vs α=1.0 (standard eval) ===")
p_star, l_star = _eval_under_alpha(eval_loaders['official_test'], best_alpha)
p_one, l_one = _eval_under_alpha(eval_loaders['official_test'], 1.0)
acc_star = float((p_star == l_star).mean())
acc_one = float((p_one == l_one).mean())
pca_star, pcn_star = _per_class_acc(p_star, l_star)
pca_one, pcn_one = _per_class_acc(p_one, l_one)
mca_star = float(_np_17.mean([pca_star[c] for c in range(NUM_CLASSES) if pcn_star[c] > 0]))
mca_one = float(_np_17.mean([pca_one[c] for c in range(NUM_CLASSES) if pcn_one[c] > 0]))

print(f"  {'metric':<14}{'α=1.0':>12}{'α*=' + f'{best_alpha:.2f}':>12}{'delta':>12}")
print(f"  {'test acc':<14}{acc_one*100:>11.2f}%{acc_star*100:>11.2f}%"
      f"{(acc_star - acc_one)*100:>+11.2f}pp")
print(f"  {'test MCA':<14}{mca_one*100:>11.2f}%{mca_star*100:>11.2f}%"
      f"{(mca_star - mca_one)*100:>+11.2f}pp")

# ---------------------------------------------------------------------------
# Per-class deltas at α*
# ---------------------------------------------------------------------------
deltas = []
for c in range(NUM_CLASSES):
    if pcn_star[c] >= 10:
        d = float(pca_star[c] - pca_one[c])
        deltas.append({
            'class': _clean_names[c],
            'baseline': float(pca_one[c]),
            'alpha_star': float(pca_star[c]),
            'delta': d, 'n': int(pcn_star[c]),
        })
deltas.sort(key=lambda r: r['delta'])
print(f"\n=== Per-class deltas at α* = {best_alpha:.2f} ===")
print(f"  Top 5 REGRESSIONS:")
print(f"  {'class':<22}{'N':>6}{'α=1.0':>12}{'α*':>12}{'delta':>12}")
for r in deltas[:5]:
    print(f"  {r['class']:<22}{r['n']:>6}{r['baseline']*100:>11.1f}%"
          f"{r['alpha_star']*100:>11.1f}%{r['delta']*100:>+11.1f}pp")
print(f"\n  Top 5 IMPROVEMENTS:")
for r in deltas[-5:][::-1]:
    print(f"  {r['class']:<22}{r['n']:>6}{r['baseline']*100:>11.1f}%"
          f"{r['alpha_star']*100:>11.1f}%{r['delta']*100:>+11.1f}pp")

# REPRESENTATION drift bucket effect
rep_drift_classes = [r['class'] for r in failure_rows if r['failure_mode'] == 'REPRESENTATION drift']
rep_deltas = [r['delta'] for r in deltas if r['class'] in rep_drift_classes]
print(f"\n  Mean delta on REPRESENTATION drift classes ({len(rep_deltas)} reliable): "
      f"{_np_17.mean(rep_deltas)*100:+.2f}pp" if rep_deltas else "  (no rep-drift classes had n>=10)")

# ---------------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------------
_out17 = {
    'alphas_swept': ALPHAS,
    'val_results': {str(a): {'acc': val_results[a]['acc'],
                             'mca': val_results[a]['mca']}
                    for a in ALPHAS},
    'best_alpha': best_alpha,
    'test_at_alpha_star': {'acc': acc_star, 'mca': mca_star,
                            'per_class_acc': pca_star.tolist()},
    'test_at_alpha_one': {'acc': acc_one, 'mca': mca_one,
                           'per_class_acc': pca_one.tolist()},
    'top_line_delta_pp': {'acc': (acc_star - acc_one) * 100,
                           'mca': (mca_star - mca_one) * 100},
    'per_class_deltas': deltas,
    'rep_drift_mean_delta_pp': float(_np_17.mean(rep_deltas) * 100) if rep_deltas else None,
}
with open(_os_17.path.join(GEN_DIR, 'bn_source_target_mix.json'), 'w') as _f:
    _json_17.dump(_out17, _f, indent=2, default=float)
print(f"\nSaved → {GEN_DIR}/bn_source_target_mix.json")


## 20. Data-Pipeline Parity Checks

Defensive checks on the raw data and on the model's forward path. Run these any
time Section 19 numbers look surprising — they'll distinguish data-level issues
(preprocessing drift, normalization mismatch, sensor-specific encoding) from
model-level issues (BN drift, feature collapse).

**Expected under a healthy pipeline:**
- Same dtype and spatial shape on both train and test `.pt` files
- Per-channel RGB means within a few units between splits
- Depth means within ~10% globally AND per-sensor (kv1/kv2/realsense/xtion)
- `norm_stats.json` close to the actual pooled train statistics
- `model.evaluate()` and Section 19's hook pass agree within `PARITY_TOL_PP`
- Model input ranges identical across hpo_train / hpo_val / official_test

**Suspect signatures:**
- Depth `min`/`max`/`mean` differing by >15% between splits — depth-encoding drift
- RGB channel means differing by >10 — different color preprocessing
- Per-sensor depth means consistent within a split but very different between splits
  — sensor-specific preprocessing applied asymmetrically
- `*.pt` file mtimes days apart — non-atomic preprocessing
- `model.evaluate` disagrees with Section 19 hook pass — `_validate` changed


In [ ]:
# --- 20.1 Tensor dtype / shape / range per split ---
import os
import torch

TRAIN_DIR = os.path.join(DATASET_CONFIG['data_root'], 'train')
TEST_DIR  = os.path.join(DATASET_CONFIG['data_root'], 'test')

def summarize(path, label):
    t = torch.load(path, weights_only=True, mmap=True)
    tf = t.float()
    print(f"  {label:<12} dtype={str(t.dtype):<14} shape={tuple(t.shape)}  "
          f"min={tf.min().item():10.3f}  max={tf.max().item():12.3f}  "
          f"mean={tf.mean().item():10.3f}  std={tf.std().item():10.3f}")
    return t

print("=== Tensor summary (train vs test) ===")
for fname in ("rgb_tensors.pt", "depth_tensors.pt"):
    print(f"\n{fname}:")
    _ = summarize(os.path.join(TRAIN_DIR, fname), "train")
    _ = summarize(os.path.join(TEST_DIR,  fname), "test ")

In [ ]:
# --- 20.2 File provenance: mtimes, dataset_info.txt, norm_stats.json ---
import os
import json
import time

ROOT = DATASET_CONFIG['data_root']

print("=== File mtimes (train vs test .pt files) ===")
files = [
    ("train/rgb_tensors.pt", os.path.join(ROOT, "train", "rgb_tensors.pt")),
    ("train/depth_tensors.pt", os.path.join(ROOT, "train", "depth_tensors.pt")),
    ("test/rgb_tensors.pt",  os.path.join(ROOT, "test",  "rgb_tensors.pt")),
    ("test/depth_tensors.pt",os.path.join(ROOT, "test",  "depth_tensors.pt")),
    ("norm_stats.json",      os.path.join(ROOT, "norm_stats.json")),
    ("class_names.txt",      os.path.join(ROOT, "class_names.txt")),
    ("dataset_info.txt",     os.path.join(ROOT, "dataset_info.txt")),
]
for label, path in files:
    if os.path.exists(path):
        mt = os.path.getmtime(path)
        sz = os.path.getsize(path)
        print(f"  {label:<28} {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(mt))}  "
              f"({sz/1024/1024:.1f} MB)")
    else:
        print(f"  {label:<28} MISSING")

# dataset_info.txt
info_path = os.path.join(ROOT, "dataset_info.txt")
if os.path.exists(info_path):
    print("\n=== dataset_info.txt ===")
    print(open(info_path).read())

# norm_stats.json
ns_path = os.path.join(ROOT, "norm_stats.json")
if os.path.exists(ns_path):
    print("=== norm_stats.json ===")
    print(json.dumps(json.load(open(ns_path)), indent=2))

In [ ]:
# --- 20.3 Channel-wise stats for train vs test, and vs norm_stats.json ---
# If norm_stats were computed from a pool that no longer matches the on-disk train/
# (e.g. preprocessing re-run without recomputing stats), normalization will land the
# data in the wrong region of input space.
import os
import json
import numpy as np
import torch

ROOT = DATASET_CONFIG['data_root']

def channel_stats(path, sample_every=1):
    t = torch.load(path, weights_only=True, mmap=True).float()
    if sample_every > 1:
        t = t[::sample_every]
    # RGB tensors: [N, 3, H, W]. Depth: [N, 1, H, W].
    means = t.mean(dim=(0, 2, 3))
    stds  = t.std(dim=(0, 2, 3))
    return means.tolist(), stds.tolist()

print("Computing per-channel stats (this can take ~10-30s)...")
tr_rgb_mean,  tr_rgb_std  = channel_stats(os.path.join(ROOT, "train", "rgb_tensors.pt"))
te_rgb_mean,  te_rgb_std  = channel_stats(os.path.join(ROOT, "test",  "rgb_tensors.pt"))
tr_dpt_mean,  tr_dpt_std  = channel_stats(os.path.join(ROOT, "train", "depth_tensors.pt"))
te_dpt_mean,  te_dpt_std  = channel_stats(os.path.join(ROOT, "test",  "depth_tensors.pt"))

print("\n=== RGB channel means (on stored dtype scale) ===")
print(f"  train : {[f'{x:.2f}' for x in tr_rgb_mean]}  std={[f'{x:.2f}' for x in tr_rgb_std]}")
print(f"  test  : {[f'{x:.2f}' for x in te_rgb_mean]}  std={[f'{x:.2f}' for x in te_rgb_std]}")
diff = [abs(a - b) for a, b in zip(tr_rgb_mean, te_rgb_mean)]
print(f"  |train-test| per-channel: {[f'{x:.2f}' for x in diff]}   "
      f"{'[OK]' if max(diff) < 10 else '[SUSPICIOUS >10]'}")

print("\n=== Depth channel mean/std (on stored dtype scale) ===")
print(f"  train : mean={tr_dpt_mean[0]:.2f}  std={tr_dpt_std[0]:.2f}")
print(f"  test  : mean={te_dpt_mean[0]:.2f}  std={te_dpt_std[0]:.2f}")
ratio = te_dpt_mean[0] / max(tr_dpt_mean[0], 1e-9)
print(f"  test/train mean ratio: {ratio:.3f}  "
      f"{'[OK]' if 0.85 <= ratio <= 1.18 else '[SUSPICIOUS — possible encoding mismatch]'}")

# Compare to norm_stats.json
ns_path = os.path.join(ROOT, "norm_stats.json")
if os.path.exists(ns_path):
    ns = json.load(open(ns_path))
    print("\n=== norm_stats.json vs actual-train vs actual-test ===")
    print(f"  norm_stats rgb_mean:  {ns.get('rgb_mean')}")
    print(f"  train      rgb_mean:  {[round(x, 4) for x in tr_rgb_mean]}")
    print(f"  test       rgb_mean:  {[round(x, 4) for x in te_rgb_mean]}")
    print(f"  norm_stats rgb_std:   {ns.get('rgb_std')}")
    print(f"  train      rgb_std:   {[round(x, 4) for x in tr_rgb_std]}")
    print(f"  test       rgb_std:   {[round(x, 4) for x in te_rgb_std]}")
    print(f"  norm_stats depth_mean:{ns.get('depth_mean')}")
    print(f"  train      depth_mean:{round(tr_dpt_mean[0], 4)}")
    print(f"  test       depth_mean:{round(te_dpt_mean[0], 4)}")
    print(f"  norm_stats depth_std: {ns.get('depth_std')}")
    print(f"  train      depth_std: {round(tr_dpt_std[0], 4)}")
    print(f"  test       depth_std: {round(te_dpt_std[0], 4)}")
    print("\nIf norm_stats significantly disagrees with TRAIN actuals, norm_stats is stale.")
    print("If TRAIN and TEST actuals differ substantially, preprocessing drifted between splits.")

In [ ]:
# --- 20.4 Per-sensor depth stats (detects sensor-specific encoding drift) ---
# SUN RGB-D raw depth uses a sensor-specific bit-rotation encoding. If the
# preprocessing that undoes that encoding was updated between train and test
# runs, per-sensor means will diverge between splits.
import os
import json
import numpy as np
import torch

ROOT = DATASET_CONFIG['data_root']
SENSORS = ("kv1", "kv2", "realsense", "xtion")

def load_paths_and_depth(split):
    with open(os.path.join(ROOT, split, "sample_paths.json")) as f:
        paths = json.load(f)
    depth = torch.load(os.path.join(ROOT, split, "depth_tensors.pt"),
                       weights_only=True, mmap=True)
    return paths, depth

def per_sensor_stats(paths, depth):
    out = {}
    for s in SENSORS:
        idx = [i for i, p in enumerate(paths) if p.split('/', 1)[0] == s]
        if len(idx) == 0:
            out[s] = (0, float("nan"), float("nan"))
            continue
        d = depth.index_select(0, torch.tensor(idx, dtype=torch.long)).float()
        out[s] = (len(idx), float(d.mean().item()), float(d.std().item()))
    return out

tr_paths, tr_depth = load_paths_and_depth("train")
te_paths, te_depth = load_paths_and_depth("test")
tr_stats = per_sensor_stats(tr_paths, tr_depth)
te_stats = per_sensor_stats(te_paths, te_depth)

print("=== Per-sensor depth mean/std (train vs test) ===")
print(f"  {'sensor':<10} {'train n':>8} {'train mean':>12} {'train std':>11}   "
      f"{'test n':>8} {'test mean':>12} {'test std':>11}   {'ratio te/tr':>11}")
any_mismatch = False
for s in SENSORS:
    n_tr, m_tr, sd_tr = tr_stats[s]
    n_te, m_te, sd_te = te_stats[s]
    ratio = m_te / m_tr if (m_tr and not np.isnan(m_tr) and m_tr != 0) else float("nan")
    flag = ""
    if not np.isnan(ratio) and (ratio < 0.85 or ratio > 1.18):
        flag = "  <-- MISMATCH"
        any_mismatch = True
    print(f"  {s:<10} {n_tr:>8d} {m_tr:>12.2f} {sd_tr:>11.2f}   "
          f"{n_te:>8d} {m_te:>12.2f} {sd_te:>11.2f}   {ratio:>11.3f}{flag}")

print()
if any_mismatch:
    print("=> Per-sensor depth means differ >15% between train and test for at least one sensor.")
    print("   Most likely cause: preprocessing (bit-rotation unpacking, unit conversion) was")
    print("   applied to train but not test, or to different versions. Re-preprocess both")
    print("   splits together with the current scripts/preprocess_sunrgbd_19.py.")
else:
    print("=> Per-sensor depth means agree within 15% between train and test.")
    print("   Depth encoding is consistent; look elsewhere for the pipeline bug.")

In [ ]:
# --- 20.5 Visual spot-check: 4 train + 4 test samples side-by-side ---
# Does test data look like the same kind of images as train? Works directly on
# the stored tensors (pre-normalization) to reveal any raw-format mismatch.
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = DATASET_CONFIG['data_root']
tr_rgb   = torch.load(os.path.join(ROOT, "train", "rgb_tensors.pt"),   weights_only=True, mmap=True)
te_rgb   = torch.load(os.path.join(ROOT, "test",  "rgb_tensors.pt"),   weights_only=True, mmap=True)
tr_depth = torch.load(os.path.join(ROOT, "train", "depth_tensors.pt"), weights_only=True, mmap=True)
te_depth = torch.load(os.path.join(ROOT, "test",  "depth_tensors.pt"), weights_only=True, mmap=True)

rng = np.random.default_rng(0)
n_show = 4
tr_idx = rng.choice(len(tr_rgb), size=n_show, replace=False)
te_idx = rng.choice(len(te_rgb), size=n_show, replace=False)

fig, ax = plt.subplots(4, n_show, figsize=(3.3 * n_show, 12))
for j in range(n_show):
    rgb = tr_rgb[tr_idx[j]].float()
    rgb = (rgb / rgb.max()).permute(1, 2, 0).numpy() if rgb.max() > 1 else rgb.permute(1, 2, 0).numpy()
    ax[0, j].imshow(np.clip(rgb, 0, 1))
    ax[0, j].set_title(f"TRAIN rgb [{tr_idx[j]}]")
    ax[0, j].axis("off")

    d = tr_depth[tr_idx[j]].float().squeeze().numpy()
    im = ax[1, j].imshow(d, cmap="viridis")
    ax[1, j].set_title(f"TRAIN depth  mean={d.mean():.0f}")
    ax[1, j].axis("off")
    plt.colorbar(im, ax=ax[1, j], fraction=0.046)

    rgb = te_rgb[te_idx[j]].float()
    rgb = (rgb / rgb.max()).permute(1, 2, 0).numpy() if rgb.max() > 1 else rgb.permute(1, 2, 0).numpy()
    ax[2, j].imshow(np.clip(rgb, 0, 1))
    ax[2, j].set_title(f"TEST  rgb [{te_idx[j]}]")
    ax[2, j].axis("off")

    d = te_depth[te_idx[j]].float().squeeze().numpy()
    im = ax[3, j].imshow(d, cmap="viridis")
    ax[3, j].set_title(f"TEST  depth  mean={d.mean():.0f}")
    ax[3, j].axis("off")
    plt.colorbar(im, ax=ax[3, j], fraction=0.046)

plt.tight_layout()
plt.savefig(os.path.join(GEN_DIR, "train_vs_test_samples.png"), dpi=120)
plt.show()

In [ ]:
# --- 20.6 First-conv activation sanity: does the model see test inputs as in-distribution? ---
# Pushes one batch per split through the full model and reports:
#   - input range AS THE MODEL SEES IT (post gpu_aug if enabled, else raw from dataset)
#   - penultimate feature mean / std / abs_median (captured at model.fc input)
# If the test batch produces features with a wildly different distribution than
# the train batch, the problem is upstream (input normalization, first conv, BN).
# Mirrors the exact forward path _validate uses (li_net.py:1893-1916).
import numpy as np
import torch

device = model.device
model.eval()

@torch.no_grad()
def one_batch_feature_stats(loader, label):
    batch = next(iter(loader))
    *streams, _ = batch
    streams = [s.to(device, non_blocking=True) for s in streams]
    # Match _validate: apply gpu_aug if the model was compiled with it.
    if getattr(model, 'gpu_augmentation', False) and getattr(model, 'gpu_aug', None) is not None:
        streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])
    # Ranges AS THE MODEL SEES THEM (post gpu_aug) — this is what actually enters conv1.
    rgb_lo,   rgb_hi   = streams[0].min().item(), streams[0].max().item()
    depth_lo, depth_hi = streams[1].min().item(), streams[1].max().item()

    captured = {}
    def _h(mod, inp, out):
        x = inp[0]
        captured['feat'] = x.flatten(1).float().cpu()
    h = model.fc.register_forward_hook(_h)
    with torch.amp.autocast(device_type=device.type, enabled=getattr(model, 'use_amp', False)):
        _ = model(streams)
    h.remove()
    f = captured['feat'].numpy()
    print(f"  {label:<14} model-input rgb=[{rgb_lo:+.3f},{rgb_hi:+.3f}]  "
          f"depth=[{depth_lo:+.3f},{depth_hi:+.3f}]")
    print(f"  {label:<14} penult feat   mean={f.mean():+.3f}  std={f.std():.3f}  "
          f"abs_median={np.median(np.abs(f)):.3f}")
    return f

print("First-batch model-input ranges + penultimate feature stats:")
fa = one_batch_feature_stats(eval_loaders['hpo_train'],     'hpo_train')
fb = one_batch_feature_stats(eval_loaders['hpo_val'],       'hpo_val')
fc_feat = one_batch_feature_stats(eval_loaders['official_test'], 'official_test')

print(f"\nMean-feature L2 distance train<->test: {np.linalg.norm(fa.mean(0) - fc_feat.mean(0)):.3f}")
print(f"Mean-feature L2 distance train<->val : {np.linalg.norm(fa.mean(0) - fb.mean(0)):.3f}")
print("If model-input ranges disagree -> normalization bug (19.0b should have caught this).")
print("If input ranges match but feature distance is large -> deeper issue (BN drift, feature collapse).")


In [ ]:
# --- 20.7 Eval parity: model.evaluate() vs eval_collect (hard check) ---
# Runs the canonical model.evaluate() on each loader and compares to the numbers
# Section 19 reported. If these disagree, Section 19's numbers are wrong and the
# whole analysis is suspect. Stream_monitoring=False so only the baseline clean
# forward pass runs (comparable to eval_collect).
print("=== Parity check: model.evaluate() vs Section 19 eval_collect() ===")
print(f"  {'split':<16} {'path':<14} {'loss':>10} {'acc':>10} {'mca':>10}")

TOL_ACC_PP = 0.5
TOL_LOSS   = 0.05
any_mismatch = False
for name, loader in eval_loaders.items():
    canon = model.evaluate(loader, stream_monitoring=False)
    c_loss, c_acc, c_mca = canon['loss'], canon['accuracy'], canon['mean_class_accuracy']
    e = gen_results[name]
    print(f"  {name:<16} {'model.eval':<14} {c_loss:>10.4f} {c_acc*100:>9.2f}% {c_mca*100:>9.2f}%")
    print(f"  {name:<16} {'eval_collect':<14} {e['loss']:>10.4f} {e['acc']*100:>9.2f}% {e['mca']*100:>9.2f}%")
    dacc = abs(c_acc - e['acc']) * 100
    dmca = abs(c_mca - e['mca']) * 100
    dloss = abs(c_loss - e['loss'])
    mm = (dacc > TOL_ACC_PP) or (dmca > TOL_ACC_PP) or (dloss > TOL_LOSS)
    status = "MISMATCH" if mm else "ok"
    print(f"  {name:<16} {'delta':<14} {dloss:>10.4f} {dacc:>9.2f}pp {dmca:>9.2f}pp   [{status}]")
    if mm:
        any_mismatch = True
    print()

if any_mismatch:
    print("!!! Paths disagree. Section 19 metrics cannot be trusted.")
    print("    Most common cause: gpu_augmentation mismatch between training and eval.")
    print(f"    model.gpu_augmentation = {getattr(model, 'gpu_augmentation', None)}")
    print(f"    model.gpu_aug          = {type(getattr(model, 'gpu_aug', None)).__name__}")
else:
    print("All three splits match within tolerance. Section 19 numbers are canonical.")

In [ ]:
# --- 20.8 BN reset oracle (compact; on this checkpoint's three splits) ---
# If resetting BN running stats to test-set statistics recovers significant accuracy,
# BN-stat convergence with the smaller training set is the mechanism. This is the
# single most informative check given the "same config, more data" observation.
import copy

orig = model.evaluate(eval_loaders['official_test'], stream_monitoring=False)
print(f"baseline test:               acc={orig['accuracy']*100:5.2f}%  mca={orig['mean_class_accuracy']*100:5.2f}%")

print("\nResetting BN stats on hpo_train (control; should be ~same)...")
m_ctrl = copy.deepcopy(model)
reset_bn_stats(m_ctrl, eval_loaders['hpo_train'])
ctrl = m_ctrl.evaluate(eval_loaders['official_test'], stream_monitoring=False)
print(f"after hpo_train BN reset:    acc={ctrl['accuracy']*100:5.2f}%  mca={ctrl['mean_class_accuracy']*100:5.2f}%  "
      f"(delta acc {(ctrl['accuracy']-orig['accuracy'])*100:+.2f}pp, mca {(ctrl['mean_class_accuracy']-orig['mean_class_accuracy'])*100:+.2f}pp)")

print("\nResetting BN stats on official_test (oracle)...")
m_oracle = copy.deepcopy(model)
reset_bn_stats(m_oracle, eval_loaders['official_test'])
oracle = m_oracle.evaluate(eval_loaders['official_test'], stream_monitoring=False)
d_acc = (oracle['accuracy'] - orig['accuracy']) * 100
d_mca = (oracle['mean_class_accuracy'] - orig['mean_class_accuracy']) * 100
print(f"after test BN reset:         acc={oracle['accuracy']*100:5.2f}%  mca={oracle['mean_class_accuracy']*100:5.2f}%  "
      f"(delta acc {d_acc:+.2f}pp, mca {d_mca:+.2f}pp)")

print("\nInterpretation:")
if d_mca > 10:
    print(f"  Oracle BN reset recovers {d_mca:+.1f}pp MCA.")
    print("  BN running statistics on the training distribution are a major contributor to")
    print("  the test gap. Fix options: larger batch size, tune BN momentum, SyncBN, or")
    print("  GroupNorm/LayerNorm. Not a preprocessing bug.")
elif d_mca > 3:
    print(f"  Oracle BN reset gives {d_mca:+.1f}pp MCA.")
    print("  BN drift is a meaningful contributor but not the whole story. Feature-level")
    print("  divergence or classifier overfit likely also plays a role.")
else:
    print(f"  Oracle BN reset only gives {d_mca:+.1f}pp MCA.")
    print("  BN is NOT the mechanism. The collapse is in the feature extractor or classifier.")

del m_ctrl, m_oracle
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# --- 20.9 Per-stream eval: does RGB-only or Depth-only work on test? ---
# If the integrated model fails on test but a single stream alone works reasonably,
# the integration/fusion path is what's collapsing. If both single-stream paths also
# fail on test, the failure is at the stream encoders (closer to inputs) or in BN.
print("=== Per-stream accuracy on each split (via model.evaluate(blanked_streams=...)) ===")
print(f"  {'split':<16} {'full':>12} {'RGB only':>12} {'Depth only':>12}")

def fmt(r):
    return f"acc={r['accuracy']*100:5.2f}% mca={r['mean_class_accuracy']*100:5.2f}%"

for name, loader in eval_loaders.items():
    full = model.evaluate(loader, stream_monitoring=False)
    rgb_only   = model.evaluate(loader, stream_monitoring=False, blanked_streams={1})  # blank depth -> RGB only
    depth_only = model.evaluate(loader, stream_monitoring=False, blanked_streams={0})  # blank RGB -> depth only
    print(f"  {name:<16} {fmt(full):>30} {fmt(rgb_only):>32} {fmt(depth_only):>32}")

In [ ]:
# --- 20.10 Logit-magnitude + prediction-entropy statistics per split ---
# If the model's logits collapse to tiny values on test (or the predicted-class
# distribution concentrates on a handful of classes), we see mode collapse
# numerically, not just in the confusion matrix.
from collections import Counter

import torch

device_ = model.device
model.eval()

print(f"{'split':<16} {'logit max':>12} {'logit mean':>12} {'entropy':>10} "
      f"{'top1 frac':>10} {'unique pred classes':>22}")

@torch.no_grad()
def logit_stats(loader, name):
    maxes, means, entropies, preds = [], [], [], []
    for batch in loader:
        *streams, labels = batch
        streams = [s.to(device_, non_blocking=True) for s in streams]
        if getattr(model, 'gpu_augmentation', False) and getattr(model, 'gpu_aug', None) is not None:
            streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])
        with torch.amp.autocast(device_type=device_.type, enabled=getattr(model, 'use_amp', False)):
            logits = model(streams).float()
        maxes.append(logits.max(dim=1).values.cpu())
        means.append(logits.mean(dim=1).cpu())
        probs = torch.softmax(logits, dim=1)
        ent = -(probs * probs.clamp_min(1e-12).log()).sum(dim=1).cpu()
        entropies.append(ent)
        preds.append(logits.argmax(dim=1).cpu())
    maxes = torch.cat(maxes); means = torch.cat(means); entropies = torch.cat(entropies); preds = torch.cat(preds)
    pred_ctr = Counter(preds.tolist())
    top1_frac = pred_ctr.most_common(1)[0][1] / len(preds)
    print(f"  {name:<14} {maxes.mean().item():>12.3f} {means.mean().item():>12.3f} "
          f"{entropies.mean().item():>10.3f} {top1_frac:>10.3f} "
          f"{len(pred_ctr):>3d}/{NUM_CLASSES}  "
          f"(most predicted: {CLASS_NAMES[pred_ctr.most_common(1)[0][0]]})")

for name, loader in eval_loaders.items():
    logit_stats(loader, name)

# Reference: random uniform softmax entropy on 19 classes = ln(19) ~= 2.944
print(f"\n(reference: uniform 19-class entropy = {float(np.log(NUM_CLASSES)):.3f};")
print("  if test entropy is close to uniform, the model is near-random; if entropy is")
print("  low AND top1_frac is high, the model is confidently predicting ~1 class.)")

In [ ]:
# --- 20.11 Load full-training checkpoint and evaluate on this loader (A/B) ---
# Confirms the 46% vs 16% claim is real and apples-to-apples on identical test data.
# EDIT FULL_TRAIN_CKPT below to the path of the full-training run's best_model.pt.
import copy
import torch

FULL_TRAIN_CKPT = "/content/drive/MyDrive/linet_checkpoints/run_20260423_222229/final_model.pt"  # EDIT: path to the val-in-train "46%" checkpoint

if not FULL_TRAIN_CKPT:
    print("Set FULL_TRAIN_CKPT above to the full-training run's checkpoint path and rerun.")
    print("Skipping A/B comparison.")
else:
    _dev = model.device
    # Fresh model with identical architecture
    from src.models.linear_integration.li_net3 import li_resnet18
    model_full = li_resnet18(
        num_classes=NUM_CLASSES,
        stream_input_channels=MODEL_CONFIG['stream_input_channels'],
        dropout_p=MODEL_CONFIG['dropout_p'],
        width_multiplier=MODEL_CONFIG['width_multiplier'],
        device=_dev.type,
        use_amp=MODEL_CONFIG.get('use_amp', True),
    )
    model_full.compile(
        optimizer=torch.optim.AdamW(model_full.parameters(), lr=1e-4),
        scheduler=None,
        loss='cross_entropy',
        label_smoothing=0.0,
        gpu_augmentation=getattr(model, 'gpu_augmentation', False),
    )
    ckpt = torch.load(FULL_TRAIN_CKPT, map_location=_dev, weights_only=False)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
    missing, unexpected = model_full.load_state_dict(state, strict=False)
    print(f"Loaded full-training ckpt: missing={len(missing)}  unexpected={len(unexpected)}")
    model_full.to(_dev); model_full.eval()

    print("\n=== A/B on official_test (identical loader) ===")
    for tag, m in [('with_val (current)', model), ('full_train (loaded)', model_full)]:
        r = m.evaluate(eval_loaders['official_test'], stream_monitoring=False)
        print(f"  {tag:<22}  acc={r['accuracy']*100:5.2f}%  mca={r['mean_class_accuracy']*100:5.2f}%  loss={r['loss']:.4f}")

    print("\n=== A/B on hpo_val (identical loader) ===")
    for tag, m in [('with_val (current)', model), ('full_train (loaded)', model_full)]:
        r = m.evaluate(eval_loaders['hpo_val'], stream_monitoring=False)
        print(f"  {tag:<22}  acc={r['accuracy']*100:5.2f}%  mca={r['mean_class_accuracy']*100:5.2f}%  loss={r['loss']:.4f}")

    print("\nBoth checkpoints evaluated against the SAME test_loader above.")
    print("If the two disagree substantially on official_test, the gap is between checkpoints")
    print("(config or training-data difference), not between evaluation paths. 20.8 tells you")
    print("how much of any gap is BN-statistics-driven on each checkpoint individually.")

    del model_full
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# --- 20.12 Split parity: does this run's SGKF fold match HPO's? ---
# The documentation (docs/scene_aware_splitting.md) says StratifiedGroupKFold
# with random_state=152 produces 3,898 train / 947 val. This run reported
# 3,876 / 969 — a 22-sample partition difference. This cell reproduces the
# canonical split with seed=152 and compares to the indices this run actually
# used, surfacing any drift from SEED reassignment, scene_groups changes, or
# labels.txt changes.
import hashlib
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

print(f"sklearn version:                {sklearn.__version__}")
print(f"SEED in scope right now:        {SEED}")
print(f"SEED used in cell 19 (training): {SEED}  (same variable — no reassignment found)")

CANONICAL_SEED = 152
DATA_ROOT = DATASET_CONFIG['data_root']

with open(os.path.join(DATA_ROOT, 'train', 'scene_groups.json'), 'rb') as f:
    _sg_bytes = f.read()
with open(os.path.join(DATA_ROOT, 'train', 'labels.txt'), 'rb') as f:
    _lbl_bytes = f.read()
_scene_groups = json.loads(_sg_bytes.decode())
_all_labels   = [int(x) for x in _lbl_bytes.decode().strip().splitlines()]

print(f"\nscene_groups.json  SHA-256: {hashlib.sha256(_sg_bytes).hexdigest()[:16]}  "
      f"(entries: {len(_scene_groups)}, unique scenes: {len(set(_scene_groups))})")
print(f"labels.txt         SHA-256: {hashlib.sha256(_lbl_bytes).hexdigest()[:16]}  "
      f"(entries: {len(_all_labels)})")

# Canonical split (seed=152, exactly as docs/scene_aware_splitting.md specifies)
sgkf_canon = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CANONICAL_SEED)
canon_train, canon_val = next(sgkf_canon.split(
    range(len(_all_labels)), _all_labels, _scene_groups,
))
canon_train_s = set(canon_train.tolist())
canon_val_s   = set(canon_val.tolist())
print(f"\nCanonical split (seed=152):     train={len(canon_train)}  val={len(canon_val)}")
print(f"docs/scene_aware_splitting.md:  train=3898       val=947")
if len(canon_train) == 3898 and len(canon_val) == 947:
    print("  [OK] canonical reproduction matches documented sizes.")
else:
    print(f"  [DRIFT] canonical SGKF(seed=152) no longer produces documented 3898/947.")
    print(f"          scene_groups.json or labels.txt differ from when the doc was written.")
    print(f"          Any HPO trial that ran against the DOCUMENTED split used a different")
    print(f"          partition than can be reproduced now.")

# This run's actual indices
actual_train_s = set(train_indices)
actual_val_s   = set(val_indices)
print(f"\nThis run's indices (from cell 19): train={len(actual_train_s)}  val={len(actual_val_s)}")

# Diff: canonical vs this-run
train_only_canon  = canon_train_s - actual_train_s
train_only_actual = actual_train_s - canon_train_s
val_only_canon    = canon_val_s   - actual_val_s
val_only_actual   = actual_val_s  - canon_val_s
print(f"\nCanonical(seed=152) vs this-run fold diff:")
print(f"  in canonical train but not this-run train: {len(train_only_canon)}")
print(f"  in this-run train but not canonical train: {len(train_only_actual)}")
print(f"  in canonical val   but not this-run val:   {len(val_only_canon)}")
print(f"  in this-run val    but not canonical val:  {len(val_only_actual)}")

identical = (train_only_canon == set() and train_only_actual == set()
             and val_only_canon == set() and val_only_actual == set())
print(f"\n  [{'OK' if identical else 'MISMATCH'}] folds are "
      f"{'identical' if identical else 'DIFFERENT'}")

# If they don't match, show what moved
if not identical:
    moved_val_to_train = val_only_canon & actual_train_s   # canonical put in val, this-run put in train
    moved_train_to_val = train_only_canon & actual_val_s   # canonical put in train, this-run put in val
    print(f"\n  Samples canonical->val but this-run->train: {len(moved_val_to_train)}")
    print(f"  Samples canonical->train but this-run->val: {len(moved_train_to_val)}")
    # Show a few, to see if they cluster by class or scene
    if moved_train_to_val:
        print("\n  First 5 samples moved from canonical-train to this-run-val:")
        for idx in sorted(moved_train_to_val)[:5]:
            print(f"    idx={idx:5d}  label={_all_labels[idx]:2d} ({CLASS_NAMES[_all_labels[idx]]})  "
                  f"scene={_scene_groups[idx]}")
    if moved_val_to_train:
        print("\n  First 5 samples moved from canonical-val to this-run-train:")
        for idx in sorted(moved_val_to_train)[:5]:
            print(f"    idx={idx:5d}  label={_all_labels[idx]:2d} ({CLASS_NAMES[_all_labels[idx]]})  "
                  f"scene={_scene_groups[idx]}")

# Also compute per-class counts in each partition, side-by-side
print("\nPer-class counts: canonical(seed=152) vs this-run")
print(f"  {'class':<20} {'canon_train':>12} {'run_train':>10} {'canon_val':>10} {'run_val':>8}")
for c in range(NUM_CLASSES):
    ct = sum(1 for i in canon_train_s if _all_labels[i] == c)
    at = sum(1 for i in actual_train_s if _all_labels[i] == c)
    cv = sum(1 for i in canon_val_s   if _all_labels[i] == c)
    av = sum(1 for i in actual_val_s  if _all_labels[i] == c)
    flag = "" if (ct == at and cv == av) else "  <-- class moved"
    print(f"  {CLASS_NAMES[c]:<20} {ct:>12d} {at:>10d} {cv:>10d} {av:>8d}{flag}")

In [ ]:
# --- 20.13 Test-time augmentation: original + horizontal flip ---
# Runs each sample through the model twice — once as-is, once horizontally
# flipped — and averages softmax probabilities before argmax. Flip is a
# spatial operation that commutes with normalization, so we flip the
# already-normalized input tensors directly (no re-normalization needed).
# Mirrors _validate's forward path (model.device, gpu_aug conditional, AMP autocast).
# Typical gain on scene classification: +1-2pp MCA. Zero training cost.
import json

import numpy as np
import torch

@torch.no_grad()
def tta_evaluate(model, loader, flip_dim=-1):
    """Evaluate with TTA (original + horizontal flip). Returns per-sample
    predictions for both the original and the TTA-averaged passes, plus
    aggregate acc/mca and per-class accuracy. No loss computation — TTA
    composes probabilities, not losses."""
    model.eval()
    device = model.device

    all_preds_orig, all_preds_tta, all_labels = [], [], []

    for batch in loader:
        *streams, labels = batch
        streams = [s.to(device, non_blocking=True) for s in streams]
        labels = labels.to(device, non_blocking=True)

        # Match _validate: apply gpu_aug if the model was compiled with it.
        if getattr(model, 'gpu_augmentation', False) and getattr(model, 'gpu_aug', None) is not None:
            streams[0], streams[1] = model.gpu_aug(streams[0], streams[1])

        # Pass 1: original
        with torch.amp.autocast(device_type=device.type, enabled=getattr(model, 'use_amp', False)):
            logits_orig = model(streams)
        probs_orig = torch.softmax(logits_orig.float(), dim=1)

        # Pass 2: horizontal flip (flip_dim = -1 => width)
        flipped = [s.flip(dims=(flip_dim,)) for s in streams]
        with torch.amp.autocast(device_type=device.type, enabled=getattr(model, 'use_amp', False)):
            logits_flip = model(flipped)
        probs_flip = torch.softmax(logits_flip.float(), dim=1)

        probs_avg = (probs_orig + probs_flip) / 2.0

        all_preds_orig.append(probs_orig.argmax(dim=1).cpu().numpy())
        all_preds_tta.append(probs_avg.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    preds_orig = np.concatenate(all_preds_orig)
    preds_tta  = np.concatenate(all_preds_tta)
    labels_np  = np.concatenate(all_labels)

    def metrics(preds):
        acc = float((preds == labels_np).mean())
        pca = np.zeros(NUM_CLASSES)
        pcn = np.zeros(NUM_CLASSES, dtype=np.int64)
        for c in range(NUM_CLASSES):
            m = labels_np == c
            pcn[c] = int(m.sum())
            pca[c] = float((preds[m] == c).mean()) if m.any() else float('nan')
        return acc, float(np.nanmean(pca)), pca, pcn

    acc_o, mca_o, pca_o, pcn = metrics(preds_orig)
    acc_t, mca_t, pca_t, _   = metrics(preds_tta)

    return {
        'orig': {'acc': acc_o, 'mca': mca_o, 'per_class_acc': pca_o, 'preds': preds_orig},
        'tta':  {'acc': acc_t, 'mca': mca_t, 'per_class_acc': pca_t, 'preds': preds_tta},
        'labels': labels_np,
        'per_class_n': pcn,
    }

# Sanity: "orig" TTA-pass accuracy must match Section 19.1's acc exactly (same forward).
print("=== Test-Time Augmentation (original + horizontal flip) ===")
print(f"  {'split':<16} {'orig acc':>10} {'TTA acc':>10} {'Δ acc':>9}   "
      f"{'orig mca':>10} {'TTA mca':>10} {'Δ mca':>9}")
tta_results = {}
for name, loader in eval_loaders.items():
    r = tta_evaluate(model, loader)
    tta_results[name] = r
    d_acc = (r['tta']['acc'] - r['orig']['acc']) * 100
    d_mca = (r['tta']['mca'] - r['orig']['mca']) * 100
    print(f"  {name:<16} {r['orig']['acc']*100:>9.2f}% {r['tta']['acc']*100:>9.2f}% "
          f"{d_acc:>+8.2f}pp  {r['orig']['mca']*100:>9.2f}% {r['tta']['mca']*100:>9.2f}% "
          f"{d_mca:>+8.2f}pp")

    # Parity sanity: orig-path acc should exactly match gen_results' acc
    ref_acc = gen_results[name]['acc']
    if abs(r['orig']['acc'] - ref_acc) * 100 > 0.1:
        print(f"    [WARN] TTA orig-pass acc {r['orig']['acc']*100:.4f}% disagrees with "
              f"Section 19.1 acc {ref_acc*100:.4f}% by >0.1pp")

# Per-class delta on test (the split that matters for generalization claims)
test_r = tta_results['official_test']
delta_pp = (test_r['tta']['per_class_acc'] - test_r['orig']['per_class_acc']) * 100

print("\n=== Per-class TTA improvement on official_test (sorted by Δ, best first) ===")
print(f"  {'class':<20} {'orig':>8} {'TTA':>8} {'Δ pp':>10}   {'N_test':>7}")
order = np.argsort(-delta_pp)
for c in order:
    nm = CLASS_NAMES[c]
    print(f"  {nm:<20} {test_r['orig']['per_class_acc'][c]*100:>7.1f}% "
          f"{test_r['tta']['per_class_acc'][c]*100:>7.1f}% "
          f"{delta_pp[c]:>+9.1f}pp   {int(test_r['per_class_n'][c]):>7d}")

# Summary of the classes that drove the original gap (from 19.5/19.10)
print("\n=== TTA effect on the top-5 reliable gap contributors ===")
watch_list = ['6: dining_area', '14: library', '1: bedroom', '16: office', '2: classroom']
for nm in watch_list:
    if nm in CLASS_NAMES:
        c = CLASS_NAMES.index(nm)
        print(f"  {nm:<20} {test_r['orig']['per_class_acc'][c]*100:>6.1f}% -> "
              f"{test_r['tta']['per_class_acc'][c]*100:>6.1f}%  ({delta_pp[c]:+.1f}pp)")

# Save
summary = {
    s: {
        'original': {'acc': tta_results[s]['orig']['acc'],
                     'mca': tta_results[s]['orig']['mca']},
        'tta':      {'acc': tta_results[s]['tta']['acc'],
                     'mca': tta_results[s]['tta']['mca']},
        'delta_pp': {'acc': (tta_results[s]['tta']['acc'] - tta_results[s]['orig']['acc']) * 100,
                     'mca': (tta_results[s]['tta']['mca'] - tta_results[s]['orig']['mca']) * 100},
    } for s in eval_loaders
}
with open(os.path.join(GEN_DIR, 'tta_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved TTA summary -> {GEN_DIR}/tta_summary.json")

In [ ]:
# --- 20.14 Confusion inspection for top-5 gap-contributor classes ---
# For each class in WATCH_CLASSES, shows the top-K predicted labels on test
# and on val, using the SAME trained model. Side-by-side exposes whether the
# failure is:
#   - semantic confusion (same confused classes on val and test, just worse on test)
#   - distribution shift (val predicts cleanly, test scatters)
#   - mode collapse (test predictions concentrate on a handful of unrelated classes)
import json

import numpy as np

WATCH_CLASSES = ['6: dining_area', '14: library', '1: bedroom', '16: office', '2: classroom']
TOP_K = 7

def inspect_class(class_name, split):
    """Return top-K predicted classes (and counts/pct) for test samples
    whose true label is class_name. Uses gen_results from 19.1."""
    if class_name not in CLASS_NAMES:
        return None
    c = CLASS_NAMES.index(class_name)
    labels = gen_results[split]['labels']
    preds  = gen_results[split]['preds']
    mask = labels == c
    n = int(mask.sum())
    if n == 0:
        return {'n': 0, 'rows': [], 'correct_pct': float('nan')}
    counts = np.bincount(preds[mask], minlength=NUM_CLASSES)
    order = np.argsort(-counts)
    rows = []
    for rank, pc in enumerate(order[:TOP_K]):
        if counts[pc] == 0:
            break
        rows.append({
            'rank': rank + 1,
            'pred_class_idx': int(pc),
            'pred_class': CLASS_NAMES[pc],
            'count': int(counts[pc]),
            'pct_of_true': float(100.0 * counts[pc] / n),
            'correct': bool(pc == c),
        })
    return {'n': n, 'rows': rows, 'correct_pct': float(100.0 * counts[c] / n)}

print("=== Per-class confusion inspection (test vs val, same model) ===\n")

for cls_name in WATCH_CLASSES:
    test_info = inspect_class(cls_name, 'official_test')
    val_info  = inspect_class(cls_name, 'hpo_val')

    print(f"--- {cls_name} ---")
    print(f"  TEST  (N={test_info['n']:>3}, correct={test_info['correct_pct']:>5.1f}%)")
    for r in test_info['rows']:
        tag = '  <-- correct' if r['correct'] else ''
        print(f"    {r['rank']:>2}. {r['pred_class']:<22} n={r['count']:>3}  ({r['pct_of_true']:>5.1f}%){tag}")
    if val_info['n'] > 0:
        print(f"  VAL   (N={val_info['n']:>3}, correct={val_info['correct_pct']:>5.1f}%)")
        for r in val_info['rows']:
            tag = '  <-- correct' if r['correct'] else ''
            print(f"    {r['rank']:>2}. {r['pred_class']:<22} n={r['count']:>3}  ({r['pct_of_true']:>5.1f}%){tag}")
    else:
        print(f"  VAL   (no samples)")
    print()

# Also compute: are the top wrong classes the SAME on val and test?
# If yes -> semantic confusion; if no -> distribution-shift or mode-collapse.
print("=== Val-vs-test confusion overlap (top-3 wrong classes each) ===")
print(f"  {'class':<22} {'test top-3 wrong':<40} {'val top-3 wrong':<40} overlap")
for cls_name in WATCH_CLASSES:
    c = CLASS_NAMES.index(cls_name)
    def top3_wrong(info):
        wrong = [r for r in info['rows'] if not r['correct']]
        return [r['pred_class_idx'] for r in wrong[:3]]
    t_info = inspect_class(cls_name, 'official_test')
    v_info = inspect_class(cls_name, 'hpo_val')
    t_top = top3_wrong(t_info)
    v_top = top3_wrong(v_info) if v_info['n'] > 0 else []
    overlap = set(t_top) & set(v_top)
    t_str = ', '.join(CLASS_NAMES[i].split(': ')[1] for i in t_top) or '(none)'
    v_str = ', '.join(CLASS_NAMES[i].split(': ')[1] for i in v_top) or '(none)'
    print(f"  {cls_name:<22} {t_str:<40} {v_str:<40} {len(overlap)}/{min(3, len(t_top), len(v_top)) or 0}")

# Persist detail for later
detail = {
    cls: {
        'test': inspect_class(cls, 'official_test'),
        'val':  inspect_class(cls, 'hpo_val'),
    } for cls in WATCH_CLASSES
}
with open(os.path.join(GEN_DIR, 'confusion_inspection.json'), 'w') as f:
    json.dump(detail, f, indent=2)
print(f"\nSaved confusion detail -> {GEN_DIR}/confusion_inspection.json")

In [ ]:
# --- 20.15 Per-class sensor + scene-group composition across splits ---
# For each watch class, compare hpo_train / hpo_val / official_test composition
# by:
#   1. Sensor (kv1/kv2/realsense/xtion) — first path component
#   2. Scene group (from scene_groups.json) — location / site key
# If val and test have different sensor mixes or draw from different scene-group
# pools for a given class, that's distribution shift at the subscene level
# and explains why val per-class accuracy doesn't transfer to test.
import os
import json
from collections import Counter

import numpy as np

SENSORS = ("kv1", "kv2", "realsense", "xtion")
ROOT = DATASET_CONFIG['data_root']

# Load sample_paths + scene_groups for both splits
with open(os.path.join(ROOT, "train", "sample_paths.json")) as f:
    train_paths_all = json.load(f)
with open(os.path.join(ROOT, "train", "scene_groups.json")) as f:
    train_scene_groups_all = json.load(f)
with open(os.path.join(ROOT, "test",  "sample_paths.json")) as f:
    test_paths_all = json.load(f)
with open(os.path.join(ROOT, "test",  "scene_groups.json")) as f:
    test_scene_groups_all = json.load(f)

def sensor_of(path):
    return path.split('/', 1)[0]

def class_sensors_and_scenes(class_idx, split):
    """Return (sensors, scene_groups) for samples of class_idx in split,
    aligned with gen_results[split]['labels'] ordering."""
    labels = gen_results[split]['labels']
    mask = labels == class_idx
    positions = np.where(mask)[0]

    if split == 'official_test':
        paths  = test_paths_all
        scenes = test_scene_groups_all
    elif split == 'hpo_train':
        paths  = [train_paths_all[i]          for i in train_indices]
        scenes = [train_scene_groups_all[i]   for i in train_indices]
    elif split == 'hpo_val':
        paths  = [train_paths_all[i]          for i in val_indices]
        scenes = [train_scene_groups_all[i]   for i in val_indices]
    else:
        raise ValueError(split)
    sens = [sensor_of(paths[p]) for p in positions]
    sgs  = [scenes[p]           for p in positions]
    return sens, sgs

for cls_name in WATCH_CLASSES:
    c = CLASS_NAMES.index(cls_name)
    print(f"\n=== {cls_name} ===")

    # 1. Sensor distribution
    print(f"  Sensor distribution:")
    header = f"    {'split':<16}" + ''.join(f"{s:>12}" for s in SENSORS) + f"{'total':>8}"
    print(header)
    split_sens = {}
    for split in ('hpo_train', 'hpo_val', 'official_test'):
        sens, _ = class_sensors_and_scenes(c, split)
        split_sens[split] = sens
        total = len(sens)
        counts = Counter(sens)
        row = f"    {split:<16}"
        for s in SENSORS:
            n = counts.get(s, 0)
            pct = 100.0 * n / max(total, 1)
            row += f"{n:>5}({pct:>4.1f}%)"
        row += f"{total:>8}"
        print(row)

    # 2. Scene group unique counts + overlap
    sg_train = class_sensors_and_scenes(c, 'hpo_train')[1]
    sg_val   = class_sensors_and_scenes(c, 'hpo_val')[1]
    sg_test  = class_sensors_and_scenes(c, 'official_test')[1]
    uniq_train, uniq_val, uniq_test = set(sg_train), set(sg_val), set(sg_test)

    print(f"\n  Unique scene groups (per split):")
    print(f"    hpo_train:      {len(uniq_train):>4} scenes, {len(sg_train):>4} samples")
    print(f"    hpo_val:        {len(uniq_val):>4} scenes, {len(sg_val):>4} samples")
    print(f"    official_test:  {len(uniq_test):>4} scenes, {len(sg_test):>4} samples")

    # Where do test samples come from, relative to train+val?
    seen_set = uniq_train | uniq_val
    test_in_seen = [s for s in sg_test if s in seen_set]
    test_exclusive_samples = [s for s in sg_test if s not in seen_set]
    test_excl_uniq = set(test_exclusive_samples)
    print(f"\n  Test scene-group overlap with train+val:")
    print(f"    in train+val scenes:  {len(test_in_seen):>4} test samples ({100.0*len(test_in_seen)/max(len(sg_test),1):.1f}%)")
    print(f"    test-exclusive:       {len(test_exclusive_samples):>4} test samples across {len(test_excl_uniq)} scenes ({100.0*len(test_exclusive_samples)/max(len(sg_test),1):.1f}%)")

    # Top-N scene groups per split (sorted by sample count)
    print(f"\n  Top-6 scene groups by sample count:")
    for split, sg in (('hpo_train', sg_train), ('hpo_val', sg_val), ('official_test', sg_test)):
        ctr = Counter(sg).most_common(6)
        print(f"    {split}:")
        for scene, n in ctr:
            print(f"      {scene:<60} n={n}")

# Save
detail = {}
for cls_name in WATCH_CLASSES:
    c = CLASS_NAMES.index(cls_name)
    row = {}
    for split in ('hpo_train', 'hpo_val', 'official_test'):
        sens, sgs = class_sensors_and_scenes(c, split)
        row[split] = {
            'n_samples': len(sgs),
            'n_unique_scenes': len(set(sgs)),
            'sensor_counts': dict(Counter(sens)),
            'top_scenes': Counter(sgs).most_common(10),
        }
    detail[cls_name] = row
with open(os.path.join(GEN_DIR, 'class_subscene_composition.json'), 'w') as f:
    json.dump(detail, f, indent=2, default=str)
print(f"\nSaved per-class subscene composition -> {GEN_DIR}/class_subscene_composition.json")

In [ ]:
# --- 20.16 Per-(class × sensor) coverage upper-bound check ---
# Diagnostic only: asks what fraction of test could be covered by ANY train-only
# reweighting scheme (uniform, inverse-frequency, or otherwise). If test has
# samples in cells where hpo_train has ZERO samples, no reweighting of train
# can produce those combinations — the model would never see them during
# training, regardless of weighting choice.
#
# This is a fundamental coverage limit, independent of the specific weighting
# scheme chosen. Reads the test grid for this comparison only; no test
# statistics feed into training weights (see 20.17, which uses train only).
import os
import json

import numpy as np
import pandas as pd

SENSORS = ("kv1", "kv2", "realsense", "xtion")

with open(os.path.join(DATASET_CONFIG['data_root'], 'train', 'sample_paths.json')) as f:
    train_paths_all = json.load(f)
with open(os.path.join(DATASET_CONFIG['data_root'], 'test',  'sample_paths.json')) as f:
    test_paths_all = json.load(f)

def sensor_of(path):
    return path.split('/', 1)[0]

def build_grid(paths, labels, indices=None):
    if indices is None:
        indices = range(len(labels))
    grid = np.zeros((NUM_CLASSES, len(SENSORS)), dtype=np.int64)
    for i in indices:
        y = int(labels[i])
        s = sensor_of(paths[i])
        if s in SENSORS:
            grid[y, SENSORS.index(s)] += 1
    return grid

# Grids
train_labels_full = train_full_dataset.labels
hpo_train_grid   = build_grid(train_paths_all, train_labels_full, train_indices)
full_train_grid  = build_grid(train_paths_all, train_labels_full, None)
test_grid        = build_grid(test_paths_all, gen_results['official_test']['labels'], None)

def show_grid(grid, label):
    df = pd.DataFrame(grid, columns=list(SENSORS))
    df.insert(0, 'class', CLASS_NAMES)
    df['total'] = grid.sum(axis=1)
    print(f"\n=== {label} (class x sensor counts) ===")
    print(df.to_string(index=False))

show_grid(hpo_train_grid, f"hpo_train (N={len(train_indices)})")
show_grid(test_grid,      f"official_test (N={test_grid.sum()})")

# Reachability: which (class, sensor) cells have test samples but hpo_train has zero?
print("\n=== Reachability analysis ===")
total_test = int(test_grid.sum())

def count_unreachable(source_grid, name):
    unreachable_cells = []
    unreachable_samples = 0
    for c in range(NUM_CLASSES):
        for s_idx, s in enumerate(SENSORS):
            if test_grid[c, s_idx] > 0 and source_grid[c, s_idx] == 0:
                unreachable_samples += int(test_grid[c, s_idx])
                unreachable_cells.append((CLASS_NAMES[c], s, int(test_grid[c, s_idx])))
    reachable = total_test - unreachable_samples
    print(f"\n  Source: {name}")
    print(f"    Test samples with >0 matching source cells: "
          f"{reachable:>5} ({100.0*reachable/total_test:.1f}%)")
    print(f"    Test samples with ZERO matching source cells: "
          f"{unreachable_samples:>5} ({100.0*unreachable_samples/total_test:.1f}%)")
    if unreachable_cells:
        print(f"    Unreachable (class x sensor) cells (class, sensor, #test_samples):")
        for cls, sens, n in sorted(unreachable_cells, key=lambda x: -x[2])[:15]:
            print(f"      {cls:<22} + {sens:<10} : {n:>3}")
        if len(unreachable_cells) > 15:
            print(f"      ... ({len(unreachable_cells)-15} more)")
    return unreachable_samples, unreachable_cells

hpo_unreach, hpo_cells   = count_unreachable(hpo_train_grid,  "hpo_train")
full_unreach, full_cells = count_unreachable(full_train_grid, "full train pool (4845)")

print("\n=== Interpretation ===")
if hpo_unreach / total_test < 0.03:
    print(f"  [OK] Only {100*hpo_unreach/total_test:.1f}% of test is in hpo_train-unreachable cells.")
    print("       Per-(class x sensor) reweighting of hpo_train CAN meaningfully")
    print("       address most of the test distribution shift.")
elif hpo_unreach / total_test < 0.15:
    print(f"  [CAUTION] {100*hpo_unreach/total_test:.1f}% of test is in unreachable cells.")
    print("       Reweighting will partially help; residual gap comes from these cells.")
else:
    print(f"  [PROBLEM] {100*hpo_unreach/total_test:.1f}% of test is in unreachable cells.")
    print("       Reweighting alone cannot solve this — you need training data from")
    print("       those (class x sensor) combinations or cross-campaign augmentation.")

# Save the grid
np.savez_compressed(os.path.join(GEN_DIR, 'class_sensor_grids.npz'),
                    hpo_train=hpo_train_grid, full_train=full_train_grid, test=test_grid,
                    sensors=np.array(SENSORS), class_names=np.array(CLASS_NAMES))
print(f"\nSaved grids -> {GEN_DIR}/class_sensor_grids.npz")

In [ ]:
# --- 20.17 Per-(class × sensor) weighted sampler: construction + drop-in for cell 19 ---
# Replaces the existing class-weighted sampler (weights = num_train / count_per_class)
# with a per-(class × sensor) sampler that makes:
#   P(class)           = 1/NUM_CLASSES   (uniform across classes)
#   P(sensor | class)  = 1/S_c           (uniform across sensors represented in class c)
# This decorrelates class from sensor: for each class, the model sees each sensor
# that exists for that class with equal expected rate.
#
# IMPORTANT — no test-set leakage:
# The weights computed below use ONLY training statistics — specifically, the
# (class, sensor) counts within hpo_train. No test-set frequencies, labels, or
# statistics are read during weight construction. The goal is to make the
# training distribution (class ⫫ sensor)-independent, not to match test's
# specific (class, sensor) frequencies. Alternative schemes that DO read
# test statistics (e.g., matching test's per-class sensor mix) would constitute
# test-set leakage and are not used here.
#
# Outputs:
#   - sample_weights_cxs: torch tensor of shape (len(train_indices),)
#   - Saves to disk for reuse in next training run
#   - Prints a drop-in code block for cell 19
#   - Shows before/after effective (class × sensor) distribution
from collections import Counter, defaultdict

import numpy as np
import torch

# Labels + sensors for hpo_train (the subset we train on)
hpo_train_labels  = [int(train_full_dataset.labels[i]) for i in train_indices]
hpo_train_sensors = [sensor_of(train_paths_all[i]) for i in train_indices]

# Group sample positions by (class, sensor) cell
cell_to_positions = defaultdict(list)
for pos, (y, s) in enumerate(zip(hpo_train_labels, hpo_train_sensors)):
    cell_to_positions[(y, s)].append(pos)

# Count represented sensors per class (for the 1/S_c factor)
sensors_per_class = defaultdict(set)
for (y, s) in cell_to_positions:
    sensors_per_class[y].add(s)

# Build weights: w_i = 1 / (NUM_CLASSES * S_{c_i} * N_{c_i, s_i})
num_samples = len(train_indices)
sample_weights_cxs = torch.zeros(num_samples, dtype=torch.float32)
for (y, s), positions in cell_to_positions.items():
    n_sensors_for_class = len(sensors_per_class[y])
    n_in_cell = len(positions)
    w_per_sample = 1.0 / (NUM_CLASSES * n_sensors_for_class * n_in_cell)
    for pos in positions:
        sample_weights_cxs[pos] = w_per_sample

# Normalize so sum of weights = num_samples (expected draws per epoch stays unchanged)
sample_weights_cxs = sample_weights_cxs * num_samples / sample_weights_cxs.sum()

# --- Before / after: effective (class × sensor) distribution over one epoch ---
# Current cell 19 sampler: weights = num_train / count_per_class, normalized implicitly via WeightedRandomSampler
current_label_counts = Counter(hpo_train_labels)
num_train = num_samples
current_class_weights = {y: num_train / c for y, c in current_label_counts.items()}
sample_weights_current = torch.tensor(
    [current_class_weights[y] for y in hpo_train_labels], dtype=torch.float32
)
sample_weights_current = sample_weights_current * num_samples / sample_weights_current.sum()

def effective_grid(weights):
    g = np.zeros((NUM_CLASSES, len(SENSORS)), dtype=np.float64)
    for w, y, s in zip(weights.tolist(), hpo_train_labels, hpo_train_sensors):
        if s in SENSORS:
            g[y, SENSORS.index(s)] += w
    return g

before_grid = effective_grid(sample_weights_current)
after_grid  = effective_grid(sample_weights_cxs)

def show_dist(grid, label, classes_to_show=None):
    if classes_to_show is None:
        classes_to_show = range(NUM_CLASSES)
    print(f"\n  {label} — expected samples per (class, sensor) cell per epoch:")
    print(f"    {'class':<22}" + ''.join(f"{s:>10}" for s in SENSORS) + f"{'total':>10}")
    for c in classes_to_show:
        row = f"    {CLASS_NAMES[c]:<22}"
        total = 0.0
        for s_idx in range(len(SENSORS)):
            v = grid[c, s_idx]
            row += f"{v:>10.1f}"
            total += v
        row += f"{total:>10.1f}"
        print(row)

# Show effect on the 5 gap-contributor classes specifically
watch_class_idxs = [CLASS_NAMES.index(n) for n in WATCH_CLASSES if n in CLASS_NAMES]
print("=== Effect of new sampler on the top-5 gap classes ===")
show_dist(before_grid, "BEFORE (current class-only sampler)", watch_class_idxs)
show_dist(after_grid,  "AFTER  (per-(class × sensor) sampler)", watch_class_idxs)

# Class marginals — should go from current (proportional to inverse class count) to uniform 1/K
print("\n  Class marginal totals (should be uniform under new sampler):")
print(f"    {'class':<22}{'BEFORE':>14}{'AFTER':>14}")
for c in range(NUM_CLASSES):
    b = float(before_grid[c].sum())
    a = float(after_grid[c].sum())
    print(f"    {CLASS_NAMES[c]:<22}{b:>14.1f}{a:>14.1f}")

# Save weights for reuse
out_path = os.path.join(GEN_DIR, 'sample_weights_class_x_sensor.pt')
torch.save({
    'sample_weights': sample_weights_cxs,
    'train_indices': list(train_indices),
    'hpo_train_labels': hpo_train_labels,
    'hpo_train_sensors': hpo_train_sensors,
    'cell_counts': {f"{y}|{s}": len(v) for (y, s), v in cell_to_positions.items()},
    'sensors': list(SENSORS),
    'class_names': list(CLASS_NAMES),
}, out_path)
print(f"\n  Saved sample weights -> {out_path}")

# Drop-in snippet for cell 19
print("\n=== DROP-IN CODE for cell 19 (replace the WeightedRandomSampler block) ===")
print('''
# --- Per-(class × sensor) weighted sampler (replaces class-only sampler) ---
from collections import defaultdict

def _sensor_of(path):
    return path.split('/', 1)[0]

with open(os.path.join(DATASET_CONFIG['data_root'], 'train', 'sample_paths.json')) as _f:
    _train_paths_all = json.load(_f)

_subset_labels  = [all_labels[i] for i in train_indices]
_subset_sensors = [_sensor_of(_train_paths_all[i]) for i in train_indices]

_cell_positions = defaultdict(list)
for _pos, (_y, _s) in enumerate(zip(_subset_labels, _subset_sensors)):
    _cell_positions[(_y, _s)].append(_pos)

_sensors_per_class = defaultdict(set)
for (_y, _s) in _cell_positions:
    _sensors_per_class[_y].add(_s)

_num_classes = DATASET_CONFIG['num_classes']
_n = len(_subset_labels)
sample_weights = torch.zeros(_n, dtype=torch.float32)
for (_y, _s), _pos_list in _cell_positions.items():
    _w = 1.0 / (_num_classes * len(_sensors_per_class[_y]) * len(_pos_list))
    for _pos in _pos_list:
        sample_weights[_pos] = _w
sample_weights = sample_weights * _n / sample_weights.sum()

g = torch.Generator().manual_seed(SEED)
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=_n,
    replacement=True, generator=g,
)
''')

In [ ]:
# --- 20.18 Per-(class × sensor) train/test accuracy: memorization check ---
# For each training cell (class, sensor) with samples, compute:
#   - Train-cell accuracy (measured on hpo_train with no aug, via gen_results)
#   - Test-cell accuracy (same model, same cell)
#   - Expected draws per epoch under the active sampler
# Flags memorization: small cells with high train acc AND large train-test gap.
# Directly quantifies the concern from cell 19's [WARN] about rare-cell exposure
# (e.g., lab + kv1 with 2 samples drawn 51x/epoch).
import os
import json
from collections import defaultdict

import numpy as np
import pandas as pd

# Paths (read defensively; may already be loaded elsewhere)
with open(os.path.join(DATASET_CONFIG['data_root'], 'train', 'sample_paths.json')) as _f:
    _train_paths_all = json.load(_f)
with open(os.path.join(DATASET_CONFIG['data_root'], 'test',  'sample_paths.json')) as _f:
    _test_paths_all = json.load(_f)

def _sensor(path):
    return path.split('/', 1)[0]

# hpo_train per-position (class, sensor)
hpo_train_cs = [(int(train_full_dataset.labels[i]), _sensor(_train_paths_all[i])) for i in train_indices]
test_sensors = [_sensor(p) for p in _test_paths_all]

train_preds  = gen_results['hpo_train']['preds']
train_labels = gen_results['hpo_train']['labels']
test_preds   = gen_results['official_test']['preds']
test_labels_ = gen_results['official_test']['labels']

# Aggregate per-cell counts + correctness
train_cells = defaultdict(lambda: {'correct': 0, 'total': 0})
for pos, (y, s) in enumerate(hpo_train_cs):
    train_cells[(y, s)]['total'] += 1
    if train_preds[pos] == train_labels[pos]:
        train_cells[(y, s)]['correct'] += 1

test_cells = defaultdict(lambda: {'correct': 0, 'total': 0})
for pos in range(len(test_labels_)):
    y = int(test_labels_[pos])
    s = test_sensors[pos]
    test_cells[(y, s)]['total'] += 1
    if test_preds[pos] == test_labels_[pos]:
        test_cells[(y, s)]['correct'] += 1

# Draws per epoch under the current per-(class × sensor) sampler:
#   draws = num_train / (K * S_c * N_{c,s})
sensors_per_class = defaultdict(set)
for (y, s) in train_cells:
    sensors_per_class[y].add(s)

K = NUM_CLASSES
num_train = len(hpo_train_cs)

rows = []
for (y, s), st in train_cells.items():
    N_tr = st['total']
    S_c = len(sensors_per_class[y])
    draws = num_train / (K * S_c * N_tr)
    train_acc = st['correct'] / N_tr
    te = test_cells.get((y, s), {'correct': 0, 'total': 0})
    test_acc = (te['correct'] / te['total']) if te['total'] > 0 else float('nan')
    gap = (train_acc - test_acc) if te['total'] > 0 else float('nan')
    rows.append({
        'class': CLASS_NAMES[y], 'sensor': s,
        'N_train': N_tr, 'train_acc': train_acc, 'draws_per_ep': draws,
        'N_test': te['total'], 'test_acc': test_acc, 'train_test_gap': gap,
    })
df = pd.DataFrame(rows)
df.to_csv(os.path.join(GEN_DIR, 'per_cell_train_test_accuracy.csv'), index=False)

# Memorization heuristic
SMALL_CELL      = 10    # train cells smaller than this are at risk
HIGH_TRAIN_ACC  = 0.95  # memorized cells tend to be near-perfect on train
MEMO_GAP_PP     = 30.0  # train-test gap > 30pp is a strong signal

def is_memorized(r):
    if r['N_train'] >= SMALL_CELL: return False
    if r['train_acc'] < HIGH_TRAIN_ACC: return False
    if np.isnan(r['test_acc']): return False  # no test cell to compare against
    return (r['train_acc'] - r['test_acc']) * 100 > MEMO_GAP_PP

df['memorized'] = df.apply(is_memorized, axis=1)

# Print sorted by draws/epoch descending (worst offenders first)
print("=== Per-cell train vs test accuracy (sorted by draws/epoch) ===")
print(f"  Flag criteria: N_train<{SMALL_CELL} AND train_acc>{HIGH_TRAIN_ACC*100:.0f}% AND gap>{MEMO_GAP_PP:.0f}pp\n")
print(f"  {'class':<22}{'sensor':<11}{'N_tr':>5}{'tr_acc':>9}{'draws/ep':>10}"
      f"{'N_te':>6}{'te_acc':>9}{'gap':>10}  FLAG")
df_sorted = df.sort_values('draws_per_ep', ascending=False)
for _, r in df_sorted.head(20).iterrows():
    tacc = f"{r['test_acc']*100:.1f}%" if not np.isnan(r['test_acc']) else '   n/a'
    gap  = f"{r['train_test_gap']*100:+.1f}pp" if not np.isnan(r['train_test_gap']) else '   n/a'
    flag = '  MEMORIZATION' if r['memorized'] else ''
    print(f"  {r['class']:<22}{r['sensor']:<11}{int(r['N_train']):>5}"
          f"{r['train_acc']*100:>8.1f}%{r['draws_per_ep']:>10.1f}"
          f"{int(r['N_test']):>6}{tacc:>9}{gap:>10}{flag}")

# Summary
n_flagged = int(df['memorized'].sum())
n_total   = len(df)
tot_test_in_flagged = int(df[df['memorized']]['N_test'].sum())
print(f"\n  Flagged cells: {n_flagged}/{n_total}")
print(f"  Test samples in flagged cells: {tot_test_in_flagged} "
      f"({100.0*tot_test_in_flagged/len(test_labels_):.1f}% of test)")

# Focus on the hammered cells (the warning from cell 19)
print(f"\n=== Hammered cells (draws/ep > 10) — the [WARN] candidates ===")
hammered = df[df['draws_per_ep'] > 10].sort_values('draws_per_ep', ascending=False)
if len(hammered) == 0:
    print("  None.")
else:
    print(f"  {'class':<22}{'sensor':<11}{'N_tr':>5}{'tr_acc':>9}{'draws/ep':>10}"
          f"{'N_te':>6}{'te_acc':>9}{'gap':>10}")
    for _, r in hammered.iterrows():
        tacc = f"{r['test_acc']*100:.1f}%" if not np.isnan(r['test_acc']) else '   n/a'
        gap  = f"{r['train_test_gap']*100:+.1f}pp" if not np.isnan(r['train_test_gap']) else '   n/a'
        print(f"  {r['class']:<22}{r['sensor']:<11}{int(r['N_train']):>5}"
              f"{r['train_acc']*100:>8.1f}%{r['draws_per_ep']:>10.1f}"
              f"{int(r['N_test']):>6}{tacc:>9}{gap:>10}")

# Test samples that fall in cells with ZERO training (unreachable from cell 20.16)
print(f"\n=== Test cells with zero training (unreachable) — can't be fixed by any weighting ===")
unreachable_test = 0
for (y, s), te in test_cells.items():
    if (y, s) not in train_cells and te['total'] > 0:
        acc = te['correct'] / te['total']
        print(f"  {CLASS_NAMES[y]:<22} + {s:<10}  N_test={te['total']}, test_acc={acc*100:.1f}%")
        unreachable_test += te['total']
print(f"  Total test samples in unreachable cells: {unreachable_test} "
      f"({100.0*unreachable_test/len(test_labels_):.1f}% of test)")

print(f"\nSaved per-cell accuracy -> {GEN_DIR}/per_cell_train_test_accuracy.csv")

In [ ]:
# --- 20.19 Cross-run accumulated results ---
# Saves this run's test metrics to run_summary.json in GEN_DIR, then scans
# linet_checkpoints/run_*/gen_diagnostics/run_summary.json across all runs
# and prints a side-by-side comparison (test MCA + per-class accuracy on
# spotlight classes). First run just deposits its own summary; subsequent
# runs get comparison automatically.
import os
import json
from pathlib import Path

import pandas as pd

# CLASS_NAMES entries are stored as 'N: name' (e.g. '9: furniture_store');
# normalize to bare names for SPOTLIGHT_CLASSES lookup and as keys in the
# saved run_summary.json so cross-run comparison is stable.
def _clean_class_name(raw):
    return raw.split(': ', 1)[1] if ': ' in raw else raw
CLASS_NAMES_CLEAN = [_clean_class_name(n) for n in CLASS_NAMES]

SPOTLIGHT_CLASSES = [
    'furniture_store', 'dining_area', 'library', 'classroom', 'bedroom', 'office',
]

# ------------------------------------------------------------------

# ------------------------------------------------------------------
this_run_summary = {
    'run_dir': checkpoint_dir,
    'timestamp': timestamp,
    'seed': int(SEED),
    'sampler_variant':    TRAIN_CONFIG.get('sampler_variant'),
    'use_mixup':          bool(TRAIN_CONFIG.get('use_mixup', False)),
    'mixup_alpha':        float(TRAIN_CONFIG.get('mixup_alpha', 0.0)),
    'use_sam':            bool(TRAIN_CONFIG.get('use_sam', False)),
    'sam_rho':            float(TRAIN_CONFIG.get('sam_rho', 0.0)),
    'label_smoothing':    float(TRAIN_CONFIG.get('label_smoothing', 0.0)),
    'test_acc':           float(gen_results['official_test']['acc']),
    'test_mca':           float(gen_results['official_test']['mca']),
    'hpo_val_acc':        float(gen_results['hpo_val']['acc']),
    'hpo_val_mca':        float(gen_results['hpo_val']['mca']),
    'per_class_test_acc': {
        CLASS_NAMES_CLEAN[_c]: float(gen_results['official_test']['per_class_acc'][_c])
        for _c in range(NUM_CLASSES)
    },
}
_summary_path = os.path.join(GEN_DIR, 'run_summary.json')
with open(_summary_path, 'w') as _f:
    json.dump(this_run_summary, _f, indent=2)
print(f"\n\nSaved run summary -> {_summary_path}")

# Scan all run_summary.json files under linet_checkpoints/
_runs_root = Path(checkpoint_dir).parent
_all_summaries = []
for _p in sorted(_runs_root.glob('run_*/gen_diagnostics/run_summary.json')):
    try:
        with open(_p) as _f:
            _all_summaries.append(json.load(_f))
    except Exception as _e:
        print(f"  [warn] could not read {_p}: {_e}")

if len(_all_summaries) < 2:
    print(f"\nOnly {len(_all_summaries)} run_summary.json found in {_runs_root}.")
    print("Cross-run comparison will populate once you've run multiple variants.")
else:
    print(f"\n=== Cross-run results: {len(_all_summaries)} runs ===")
    _top_rows = []
    for _s in _all_summaries:
        _top_rows.append({
            'variant': _s.get('sampler_variant'),
            'mixup':   'y' if _s.get('use_mixup') else 'n',
            'sam':     'y' if _s.get('use_sam') else 'n',
            'LS':      _s.get('label_smoothing'),
            'seed':    _s.get('seed'),
            'test_mca': _s['test_mca'] * 100,
            'test_acc': _s['test_acc'] * 100,
            'val_mca':  _s.get('hpo_val_mca', 0) * 100,
            'run':      os.path.basename(_s.get('run_dir', '?')),
        })
    _top_df = pd.DataFrame(_top_rows).sort_values('test_mca', ascending=False)
    with pd.option_context('display.float_format', '{:.2f}'.format,
                           'display.width', 200, 'display.max_columns', None):
        print(_top_df.to_string(index=False))

    # Per-class comparison on spotlight classes
    print(f"\n=== Per-class test accuracy on spotlight classes (%) ===")
    _sp_rows = []
    for _s in _all_summaries:
        _row = {
            'variant': _s.get('sampler_variant'),
            'mixup':   'y' if _s.get('use_mixup') else 'n',
            'sam':     'y' if _s.get('use_sam') else 'n',
            'LS':      _s.get('label_smoothing'),
        }
        for _c in SPOTLIGHT_CLASSES:
            _pc = _s.get('per_class_test_acc', {}).get(_c)
            _row[_c] = (_pc * 100) if _pc is not None else float('nan')
        _sp_rows.append(_row)
    _sp_df = pd.DataFrame(_sp_rows)
    with pd.option_context('display.float_format', '{:.1f}'.format,
                           'display.width', 240, 'display.max_columns', None):
        print(_sp_df.to_string(index=False))
